<a href="https://colab.research.google.com/github/mardocheeogecime-gif/Rede_Colabora-o_Acad-mica_ECI_UFMG/blob/main/Parte_16_Semantic_Topological.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# PARTE 16 — Semantic-Topological Community Detection
Módulo autônomo: monta o Google Drive, localiza o run histórico e executa somente a nova análise semântico-topológica.


In [ ]:
# ============================================================
# PARTE 16 — SEMANTIC-TOPOLOGICAL COMMUNITY DETECTION
# Novo módulo independente para o artigo IEEE LATAM
# ============================================================
# Objetivo:
#   Integrar a topologia de coautoria do GCC com embeddings semânticos
#   das publicações e comparar condições topology-only, semantic-only
#   e semantic-topological sob múltiplos algoritmos, estabilidade,
#   perturbações e modelos nulos.
# ============================================================

# ============================================================
# 0. INSTALAÇÃO/IMPORTS, DRIVE E CONFIGURAÇÃO
# ============================================================
import os
import sys
import re
import ast
import json
import math
import time
import pickle
import random
import hashlib
import platform
import subprocess
from pathlib import Path
from datetime import datetime
from itertools import combinations
from collections import defaultdict

# ---- Instala apenas o que pode não existir no Colab ----
def ensure_packages():
    packages = [
        "sentence-transformers",
        "python-louvain",
        "igraph",
        "leidenalg",
        "openpyxl",
    ]
    cmd = [sys.executable, "-m", "pip", "install", "-q"] + packages
    subprocess.check_call(cmd)

ensure_packages()

import numpy as np
import pandas as pd
import networkx as nx
import scipy
from scipy.stats import chi2_contingency
from sklearn.metrics import normalized_mutual_info_score, adjusted_rand_score
from sklearn.preprocessing import normalize
from sentence_transformers import SentenceTransformer
import torch
import community as community_louvain
import igraph as ig
import leidenalg as la
import matplotlib.pyplot as plt

# ---- Google Drive ----
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)

# ---- Reprodutibilidade ----
GLOBAL_SEED = 42
random.seed(GLOBAL_SEED)
np.random.seed(GLOBAL_SEED)
os.environ["PYTHONHASHSEED"] = str(GLOBAL_SEED)

# ============================================================
# 0.1 CONFIGURAÇÃO EDITÁVEL
# ============================================================
PROJECT_ROOT = Path("/content/drive/MyDrive/tese_doutoral")

# Run histórico conhecido pelo notebook original.
# Se não existir, o código procura automaticamente o run completo mais recente.
PREFERRED_SOURCE_RUN = "run_20260425_002158"

# Embedding multilíngue: adequado a títulos em português/inglês/espanhol.
MODEL_NAME = "sentence-transformers/paraphrase-multilingual-mpnet-base-v2"
MODEL_CACHE = PROJECT_ROOT / "model_cache"

# A base histórica da Parte 13 usou títulos. Mantemos isso como padrão
# para comparabilidade e auditabilidade. Se futuramente houver resumos,
# mudar explicitamente para "title_abstract" após validar a coluna.
TEXT_MODE = "title_only"  # opções: title_only | title_abstract

# Escopo semântico:
# todas as publicações nas quais pelo menos 1 pesquisador do GCC aparece.
# A topologia continua sendo exclusivamente a coautoria intra-PPGCI.
SEMANTIC_SCOPE = "all_publications_with_at_least_one_gcc_researcher"

# Grafo semântico k-NN
KNN_VALUES = [5, 10, 15, 20]
MIN_SEMANTIC_SIMILARITY = 0.0

# Fusão: alpha=1 -> topologia; alpha=0 -> semântica.
FUSION_ALPHAS = [0.25, 0.50, 0.75]

# Community detection
ALGORITHMS = ["louvain", "leiden", "infomap"]
SEEDS = [42, 52, 62, 72, 82, 92, 102, 112, 122, 132]
LOUVAIN_RESOLUTION = 1.0
LEIDEN_RESOLUTION = 1.0
INFOMAP_TRIALS = 10

# Regra PRÉ-ESPECIFICADA para selecionar configuração fused:
# preservar pelo menos 95% da modularidade do topology-only do mesmo algoritmo.
MIN_STRUCTURAL_RETENTION = 0.95

# Robustez
N_PERTURB = 50
TOPOLOGY_EDGE_DROPOUT = 0.10
SEMANTIC_NOISE_SIGMA = 0.03

# Modelos nulos
N_NULL = 100
DOUBLE_EDGE_SWAP_MULTIPLIER = 5

# Embeddings
EMBED_BATCH_SIZE_GPU = 128
EMBED_BATCH_SIZE_CPU = 32
MIN_TEXT_CHARS = 3
MATCHING_TOLERANCE = 0.02
ALLOW_MATCHING_AUDIT_FAILURE = False

# ============================================================
# 0.2 UTILITÁRIOS DE ARQUIVOS E RUNS
# ============================================================
def short_hash_file(path, n=12):
    path = Path(path)
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()[:n]


def save_json(obj, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, ensure_ascii=False, default=str)


def find_file_by_substring(folder, substring):
    folder = Path(folder)
    if not folder.exists():
        return None
    matches = [p for p in folder.iterdir() if substring.lower() in p.name.lower()]
    return sorted(matches)[0] if matches else None


def source_run_is_complete(run_path):
    run_path = Path(run_path)
    required = [
        run_path / "artifacts" / "graph_gcc.pkl",
        run_path / "intermediate" / "gcc_nodes.csv",
        run_path / "intermediate" / "nodes_clean.csv",
    ]
    if not all(p.exists() for p in required):
        return False
    prod = find_file_by_substring(run_path / "input", "producoes_final")
    return prod is not None and prod.exists()


def locate_source_run(project_root, preferred_name=None):
    runs_dir = Path(project_root) / "runs"
    if not runs_dir.exists():
        raise FileNotFoundError(f"Diretório de runs não encontrado: {runs_dir}")

    if preferred_name:
        preferred = runs_dir / preferred_name
        if source_run_is_complete(preferred):
            return preferred

    candidates = []
    for p in runs_dir.glob("run_*"):
        if p.is_dir() and source_run_is_complete(p):
            candidates.append(p)

    if not candidates:
        raise FileNotFoundError(
            "Nenhum run histórico completo foi encontrado. Necessário: "
            "artifacts/graph_gcc.pkl, intermediate/gcc_nodes.csv, "
            "intermediate/nodes_clean.csv e input/producoes_final.*"
        )

    # Mais recente pelo nome timestamp; fallback para mtime.
    candidates = sorted(candidates, key=lambda p: (p.name, p.stat().st_mtime), reverse=True)
    return candidates[0]


def make_output_run(project_root):
    out_base = Path(project_root) / "semantic_topological_runs"
    ts = datetime.now().strftime("%Y%m%d_%H%M%S")
    out = out_base / f"run_{ts}"
    for sub in ["intermediate", "tables", "figures", "artifacts", "logs", "cache"]:
        (out / sub).mkdir(parents=True, exist_ok=True)
    return out


SOURCE_RUN = locate_source_run(PROJECT_ROOT, PREFERRED_SOURCE_RUN)
OUT_RUN = make_output_run(PROJECT_ROOT)

print("=" * 80)
print("PARTE 16 — SEMANTIC-TOPOLOGICAL COMMUNITY DETECTION")
print("=" * 80)
print(f"Run histórico carregado: {SOURCE_RUN}")
print(f"Novo run da Parte 16:      {OUT_RUN}")

# ============================================================
# 1. CARREGAR ARTEFATOS HISTÓRICOS SEM REEXECUTAR PARTES 0–15
# ============================================================
print("\n" + "=" * 80)
print("1. CARREGANDO ARTEFATOS HISTÓRICOS")
print("=" * 80)

GRAPH_GCC_PATH = SOURCE_RUN / "artifacts" / "graph_gcc.pkl"
GCC_NODES_PATH = SOURCE_RUN / "intermediate" / "gcc_nodes.csv"
NODES_CLEAN_PATH = SOURCE_RUN / "intermediate" / "nodes_clean.csv"
PRODUCOES_PATH = find_file_by_substring(SOURCE_RUN / "input", "producoes_final")
OLD_PARTITION_PATH = SOURCE_RUN / "artifacts" / "best_partition.pkl"

with open(GRAPH_GCC_PATH, "rb") as f:
    G_top_raw = pickle.load(f)

# Garante grafo simples, não direcionado e com pesos numéricos positivos.
G_top_raw = nx.Graph(G_top_raw)
for u, v, d in G_top_raw.edges(data=True):
    w = d.get("weight", 1.0)
    try:
        w = float(w)
    except Exception:
        w = 1.0
    d["weight"] = max(w, 1e-12)

gcc_nodes_df = pd.read_csv(GCC_NODES_PATH)
nodes_clean_df = pd.read_csv(NODES_CLEAN_PATH)

if PRODUCOES_PATH.suffix.lower() in [".xlsx", ".xls"]:
    producoes_df = pd.read_excel(PRODUCOES_PATH)
else:
    producoes_df = pd.read_csv(PRODUCOES_PATH)

old_partition = None
if OLD_PARTITION_PATH.exists():
    try:
        with open(OLD_PARTITION_PATH, "rb") as f:
            old_partition = pickle.load(f)
    except Exception:
        old_partition = None

print(f"✅ GCC pré-backbone: {G_top_raw.number_of_nodes()} nós, {G_top_raw.number_of_edges()} arestas")
print(f"✅ gcc_nodes.csv: {len(gcc_nodes_df)} linhas")
print(f"✅ nodes_clean.csv: {len(nodes_clean_df)} linhas")
print(f"✅ producoes_final: {len(producoes_df)} linhas")
print(f"✅ Arquivo de produções: {PRODUCOES_PATH.name}")

# ============================================================
# 1.1 SNAPSHOT DE AMBIENTE E MANIFESTO DE ENTRADA
# ============================================================
def package_version(name, module=None):
    try:
        import importlib.metadata as md
        return md.version(name)
    except Exception:
        if module is not None:
            return getattr(module, "__version__", "unknown")
        return "unknown"

manifest = {
    "created_at": datetime.now().isoformat(timespec="seconds"),
    "source_run": str(SOURCE_RUN),
    "out_run": str(OUT_RUN),
    "inputs": {
        "graph_gcc": {"path": str(GRAPH_GCC_PATH), "sha256_short": short_hash_file(GRAPH_GCC_PATH)},
        "gcc_nodes": {"path": str(GCC_NODES_PATH), "sha256_short": short_hash_file(GCC_NODES_PATH)},
        "nodes_clean": {"path": str(NODES_CLEAN_PATH), "sha256_short": short_hash_file(NODES_CLEAN_PATH)},
        "producoes_final": {"path": str(PRODUCOES_PATH), "sha256_short": short_hash_file(PRODUCOES_PATH)},
    },
    "config": {
        "model_name": MODEL_NAME,
        "text_mode": TEXT_MODE,
        "semantic_scope": SEMANTIC_SCOPE,
        "knn_values": KNN_VALUES,
        "fusion_alphas": FUSION_ALPHAS,
        "algorithms": ALGORITHMS,
        "seeds": SEEDS,
        "min_structural_retention": MIN_STRUCTURAL_RETENTION,
        "n_perturb": N_PERTURB,
        "n_null": N_NULL,
    },
    "environment": {
        "python": sys.version,
        "platform": platform.platform(),
        "numpy": np.__version__,
        "pandas": pd.__version__,
        "networkx": nx.__version__,
        "scipy": scipy.__version__,
        "torch": torch.__version__,
        "sentence_transformers": package_version("sentence-transformers"),
        "python_louvain": package_version("python-louvain"),
        "igraph": package_version("igraph", ig),
        "leidenalg": package_version("leidenalg"),
        "cuda_available": bool(torch.cuda.is_available()),
        "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    },
}
save_json(manifest, OUT_RUN / "logs" / "manifest.json")

# Snapshot completo de dependências para o Link-to-Code/README do artigo.
try:
    freeze = subprocess.check_output([sys.executable, "-m", "pip", "freeze"], text=True)
    with open(OUT_RUN / "logs" / "requirements_freeze.txt", "w", encoding="utf-8") as f:
        f.write(freeze)
except Exception:
    pass

# Hardware adicional (quando disponível).
try:
    import psutil
    hardware = {
        "cpu_count_logical": os.cpu_count(),
        "ram_total_gb": round(psutil.virtual_memory().total / (1024**3), 2),
        "cuda_available": bool(torch.cuda.is_available()),
        "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    }
    save_json(hardware, OUT_RUN / "logs" / "hardware.json")
except Exception:
    pass

# ============================================================
# 2. IDENTIFICAR COLUNAS E VALIDAR UNIVERSO DE AUTORES
# ============================================================
print("\n" + "=" * 80)
print("2. AUDITORIA DE COLUNAS E MATCHING")
print("=" * 80)


def pick_first_present(columns, candidates):
    low = {str(c).lower(): c for c in columns}
    for cand in candidates:
        if cand.lower() in low:
            return low[cand.lower()]
    return None


def normalize_name_token_sort(s):
    import unicodedata
    if s is None or (isinstance(s, float) and np.isnan(s)):
        return ""
    s = str(s).strip().lower()
    s = "".join(c for c in unicodedata.normalize("NFD", s) if unicodedata.category(c) != "Mn")
    s = re.sub(r"[^a-z\s]", " ", s)
    toks = [t for t in s.split() if len(t) >= 2]
    toks.sort()
    return " ".join(toks)


def parse_authors_field(value):
    """
    Parser robusto, mas compatível com a lógica do laboratório anterior.
    Prioriza listas explícitas e separadores não ambíguos; usa vírgula apenas como fallback.
    """
    if value is None or (isinstance(value, float) and np.isnan(value)):
        return []

    if isinstance(value, (list, tuple, set)):
        raw = list(value)
    else:
        s = str(value).strip()
        if not s:
            return []

        # Tenta lista serializada: ['A', 'B'] ou ["A", "B"]
        raw = None
        if (s.startswith("[") and s.endswith("]")) or (s.startswith("(") and s.endswith(")")):
            try:
                parsed = ast.literal_eval(s)
                if isinstance(parsed, (list, tuple, set)):
                    raw = list(parsed)
            except Exception:
                raw = None

        if raw is None:
            # Prioriza delimitadores mais plausíveis entre autores.
            if ";" in s:
                raw = s.split(";")
            elif "|" in s:
                raw = s.split("|")
            elif "\n" in s:
                raw = s.splitlines()
            elif " / " in s:
                raw = s.split(" / ")
            else:
                # Compatibilidade com o laboratório antigo.
                raw = s.split(",")

    out = []
    for a in raw:
        n = normalize_name_token_sort(a)
        if n:
            out.append(n)
    return list(dict.fromkeys(out))


name_col_nodes = pick_first_present(gcc_nodes_df.columns, ["nome_tokensort_norm", "autor", "nome"])
name_col_all = pick_first_present(nodes_clean_df.columns, ["nome_tokensort_norm", "autor", "nome"])
inst_col = pick_first_present(
    gcc_nodes_df.columns,
    ["SG_ENTIDADE_ENSINO", "instituicao_original", "instituicao", "IES"]
)

title_col = pick_first_present(producoes_df.columns, ["titulo", "title"])
authors_col = pick_first_present(producoes_df.columns, ["autores", "authors", "lista_autores", "author_list"])
year_col = pick_first_present(producoes_df.columns, ["ano", "year", "publication_year"])
abstract_col = pick_first_present(producoes_df.columns, ["abstract", "resumo", "summary"])
doi_col = pick_first_present(producoes_df.columns, ["doi", "id", "work_id"])

if name_col_nodes is None or name_col_all is None:
    raise KeyError("Coluna de nome normalizado não localizada nos arquivos de nós.")
if title_col is None or authors_col is None:
    raise KeyError("producoes_final precisa conter pelo menos título e autores.")

full_ppgci_set = set(nodes_clean_df[name_col_all].dropna().astype(str))
gcc_set = set(G_top_raw.nodes())

# Confere se o CSV e o pkl representam o mesmo GCC.
gcc_csv_set = set(gcc_nodes_df[name_col_nodes].dropna().astype(str))
if gcc_csv_set != gcc_set:
    print("⚠️ Aviso: gcc_nodes.csv e graph_gcc.pkl não têm exatamente o mesmo conjunto de nós.")
    common = gcc_csv_set & gcc_set
    print(f"   Interseção: {len(common)} | CSV: {len(gcc_csv_set)} | PKL: {len(gcc_set)}")

print(f"Nome em gcc_nodes: {name_col_nodes}")
print(f"Instituição:        {inst_col}")
print(f"Título:             {title_col}")
print(f"Autores:            {authors_col}")
print(f"Ano:                {year_col}")
print(f"Resumo:              {abstract_col}")
print(f"DOI/ID:              {doi_col}")

# ============================================================
# 2.1 RECONSTRUIR MAPEAMENTO PUBLICAÇÃO -> PPGCI E AUDITAR CONTRA 8.129
# ============================================================
publication_records = []
count_ge2_full_ppgci = 0
count_ge2_gcc = 0
count_ge1_gcc = 0
parse_zero = 0

for idx, row in producoes_df.iterrows():
    authors_norm = parse_authors_field(row[authors_col])
    if not authors_norm:
        parse_zero += 1

    ppgci_authors = [a for a in authors_norm if a in full_ppgci_set]
    gcc_authors = [a for a in authors_norm if a in gcc_set]

    if len(ppgci_authors) >= 2:
        count_ge2_full_ppgci += 1
    if len(gcc_authors) >= 2:
        count_ge2_gcc += 1
    if len(gcc_authors) >= 1:
        count_ge1_gcc += 1

        title = "" if pd.isna(row[title_col]) else str(row[title_col]).strip()
        abstract = ""
        if abstract_col is not None and pd.notna(row[abstract_col]):
            abstract = str(row[abstract_col]).strip()

        if TEXT_MODE == "title_abstract" and abstract:
            text = f"{title}. {abstract}".strip()
        else:
            text = title

        if len(text) >= MIN_TEXT_CHARS:
            publication_records.append({
                "row_index": int(idx),
                "publication_id": str(row[doi_col]) if doi_col is not None and pd.notna(row[doi_col]) else f"row_{idx}",
                "title": title,
                "text": text,
                "year": row[year_col] if year_col is not None else None,
                "gcc_authors": gcc_authors,
                "n_gcc_authors": len(gcc_authors),
            })

# Recupera a contagem histórica diretamente do log da Parte 1 quando possível.
historical_target = 8129
ingestion_diag_path = SOURCE_RUN / "logs" / "ingestion_diagnostics.json"
if ingestion_diag_path.exists():
    try:
        with open(ingestion_diag_path, "r", encoding="utf-8") as f:
            _diag = json.load(f)
        for _key in ["publicacoes_com_>=2_ppgci", "publicacoes_com_≥2_ppgci", "publicacoes_ge2_ppgci"]:
            if _key in _diag:
                historical_target = int(_diag[_key])
                break
    except Exception:
        pass
relative_diff = abs(count_ge2_full_ppgci - historical_target) / historical_target

print(f"Publicações com >=2 autores PPGCI (auditoria atual): {count_ge2_full_ppgci}")
print(f"Referência do laboratório histórico:                  {historical_target}")
print(f"Diferença relativa:                                   {relative_diff:.2%}")
print(f"Publicações com >=2 autores do GCC:                   {count_ge2_gcc}")
print(f"Publicações com >=1 autor do GCC e texto válido:      {len(publication_records)}")

if relative_diff > MATCHING_TOLERANCE:
    msg = (
        f"A reconstrução do matching difere {relative_diff:.2%} da contagem histórica "
        f"({historical_target}). Isso excede a tolerância de {MATCHING_TOLERANCE:.0%}. "
        "Revise o formato/delimitador da coluna de autores antes de produzir resultados do artigo."
    )
    if ALLOW_MATCHING_AUDIT_FAILURE:
        print("⚠️ " + msg)
    else:
        raise RuntimeError(msg)

coverage_audit = pd.DataFrame([{
    "n_nodes_full_ppgci": len(full_ppgci_set),
    "n_nodes_gcc": len(gcc_set),
    "n_publications_total": len(producoes_df),
    "n_publications_ge2_full_ppgci": count_ge2_full_ppgci,
    "historical_reference_ge2": historical_target,
    "relative_difference_vs_historical": relative_diff,
    "n_publications_ge2_gcc": count_ge2_gcc,
    "n_publications_ge1_gcc_with_text": len(publication_records),
    "n_rows_without_parseable_authors": parse_zero,
    "text_mode": TEXT_MODE,
    "semantic_scope": SEMANTIC_SCOPE,
}])
coverage_audit.to_csv(OUT_RUN / "tables" / "table16_1_data_coverage.csv", index=False)

# Salva mapeamento em JSONL porque gcc_authors é lista.
with open(OUT_RUN / "intermediate" / "publication_author_mapping.jsonl", "w", encoding="utf-8") as f:
    for rec in publication_records:
        f.write(json.dumps(rec, ensure_ascii=False, default=str) + "\n")

# ============================================================
# 3. EMBEDDINGS DAS PUBLICAÇÕES E DOS PESQUISADORES
# ============================================================
print("\n" + "=" * 80)
print("3. GERANDO/CARREGANDO EMBEDDINGS SEMÂNTICOS")
print("=" * 80)

if not publication_records:
    raise RuntimeError("Nenhuma publicação com autor do GCC e texto válido foi encontrada.")

pub_df = pd.DataFrame(publication_records)

# Cada linha original de producoes_final é uma unidade documental.
# Evita duplicação acidental do mesmo row_index.
pub_df = pub_df.drop_duplicates(subset=["row_index"]).reset_index(drop=True)

# Cache depende do modelo, modo textual e hash do arquivo de produções.
cache_id = hashlib.sha256(
    f"{MODEL_NAME}|{TEXT_MODE}|{short_hash_file(PRODUCOES_PATH)}".encode("utf-8")
).hexdigest()[:16]
cache_emb = OUT_RUN / "cache" / f"publication_embeddings_{cache_id}.npy"
cache_meta = OUT_RUN / "cache" / f"publication_embeddings_{cache_id}.csv"

# Procura primeiro em cache persistente global do projeto para evitar recomputação futura.
GLOBAL_EMB_CACHE = PROJECT_ROOT / "semantic_embedding_cache"
GLOBAL_EMB_CACHE.mkdir(parents=True, exist_ok=True)
global_emb = GLOBAL_EMB_CACHE / f"publication_embeddings_{cache_id}.npy"
global_meta = GLOBAL_EMB_CACHE / f"publication_embeddings_{cache_id}.csv"

if global_emb.exists() and global_meta.exists():
    cached_meta = pd.read_csv(global_meta)
    if cached_meta["row_index"].tolist() == pub_df["row_index"].tolist():
        pub_embeddings = np.load(global_emb)
        print(f"✅ Embeddings carregados do cache global: {global_emb}")
    else:
        pub_embeddings = None
else:
    pub_embeddings = None

if pub_embeddings is None:
    device = "cuda" if torch.cuda.is_available() else "cpu"
    batch_size = EMBED_BATCH_SIZE_GPU if device == "cuda" else EMBED_BATCH_SIZE_CPU
    print(f"Modelo: {MODEL_NAME}")
    print(f"Dispositivo: {device}")
    print(f"Documentos: {len(pub_df)}")

    model = SentenceTransformer(
        MODEL_NAME,
        cache_folder=str(MODEL_CACHE),
        device=device,
    )

    pub_embeddings = model.encode(
        pub_df["text"].astype(str).tolist(),
        batch_size=batch_size,
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=True,
    ).astype(np.float32)

    np.save(global_emb, pub_embeddings)
    pub_df[["row_index", "publication_id", "title"]].to_csv(global_meta, index=False)
    print(f"✅ Cache global salvo: {global_emb}")

# Copia referência/cache também para o run da Parte 16.
np.save(cache_emb, pub_embeddings)
pub_df[["row_index", "publication_id", "title"]].to_csv(cache_meta, index=False)

# ---- Agregação publicação -> pesquisador ----
node_order_all = sorted(gcc_set)
node_to_idx_all = {n: i for i, n in enumerate(node_order_all)}
author_doc_indices = defaultdict(list)

for pub_pos, authors in enumerate(pub_df["gcc_authors"]):
    for a in authors:
        if a in gcc_set:
            author_doc_indices[a].append(pub_pos)

researcher_embeddings = []
researcher_rows = []
analysis_nodes = []

for author in node_order_all:
    docs = sorted(set(author_doc_indices.get(author, [])))
    if len(docs) == 0:
        continue

    vec = pub_embeddings[docs].mean(axis=0)
    norm = np.linalg.norm(vec)
    if norm <= 0:
        continue
    vec = vec / norm

    analysis_nodes.append(author)
    researcher_embeddings.append(vec.astype(np.float32))
    researcher_rows.append({
        "researcher": author,
        "n_semantic_documents": len(docs),
    })

researcher_embeddings = np.vstack(researcher_embeddings)
researchers_df = pd.DataFrame(researcher_rows)

semantic_coverage = len(analysis_nodes) / len(gcc_set)
print(f"✅ Pesquisadores com embedding: {len(analysis_nodes)}/{len(gcc_set)} ({semantic_coverage:.1%})")

if semantic_coverage < 0.95:
    print("⚠️ Cobertura semântica <95%. O artigo deverá reportar explicitamente a perda de nós.")

# Universo comum para TODAS as condições experimentais.
analysis_set = set(analysis_nodes)
G_top = G_top_raw.subgraph(analysis_nodes).copy()

# Garante mesma ordem em todo o módulo.
node_order = list(analysis_nodes)
node_to_idx = {n: i for i, n in enumerate(node_order)}

# Adiciona instituição, se disponível.
if inst_col is not None:
    inst_map = (
        gcc_nodes_df[[name_col_nodes, inst_col]]
        .drop_duplicates(subset=[name_col_nodes])
        .set_index(name_col_nodes)[inst_col]
        .to_dict()
    )
    researchers_df["institution"] = researchers_df["researcher"].map(inst_map)
else:
    researchers_df["institution"] = np.nan

researchers_df.to_csv(OUT_RUN / "intermediate" / "researcher_semantic_coverage.csv", index=False)
np.save(OUT_RUN / "artifacts" / "researcher_embeddings.npy", researcher_embeddings)

# Similaridade coseno: embeddings já L2-normalizados.
semantic_similarity = np.clip(researcher_embeddings @ researcher_embeddings.T, -1.0, 1.0)
np.fill_diagonal(semantic_similarity, 1.0)
np.save(OUT_RUN / "artifacts" / "researcher_cosine_similarity.npy", semantic_similarity.astype(np.float32))

# ============================================================
# 4. NORMALIZAR TOPOLOGIA E CONSTRUIR GRAFOS SEMÂNTICOS
# ============================================================
print("\n" + "=" * 80)
print("4. CONSTRUINDO CAMADAS TOPOLÓGICA E SEMÂNTICA")
print("=" * 80)


def normalize_graph_weight_mass(G, transform=None):
    """
    Normaliza a massa total de pesos para 1. Isso torna alpha interpretável
    como participação relativa das camadas na fusão, evitando que diferenças
    de escala ou número de arestas dominem a combinação.
    """
    H = nx.Graph()
    H.add_nodes_from(G.nodes())
    temp = []
    for u, v, d in G.edges(data=True):
        w_raw = float(d.get("weight", 1.0))
        w = transform(w_raw) if transform is not None else w_raw
        w = max(float(w), 0.0)
        if w > 0:
            temp.append((u, v, w, w_raw))
    total = sum(w for _, _, w, _ in temp)
    if total <= 0:
        return H
    for u, v, w, w_raw in temp:
        H.add_edge(u, v, weight=w / total, raw_weight=w_raw)
    return H


def normalize_topology_graph(G):
    # log1p reduz a dominância de hipercolaborações; a massa total é então normalizada.
    return normalize_graph_weight_mass(G, transform=lambda w: math.log1p(w))


G_top_norm = normalize_topology_graph(G_top)

with open(OUT_RUN / "artifacts" / "topology_graph_gcc_common_nodes.pkl", "wb") as f:
    pickle.dump(G_top_norm, f)


def build_semantic_knn_graph(nodes, sim_matrix, k, min_similarity=0.0):
    n = len(nodes)
    k_eff = min(k, n - 1)
    H = nx.Graph()
    H.add_nodes_from(nodes)

    for i, node in enumerate(nodes):
        sims = sim_matrix[i].copy()
        sims[i] = -np.inf
        # argpartition é eficiente e suficiente; ordena depois os k vizinhos.
        idx = np.argpartition(-sims, k_eff)[:k_eff]
        idx = idx[np.argsort(-sims[idx])]

        for j in idx:
            s = float(sims[j])
            if s < min_similarity:
                continue
            u, v = node, nodes[int(j)]
            if u == v:
                continue
            # Simetrização por máximo da similaridade observada.
            if H.has_edge(u, v):
                H[u][v]["weight"] = max(H[u][v]["weight"], s)
            else:
                H.add_edge(u, v, weight=s)

    # Normalização da massa da camada semântica: soma dos pesos = 1.
    return normalize_graph_weight_mass(H, transform=None)


semantic_graphs = {}
for k in KNN_VALUES:
    Gs = build_semantic_knn_graph(node_order, semantic_similarity, k, MIN_SEMANTIC_SIMILARITY)
    semantic_graphs[k] = Gs
    with open(OUT_RUN / "artifacts" / f"semantic_graph_k{k}.pkl", "wb") as f:
        pickle.dump(Gs, f)
    print(f"k={k}: {Gs.number_of_nodes()} nós, {Gs.number_of_edges()} arestas")

# ============================================================
# 5. FUSÃO SEMÂNTICO-TOPOLÓGICA
# ============================================================
def build_fused_graph(Gt, Gs, alpha):
    if not (0.0 <= alpha <= 1.0):
        raise ValueError("alpha deve estar em [0,1]")

    H = nx.Graph()
    H.add_nodes_from(Gt.nodes())
    edge_keys = set()
    edge_keys.update(tuple(sorted((u, v))) for u, v in Gt.edges())
    edge_keys.update(tuple(sorted((u, v))) for u, v in Gs.edges())

    for u, v in edge_keys:
        wt = Gt[u][v]["weight"] if Gt.has_edge(u, v) else 0.0
        ws = Gs[u][v]["weight"] if Gs.has_edge(u, v) else 0.0
        wf = alpha * wt + (1.0 - alpha) * ws
        if wf > 0:
            H.add_edge(u, v, weight=float(wf), topo_weight=float(wt), semantic_weight=float(ws))
    return H

# ============================================================
# 6. COMMUNITY DETECTION — LOUVAIN, LEIDEN E INFOMAP
# ============================================================
def nx_to_igraph(G):
    nodes = list(G.nodes())
    index = {n: i for i, n in enumerate(nodes)}
    edges = [(index[u], index[v]) for u, v in G.edges()]
    weights = [float(G[u][v].get("weight", 1.0)) for u, v in G.edges()]
    gi = ig.Graph(n=len(nodes), edges=edges, directed=False)
    gi.vs["name"] = nodes
    gi.es["weight"] = weights
    return gi, nodes


def relabel_partition(partition):
    # Remove rótulos arbitrários e torna a serialização estável.
    groups = defaultdict(list)
    for n, c in partition.items():
        groups[c].append(n)
    ordered = sorted(groups.values(), key=lambda g: (-len(g), sorted(map(str, g))[0]))
    out = {}
    for new_c, group in enumerate(ordered):
        for n in group:
            out[n] = new_c
    return out


def detect_partition(G, algorithm, seed):
    if G.number_of_edges() == 0:
        return {n: i for i, n in enumerate(G.nodes())}

    if algorithm == "louvain":
        p = community_louvain.best_partition(
            G,
            weight="weight",
            resolution=LOUVAIN_RESOLUTION,
            random_state=seed,
        )
        return relabel_partition(p)

    gi, nodes = nx_to_igraph(G)

    if algorithm == "leiden":
        part = la.find_partition(
            gi,
            la.ModularityVertexPartition,
            weights="weight",
            n_iterations=-1,
            seed=seed,
        )
        p = {nodes[i]: int(part.membership[i]) for i in range(len(nodes))}
        return relabel_partition(p)

    if algorithm == "infomap":
        rng = random.Random(seed)
        ig.set_random_number_generator(rng)
        part = gi.community_infomap(edge_weights="weight", trials=INFOMAP_TRIALS)
        p = {nodes[i]: int(part.membership[i]) for i in range(len(nodes))}
        return relabel_partition(p)

    raise ValueError(f"Algoritmo não reconhecido: {algorithm}")

# ============================================================
# 7. MÉTRICAS DE AVALIAÇÃO
# ============================================================
def partition_labels(partition, nodes):
    return np.array([partition[n] for n in nodes], dtype=int)


def partition_sets(partition):
    groups = defaultdict(set)
    for n, c in partition.items():
        groups[c].add(n)
    return list(groups.values())


def normalized_size_entropy(partition):
    sizes = np.array([len(c) for c in partition_sets(partition)], dtype=float)
    if len(sizes) <= 1:
        return 0.0
    p = sizes / sizes.sum()
    h = -np.sum(p * np.log(p + 1e-15))
    return float(h / np.log(len(sizes)))


def weighted_coverage(G, partition):
    total = sum(float(d.get("weight", 1.0)) for _, _, d in G.edges(data=True))
    if total <= 0:
        return np.nan
    internal = 0.0
    for u, v, d in G.edges(data=True):
        if partition.get(u) == partition.get(v):
            internal += float(d.get("weight", 1.0))
    return internal / total


def conductance_stats(G, partition):
    if G.number_of_edges() == 0:
        return np.nan, np.nan

    deg = dict(G.degree(weight="weight"))
    total_vol = sum(deg.values())
    vals = []
    weighted_vals = []

    for comm in partition_sets(partition):
        if len(comm) == 0 or len(comm) == G.number_of_nodes():
            continue
        vol_s = sum(deg.get(n, 0.0) for n in comm)
        vol_t = total_vol - vol_s
        denom = min(vol_s, vol_t)
        if denom <= 0:
            continue
        cut = 0.0
        for u in comm:
            for v, d in G[u].items():
                if v not in comm:
                    cut += float(d.get("weight", 1.0))
        phi = cut / denom
        vals.append(phi)
        weighted_vals.append((phi, len(comm)))

    if not vals:
        return np.nan, np.nan
    mean_phi = float(np.mean(vals))
    size_weighted = float(
        sum(phi * size for phi, size in weighted_vals) / sum(size for _, size in weighted_vals)
    )
    return mean_phi, size_weighted


def semantic_quality(partition, nodes, sim_matrix):
    labels = partition_labels(partition, nodes)
    n = len(nodes)

    within_sum = 0.0
    within_n = 0
    between_sum = 0.0
    between_n = 0

    for i in range(n):
        for j in range(i + 1, n):
            s = float(sim_matrix[i, j])
            if labels[i] == labels[j]:
                within_sum += s
                within_n += 1
            else:
                between_sum += s
                between_n += 1

    within = within_sum / within_n if within_n else np.nan
    between = between_sum / between_n if between_n else np.nan
    gap = within - between if np.isfinite(within) and np.isfinite(between) else np.nan

    # Coerência centroide-membro ponderada por pesquisador.
    emb = researcher_embeddings
    coherence_values = []
    for c in sorted(set(labels)):
        idx = np.where(labels == c)[0]
        if len(idx) == 0:
            continue
        centroid = emb[idx].mean(axis=0)
        norm = np.linalg.norm(centroid)
        if norm <= 0:
            continue
        centroid = centroid / norm
        vals = emb[idx] @ centroid
        coherence_values.extend(vals.tolist())
    centroid_coherence = float(np.mean(coherence_values)) if coherence_values else np.nan

    return {
        "semantic_centroid_coherence": centroid_coherence,
        "semantic_within_pairwise": float(within) if np.isfinite(within) else np.nan,
        "semantic_between_pairwise": float(between) if np.isfinite(between) else np.nan,
        "semantic_gap": float(gap) if np.isfinite(gap) else np.nan,
    }


def cramers_v_stats(partition, nodes, institution_map):
    rows = []
    for n in nodes:
        inst = institution_map.get(n, None)
        if inst is None or pd.isna(inst) or str(inst).strip() == "":
            continue
        rows.append((str(inst), int(partition[n])))

    if len(rows) < 2:
        return np.nan, np.nan, np.nan, 0

    df = pd.DataFrame(rows, columns=["institution", "community"])
    tab = pd.crosstab(df["institution"], df["community"])
    if tab.shape[0] < 2 or tab.shape[1] < 2:
        return np.nan, np.nan, np.nan, len(df)

    chi2, p, _, _ = chi2_contingency(tab, correction=False)
    n = tab.values.sum()
    r, k = tab.shape
    phi2 = chi2 / n
    v = math.sqrt(phi2 / max(1, min(k - 1, r - 1)))

    # Correção de viés de Bergsma/Wicher.
    phi2corr = max(0.0, phi2 - ((k - 1) * (r - 1)) / max(1, n - 1))
    rcorr = r - ((r - 1) ** 2) / max(1, n - 1)
    kcorr = k - ((k - 1) ** 2) / max(1, n - 1)
    denom = min(kcorr - 1, rcorr - 1)
    v_corr = math.sqrt(phi2corr / denom) if denom > 0 else np.nan

    return float(v), float(v_corr), float(p), int(n)


if inst_col is not None:
    institution_map = (
        gcc_nodes_df[[name_col_nodes, inst_col]]
        .drop_duplicates(subset=[name_col_nodes])
        .set_index(name_col_nodes)[inst_col]
        .to_dict()
    )
else:
    institution_map = {}


def evaluate_partition(partition, eval_topology_graph, nodes, sim_matrix):
    comms = partition_sets(partition)
    try:
        modularity = nx.algorithms.community.quality.modularity(
            eval_topology_graph, comms, weight="weight"
        )
    except Exception:
        modularity = np.nan

    coverage = weighted_coverage(eval_topology_graph, partition)
    conductance_mean, conductance_weighted = conductance_stats(eval_topology_graph, partition)
    sem = semantic_quality(partition, nodes, sim_matrix)
    v, v_corr, p, n_inst = cramers_v_stats(partition, nodes, institution_map)

    return {
        "n_communities": len(comms),
        "size_entropy_norm": normalized_size_entropy(partition),
        "structural_modularity": float(modularity) if np.isfinite(modularity) else np.nan,
        "structural_coverage": coverage,
        "conductance_mean": conductance_mean,
        "conductance_size_weighted": conductance_weighted,
        **sem,
        "institution_cramers_v": v,
        "institution_cramers_v_bias_corrected": v_corr,
        "institution_p": p,
        "institution_n": n_inst,
    }

# ============================================================
# 8. GRADE EXPERIMENTAL: TOPOLOGY / SEMANTIC / FUSED
# ============================================================
print("\n" + "=" * 80)
print("5. EXECUTANDO GRADE EXPERIMENTAL")
print("=" * 80)

conditions = []

# Topology-only é único; não depende de k.
conditions.append({
    "condition": "topology_only",
    "k": None,
    "alpha": 1.0,
})

# Semantic-only e fused dependem de k.
for k in KNN_VALUES:
    conditions.append({
        "condition": "semantic_only",
        "k": k,
        "alpha": 0.0,
    })
    for alpha in FUSION_ALPHAS:
        conditions.append({
            "condition": "fused",
            "k": k,
            "alpha": float(alpha),
        })


def graph_for_condition(cond):
    if cond["condition"] == "topology_only":
        return G_top_norm.copy()
    Gs = semantic_graphs[int(cond["k"])]
    if cond["condition"] == "semantic_only":
        return Gs.copy()
    return build_fused_graph(G_top_norm, Gs, float(cond["alpha"]))


run_rows = []
partitions_store = defaultdict(list)

for ci, cond in enumerate(conditions, start=1):
    Gexp = graph_for_condition(cond)
    cond_key = f"{cond['condition']}|k={cond['k']}|a={cond['alpha']}"
    print(f"[{ci}/{len(conditions)}] {cond_key}: {Gexp.number_of_edges()} arestas")

    for algorithm in ALGORITHMS:
        for seed in SEEDS:
            t0 = time.perf_counter()
            partition = detect_partition(Gexp, algorithm, seed)
            elapsed = time.perf_counter() - t0

            metrics = evaluate_partition(partition, G_top_norm, node_order, semantic_similarity)
            row = {
                "condition": cond["condition"],
                "k": cond["k"],
                "alpha": cond["alpha"],
                "algorithm": algorithm,
                "seed": seed,
                "graph_edges_used_by_detector": Gexp.number_of_edges(),
                "runtime_seconds": elapsed,
                **metrics,
            }
            run_rows.append(row)
            partitions_store[(cond_key, algorithm)].append(partition)

runs_df = pd.DataFrame(run_rows)
runs_df.to_csv(OUT_RUN / "tables" / "table16_2_grid_runs.csv", index=False)

# ============================================================
# 8.1 ESTABILIDADE ENTRE SEEDS E AGREGAÇÃO
# ============================================================
def pairwise_partition_stability(partitions, nodes):
    if len(partitions) < 2:
        return {"stability_nmi_mean": np.nan, "stability_nmi_sd": np.nan,
                "stability_ari_mean": np.nan, "stability_ari_sd": np.nan}
    nmis, aris = [], []
    for p1, p2 in combinations(partitions, 2):
        l1 = partition_labels(p1, nodes)
        l2 = partition_labels(p2, nodes)
        nmis.append(normalized_mutual_info_score(l1, l2))
        aris.append(adjusted_rand_score(l1, l2))
    return {
        "stability_nmi_mean": float(np.mean(nmis)),
        "stability_nmi_sd": float(np.std(nmis, ddof=1)) if len(nmis) > 1 else 0.0,
        "stability_ari_mean": float(np.mean(aris)),
        "stability_ari_sd": float(np.std(aris, ddof=1)) if len(aris) > 1 else 0.0,
    }


agg_rows = []
group_cols = ["condition", "k", "alpha", "algorithm"]
metric_cols = [
    "n_communities", "runtime_seconds",
    "structural_modularity", "structural_coverage",
    "conductance_mean", "conductance_size_weighted",
    "semantic_centroid_coherence", "semantic_within_pairwise",
    "semantic_between_pairwise", "semantic_gap",
    "size_entropy_norm",
    "institution_cramers_v", "institution_cramers_v_bias_corrected",
]

for keys, gdf in runs_df.groupby(group_cols, dropna=False):
    condition, k, alpha, algorithm = keys
    cond_key = f"{condition}|k={None if pd.isna(k) else int(k)}|a={alpha}"
    parts = partitions_store[(cond_key, algorithm)]
    stability = pairwise_partition_stability(parts, node_order)

    row = {
        "condition": condition,
        "k": None if pd.isna(k) else int(k),
        "alpha": float(alpha),
        "algorithm": algorithm,
        "n_runs": len(gdf),
    }
    for col in metric_cols:
        vals = pd.to_numeric(gdf[col], errors="coerce")
        row[f"{col}_mean"] = float(vals.mean())
        row[f"{col}_sd"] = float(vals.std(ddof=1)) if len(vals.dropna()) > 1 else 0.0
    row.update(stability)
    agg_rows.append(row)

agg_df = pd.DataFrame(agg_rows)
agg_df.to_csv(OUT_RUN / "tables" / "table16_3_grid_aggregated.csv", index=False)

# ============================================================
# 9. SELEÇÃO TRANSPARENTE SOB RESTRIÇÃO ESTRUTURAL
# ============================================================
print("\n" + "=" * 80)
print("6. SELEÇÃO DE CONFIGURAÇÕES FUSED SOB RESTRIÇÃO PRÉ-ESPECIFICADA")
print("=" * 80)

selected_rows = []

for algorithm in ALGORITHMS:
    baseline = agg_df[(agg_df["condition"] == "topology_only") & (agg_df["algorithm"] == algorithm)]
    if baseline.empty:
        continue
    base_mod = float(baseline.iloc[0]["structural_modularity_mean"])
    min_mod = MIN_STRUCTURAL_RETENTION * base_mod

    cand = agg_df[(agg_df["condition"] == "fused") & (agg_df["algorithm"] == algorithm)].copy()
    cand["structural_retention_ratio"] = cand["structural_modularity_mean"] / base_mod
    eligible = cand[cand["structural_modularity_mean"] >= min_mod].copy()

    if eligible.empty:
        # Não força uma conclusão positiva; registra o melhor trade-off disponível.
        eligible = cand.copy()
        eligible["selection_constraint_met"] = False
    else:
        eligible["selection_constraint_met"] = True

    eligible = eligible.sort_values(
        ["semantic_centroid_coherence_mean", "stability_nmi_mean", "conductance_mean_mean"],
        ascending=[False, False, True],
    )
    best = eligible.iloc[0].to_dict()
    best["baseline_structural_modularity"] = base_mod
    best["minimum_required_modularity"] = min_mod
    selected_rows.append(best)

selected_df = pd.DataFrame(selected_rows)
selected_df.to_csv(OUT_RUN / "tables" / "table16_4_selected_by_algorithm.csv", index=False)

print(selected_df[[
    "algorithm", "k", "alpha", "selection_constraint_met",
    "structural_modularity_mean", "semantic_centroid_coherence_mean",
    "stability_nmi_mean", "stability_ari_mean"
]].to_string(index=False))

# Seleção global para testes de robustez:
# 1) apenas configurações que cumpriram a restrição, se houver;
# 2) maior coerência semântica;
# 3) maior estabilidade NMI;
sel_pool = selected_df.copy()
if "selection_constraint_met" in sel_pool.columns and sel_pool["selection_constraint_met"].fillna(False).any():
    sel_pool = sel_pool[sel_pool["selection_constraint_met"].fillna(False)]

sel_pool = sel_pool.sort_values(
    ["semantic_centroid_coherence_mean", "stability_nmi_mean", "structural_modularity_mean"],
    ascending=[False, False, False],
)
selected_global = sel_pool.iloc[0].to_dict()

SELECTED_ALGO = str(selected_global["algorithm"])
SELECTED_K = int(selected_global["k"])
SELECTED_ALPHA = float(selected_global["alpha"])

print("\nConfiguração selecionada para robustez:")
print(f"  algoritmo = {SELECTED_ALGO}")
print(f"  k         = {SELECTED_K}")
print(f"  alpha     = {SELECTED_ALPHA}")

# Partição de referência: seed global.
G_sem_selected = semantic_graphs[SELECTED_K]
G_fused_selected = build_fused_graph(G_top_norm, G_sem_selected, SELECTED_ALPHA)
reference_partition = detect_partition(G_fused_selected, SELECTED_ALGO, GLOBAL_SEED)
reference_metrics = evaluate_partition(reference_partition, G_top_norm, node_order, semantic_similarity)

with open(OUT_RUN / "artifacts" / "selected_partition.pkl", "wb") as f:
    pickle.dump(reference_partition, f)
with open(OUT_RUN / "artifacts" / "selected_fused_graph.pkl", "wb") as f:
    pickle.dump(G_fused_selected, f)

save_json({
    "algorithm": SELECTED_ALGO,
    "k": SELECTED_K,
    "alpha": SELECTED_ALPHA,
    "selection_rule": (
        f"max semantic coherence among fused configurations preserving >= "
        f"{MIN_STRUCTURAL_RETENTION:.0%} of topology-only modularity for the same algorithm; "
        "tie-breakers: stability NMI, then lower conductance"
    ),
    "reference_metrics": reference_metrics,
}, OUT_RUN / "logs" / "selected_configuration.json")

# ============================================================
# 9.1 TABELA SELECTED VS BASELINES
# ============================================================
comparison_rows = []
for condition_name, k, alpha in [
    ("topology_only", None, 1.0),
    ("semantic_only", SELECTED_K, 0.0),
    ("fused_selected", SELECTED_K, SELECTED_ALPHA),
]:
    if condition_name == "fused_selected":
        src = agg_df[
            (agg_df["condition"] == "fused") &
            (agg_df["algorithm"] == SELECTED_ALGO) &
            (agg_df["k"] == SELECTED_K) &
            (np.isclose(agg_df["alpha"], SELECTED_ALPHA))
        ]
    elif condition_name == "semantic_only":
        src = agg_df[
            (agg_df["condition"] == "semantic_only") &
            (agg_df["algorithm"] == SELECTED_ALGO) &
            (agg_df["k"] == SELECTED_K)
        ]
    else:
        src = agg_df[
            (agg_df["condition"] == "topology_only") &
            (agg_df["algorithm"] == SELECTED_ALGO)
        ]
    if not src.empty:
        r = src.iloc[0].to_dict()
        r["comparison_condition"] = condition_name
        comparison_rows.append(r)

comparison_df = pd.DataFrame(comparison_rows)
comparison_df.to_csv(OUT_RUN / "tables" / "table16_5_selected_vs_baselines.csv", index=False)

# ============================================================
# 10. ROBUSTEZ A PERTURBAÇÕES
# ============================================================
print("\n" + "=" * 80)
print("7. ROBUSTEZ A PERTURBAÇÕES")
print("=" * 80)


def dropout_edges(G, fraction, rng):
    H = G.copy()
    edges = list(H.edges())
    n_drop = int(round(fraction * len(edges)))
    if n_drop <= 0:
        return H
    drop_idx = rng.choice(len(edges), size=min(n_drop, len(edges)), replace=False)
    H.remove_edges_from([edges[i] for i in drop_idx])
    return H


def noisy_semantic_similarity(embeddings, sigma, rng):
    noisy = embeddings + rng.normal(0.0, sigma, size=embeddings.shape).astype(np.float32)
    noisy = normalize(noisy, norm="l2", axis=1)
    sim = np.clip(noisy @ noisy.T, -1.0, 1.0)
    np.fill_diagonal(sim, 1.0)
    return sim

ref_labels = partition_labels(reference_partition, node_order)
perturb_rows = []

for b in range(N_PERTURB):
    seed = GLOBAL_SEED + 1000 + b
    rng = np.random.default_rng(seed)

    # A) Perturbação topológica
    Gt_p = dropout_edges(G_top_norm, TOPOLOGY_EDGE_DROPOUT, rng)
    Gf_p = build_fused_graph(Gt_p, G_sem_selected, SELECTED_ALPHA)
    p_top = detect_partition(Gf_p, SELECTED_ALGO, seed)
    lab_top = partition_labels(p_top, node_order)
    perturb_rows.append({
        "replicate": b,
        "perturbation": "topology_edge_dropout",
        "nmi_vs_reference": normalized_mutual_info_score(ref_labels, lab_top),
        "ari_vs_reference": adjusted_rand_score(ref_labels, lab_top),
        **evaluate_partition(p_top, G_top_norm, node_order, semantic_similarity),
    })

    # B) Perturbação semântica
    sim_p = noisy_semantic_similarity(researcher_embeddings, SEMANTIC_NOISE_SIGMA, rng)
    Gs_p = build_semantic_knn_graph(node_order, sim_p, SELECTED_K, MIN_SEMANTIC_SIMILARITY)
    Gf_s = build_fused_graph(G_top_norm, Gs_p, SELECTED_ALPHA)
    p_sem = detect_partition(Gf_s, SELECTED_ALGO, seed)
    lab_sem = partition_labels(p_sem, node_order)
    perturb_rows.append({
        "replicate": b,
        "perturbation": "semantic_embedding_noise",
        "nmi_vs_reference": normalized_mutual_info_score(ref_labels, lab_sem),
        "ari_vs_reference": adjusted_rand_score(ref_labels, lab_sem),
        **evaluate_partition(p_sem, G_top_norm, node_order, semantic_similarity),
    })

    # C) Perturbação combinada
    Gf_both = build_fused_graph(Gt_p, Gs_p, SELECTED_ALPHA)
    p_both = detect_partition(Gf_both, SELECTED_ALGO, seed)
    lab_both = partition_labels(p_both, node_order)
    perturb_rows.append({
        "replicate": b,
        "perturbation": "combined",
        "nmi_vs_reference": normalized_mutual_info_score(ref_labels, lab_both),
        "ari_vs_reference": adjusted_rand_score(ref_labels, lab_both),
        **evaluate_partition(p_both, G_top_norm, node_order, semantic_similarity),
    })

perturb_df = pd.DataFrame(perturb_rows)
perturb_df.to_csv(OUT_RUN / "tables" / "table16_6_perturbation_runs.csv", index=False)

perturb_summary = (
    perturb_df.groupby("perturbation")
    .agg(
        n=("replicate", "count"),
        nmi_mean=("nmi_vs_reference", "mean"),
        nmi_sd=("nmi_vs_reference", "std"),
        ari_mean=("ari_vs_reference", "mean"),
        ari_sd=("ari_vs_reference", "std"),
        modularity_mean=("structural_modularity", "mean"),
        semantic_coherence_mean=("semantic_centroid_coherence", "mean"),
    )
    .reset_index()
)
perturb_summary.to_csv(OUT_RUN / "tables" / "table16_6_perturbation_stability.csv", index=False)
print(perturb_summary.to_string(index=False))

# ============================================================
# 11. MODELOS NULOS
# ============================================================
print("\n" + "=" * 80)
print("8. MODELOS NULOS")
print("=" * 80)


def degree_preserving_random_graph(G, rng):
    """
    Preserva exatamente a sequência de graus (quando os swaps completam),
    e reatribui aleatoriamente o multiconjunto de pesos observados.
    """
    H = nx.Graph()
    H.add_nodes_from(G.nodes())
    H.add_edges_from(G.edges())

    m = H.number_of_edges()
    if m >= 2:
        nswap = DOUBLE_EDGE_SWAP_MULTIPLIER * m
        max_tries = max(nswap * 20, 1000)
        try:
            nx.double_edge_swap(H, nswap=nswap, max_tries=max_tries, seed=int(rng.integers(1, 2**31 - 1)))
        except Exception:
            # Mantém o que foi possível; em grafos restritivos alguns swaps podem falhar.
            pass

    weights = [float(d.get("weight", 1.0)) for _, _, d in G.edges(data=True)]
    rng.shuffle(weights)
    for (u, v), w in zip(H.edges(), weights):
        H[u][v]["weight"] = float(w)
    return H


def permuted_semantic_similarity(sim_matrix, rng):
    perm = rng.permutation(sim_matrix.shape[0])
    return sim_matrix[np.ix_(perm, perm)]


null_rows = []
obs = reference_metrics

for b in range(N_NULL):
    seed = GLOBAL_SEED + 5000 + b
    rng = np.random.default_rng(seed)

    # Null A: topologia reconfigurada, semântica real.
    Gt_null = degree_preserving_random_graph(G_top_norm, rng)
    Gf_null_t = build_fused_graph(Gt_null, G_sem_selected, SELECTED_ALPHA)
    p_null_t = detect_partition(Gf_null_t, SELECTED_ALGO, seed)
    metrics_t = evaluate_partition(p_null_t, Gt_null, node_order, semantic_similarity)
    null_rows.append({
        "replicate": b,
        "null_type": "degree_preserving_topology",
        **metrics_t,
    })

    # Null B: topologia real, semântica permutada entre pesquisadores.
    sim_null = permuted_semantic_similarity(semantic_similarity, rng)
    Gs_null = build_semantic_knn_graph(node_order, sim_null, SELECTED_K, MIN_SEMANTIC_SIMILARITY)
    Gf_null_s = build_fused_graph(G_top_norm, Gs_null, SELECTED_ALPHA)
    p_null_s = detect_partition(Gf_null_s, SELECTED_ALGO, seed)
    # Avaliamos coerência nos embeddings REAIS: o null quebra a associação identidade-semântica.
    metrics_s = evaluate_partition(p_null_s, G_top_norm, node_order, semantic_similarity)
    null_rows.append({
        "replicate": b,
        "null_type": "permuted_semantic_labels",
        **metrics_s,
    })

null_df = pd.DataFrame(null_rows)
null_df.to_csv(OUT_RUN / "tables" / "table16_7_null_model_runs.csv", index=False)


def z_score_vs_null(observed, values):
    values = pd.to_numeric(pd.Series(values), errors="coerce").dropna().values
    if len(values) < 2 or np.std(values, ddof=1) == 0:
        return np.nan
    return float((observed - np.mean(values)) / np.std(values, ddof=1))


null_summary_rows = []
for null_type, g in null_df.groupby("null_type"):
    row = {
        "null_type": null_type,
        "n": len(g),
        "modularity_null_mean": g["structural_modularity"].mean(),
        "modularity_null_sd": g["structural_modularity"].std(ddof=1),
        "modularity_observed": obs["structural_modularity"],
        "modularity_z": z_score_vs_null(obs["structural_modularity"], g["structural_modularity"]),
        "semantic_coherence_null_mean": g["semantic_centroid_coherence"].mean(),
        "semantic_coherence_null_sd": g["semantic_centroid_coherence"].std(ddof=1),
        "semantic_coherence_observed": obs["semantic_centroid_coherence"],
        "semantic_coherence_z": z_score_vs_null(obs["semantic_centroid_coherence"], g["semantic_centroid_coherence"]),
    }
    null_summary_rows.append(row)

null_summary = pd.DataFrame(null_summary_rows)
null_summary.to_csv(OUT_RUN / "tables" / "table16_7_null_model_summary.csv", index=False)
print(null_summary.to_string(index=False))

# ============================================================
# 12. VALIDAÇÃO EXTERNA INSTITUCIONAL — SOMENTE EX POST
# ============================================================
print("\n" + "=" * 80)
print("9. VALIDAÇÃO EXTERNA INSTITUCIONAL (EX POST)")
print("=" * 80)

external_rows = []

# Partições comparáveis: topology, semantic, fused selected.
for label, Gexp in [
    ("topology_only", G_top_norm),
    ("semantic_only", G_sem_selected),
    ("fused_selected", G_fused_selected),
]:
    p = detect_partition(Gexp, SELECTED_ALGO, GLOBAL_SEED)
    v, v_corr, pval, n_inst = cramers_v_stats(p, node_order, institution_map)
    external_rows.append({
        "condition": label,
        "algorithm": SELECTED_ALGO,
        "k": SELECTED_K if label != "topology_only" else None,
        "alpha": 1.0 if label == "topology_only" else (0.0 if label == "semantic_only" else SELECTED_ALPHA),
        "cramers_v": v,
        "cramers_v_bias_corrected": v_corr,
        "p_value": pval,
        "n_with_institution": n_inst,
    })

external_df = pd.DataFrame(external_rows)
external_df.to_csv(OUT_RUN / "tables" / "table16_8_external_institutional_validation.csv", index=False)
print(external_df.to_string(index=False))

# ============================================================
# 12.1 CONCORDÂNCIA ENTRE REPRESENTAÇÕES E ENTRE ALGORITMOS
# ============================================================
concordance_rows = []

# Mesma técnica de detecção; muda a representação.
repr_partitions = {}
repr_graphs = {
    "topology_only": G_top_norm,
    "semantic_only": G_sem_selected,
    "fused_selected": G_fused_selected,
}
for label, graph in repr_graphs.items():
    repr_partitions[label] = detect_partition(graph, SELECTED_ALGO, GLOBAL_SEED)

for a, b in combinations(repr_partitions.keys(), 2):
    la_ = partition_labels(repr_partitions[a], node_order)
    lb_ = partition_labels(repr_partitions[b], node_order)
    concordance_rows.append({
        "comparison_type": "representation",
        "item_a": a,
        "item_b": b,
        "algorithm": SELECTED_ALGO,
        "nmi": normalized_mutual_info_score(la_, lb_),
        "ari": adjusted_rand_score(la_, lb_),
    })

# Mesma representação fused selecionada; muda o algoritmo.
algo_partitions = {
    alg: detect_partition(G_fused_selected, alg, GLOBAL_SEED)
    for alg in ALGORITHMS
}
for a, b in combinations(ALGORITHMS, 2):
    la_ = partition_labels(algo_partitions[a], node_order)
    lb_ = partition_labels(algo_partitions[b], node_order)
    concordance_rows.append({
        "comparison_type": "algorithm",
        "item_a": a,
        "item_b": b,
        "algorithm": "multiple",
        "nmi": normalized_mutual_info_score(la_, lb_),
        "ari": adjusted_rand_score(la_, lb_),
    })

concordance_df = pd.DataFrame(concordance_rows)
concordance_df.to_csv(OUT_RUN / "tables" / "table16_10_partition_concordance.csv", index=False)

# ============================================================
# 13. FIGURAS DO ARTIGO
# ============================================================
print("\n" + "=" * 80)
print("10. GERANDO FIGURAS")
print("=" * 80)

# Figura 16.1 — trade-off estrutural-semântico
fig, ax = plt.subplots(figsize=(9, 6))
markers = {"louvain": "o", "leiden": "s", "infomap": "^"}
for algorithm in ALGORITHMS:
    d = agg_df[agg_df["algorithm"] == algorithm]
    ax.scatter(
        d["structural_modularity_mean"],
        d["semantic_centroid_coherence_mean"],
        label=algorithm,
        marker=markers.get(algorithm, "o"),
        alpha=0.8,
    )
ax.set_xlabel("Structural modularity on original collaboration graph")
ax.set_ylabel("Semantic centroid coherence")
ax.set_title("Structural–semantic trade-off across representations")
ax.legend()
fig.tight_layout()
fig.savefig(OUT_RUN / "figures" / "figure16_1_structural_semantic_tradeoff.png", dpi=400, bbox_inches="tight")
plt.close(fig)

# Figura 16.2 — alpha vs coerência semântica, configuração k selecionada
fig, ax = plt.subplots(figsize=(9, 6))
for algorithm in ALGORITHMS:
    d = agg_df[
        (agg_df["algorithm"] == algorithm) &
        (
            (agg_df["condition"] == "topology_only") |
            ((agg_df["k"] == SELECTED_K) & agg_df["condition"].isin(["semantic_only", "fused"]))
        )
    ].copy()
    d = d.sort_values("alpha")
    ax.plot(d["alpha"], d["semantic_centroid_coherence_mean"], marker="o", label=algorithm)
ax.set_xlabel("Fusion alpha (0 = semantic only; 1 = topology only)")
ax.set_ylabel("Semantic centroid coherence")
ax.set_title(f"Semantic coherence across fusion weights (k={SELECTED_K})")
ax.legend()
fig.tight_layout()
fig.savefig(OUT_RUN / "figures" / "figure16_2_alpha_semantic_coherence.png", dpi=400, bbox_inches="tight")
plt.close(fig)

# Figura 16.3 — robustez sob perturbação
fig, ax = plt.subplots(figsize=(9, 6))
order = ["topology_edge_dropout", "semantic_embedding_noise", "combined"]
vals = [perturb_summary.set_index("perturbation").loc[o, "nmi_mean"] for o in order]
errs = [perturb_summary.set_index("perturbation").loc[o, "nmi_sd"] for o in order]
ax.bar(order, vals, yerr=errs, capsize=4)
ax.set_ylabel("NMI vs reference partition")
ax.set_title("Partition robustness under perturbations")
ax.tick_params(axis="x", rotation=20)
fig.tight_layout()
fig.savefig(OUT_RUN / "figures" / "figure16_3_perturbation_stability.png", dpi=400, bbox_inches="tight")
plt.close(fig)

# Figura 16.4 — modelo nulo (coerência semântica)
fig, ax = plt.subplots(figsize=(9, 6))
for null_type, g in null_df.groupby("null_type"):
    ax.hist(g["semantic_centroid_coherence"].dropna(), bins=20, alpha=0.5, label=null_type)
ax.axvline(obs["semantic_centroid_coherence"], linestyle="--", linewidth=2, label="observed")
ax.set_xlabel("Semantic centroid coherence")
ax.set_ylabel("Frequency")
ax.set_title("Observed semantic coherence against null models")
ax.legend()
fig.tight_layout()
fig.savefig(OUT_RUN / "figures" / "figure16_4_null_semantic_coherence.png", dpi=400, bbox_inches="tight")
plt.close(fig)

# ============================================================
# 14. EXPORTAR PARTIÇÕES E PERFIL FINAL DE COMUNIDADES
# ============================================================
print("\n" + "=" * 80)
print("11. EXPORTANDO PERFIL DA PARTIÇÃO SELECIONADA")
print("=" * 80)

selected_labels = partition_labels(reference_partition, node_order)
profile_nodes = researchers_df.copy()
profile_nodes["community_selected"] = profile_nodes["researcher"].map(reference_partition)
profile_nodes.to_csv(OUT_RUN / "intermediate" / "selected_partition_nodes.csv", index=False)

community_profile_rows = []
for c in sorted(set(selected_labels)):
    members = [n for n in node_order if reference_partition[n] == c]
    idx = [node_to_idx[n] for n in members]
    sub = profile_nodes[profile_nodes["community_selected"] == c]

    inst_mode = None
    inst_share = np.nan
    if "institution" in sub.columns and sub["institution"].notna().any():
        vc = sub["institution"].dropna().astype(str).value_counts()
        if len(vc):
            inst_mode = vc.index[0]
            inst_share = vc.iloc[0] / vc.sum()

    if len(idx) >= 2:
        sm = semantic_similarity[np.ix_(idx, idx)]
        tri = sm[np.triu_indices(len(idx), k=1)]
        sem_within = float(np.mean(tri)) if len(tri) else np.nan
    else:
        sem_within = np.nan

    community_profile_rows.append({
        "community": c,
        "size": len(members),
        "mean_within_semantic_similarity": sem_within,
        "dominant_institution": inst_mode,
        "dominant_institution_share": inst_share,
    })

community_profile_df = pd.DataFrame(community_profile_rows)
community_profile_df.to_csv(OUT_RUN / "tables" / "table16_9_selected_community_profiles.csv", index=False)

# ============================================================
# 15. AUDITORIA FINAL E RESUMO PARA O ARTIGO
# ============================================================
summary = {
    "source_run": str(SOURCE_RUN),
    "output_run": str(OUT_RUN),
    "input_graph": "graph_gcc.pkl (pre-backbone)",
    "n_nodes_original_gcc": int(G_top_raw.number_of_nodes()),
    "n_edges_original_gcc": int(G_top_raw.number_of_edges()),
    "n_nodes_common_semantic_topological": int(len(node_order)),
    "semantic_node_coverage": float(semantic_coverage),
    "n_publications_semantic_scope": int(len(pub_df)),
    "embedding_model": MODEL_NAME,
    "text_mode": TEXT_MODE,
    "selected_algorithm": SELECTED_ALGO,
    "selected_k": SELECTED_K,
    "selected_alpha": SELECTED_ALPHA,
    "reference_metrics": reference_metrics,
    "historical_matching_audit": {
        "current_ge2_full_ppgci": int(count_ge2_full_ppgci),
        "historical_reference": historical_target,
        "relative_difference": float(relative_diff),
    },
}
save_json(summary, OUT_RUN / "logs" / "audit_part16_final.json")

# README textual para rastreabilidade.
readme = f"""
PARTE 16 — SEMANTIC-TOPOLOGICAL COMMUNITY DETECTION
====================================================

Fonte histórica:
{SOURCE_RUN}

Princípio metodológico:
- Topologia baseada no GCC pré-backbone (graph_gcc.pkl).
- Semântica baseada em embeddings multilíngues de publicações.
- Comparação topology-only, semantic-only e fused.
- Instituição usada somente em validação externa ex post.
- Seleção fused sem informação institucional.

Modelo semântico:
{MODEL_NAME}

Escopo textual:
{TEXT_MODE}

Configuração selecionada para robustez:
- Algoritmo: {SELECTED_ALGO}
- k semântico: {SELECTED_K}
- alpha: {SELECTED_ALPHA}

Regra de seleção:
Preservar >= {MIN_STRUCTURAL_RETENTION:.0%} da modularidade topology-only do mesmo algoritmo
 e, entre as configurações elegíveis, maximizar coerência semântica; desempates por NMI e condutância.

Principais saídas:
- tables/table16_2_grid_runs.csv
- tables/table16_3_grid_aggregated.csv
- tables/table16_4_selected_by_algorithm.csv
- tables/table16_5_selected_vs_baselines.csv
- tables/table16_6_perturbation_stability.csv
- tables/table16_7_null_model_summary.csv
- tables/table16_8_external_institutional_validation.csv
- tables/table16_9_selected_community_profiles.csv
- tables/table16_10_partition_concordance.csv
- figures/figure16_1_structural_semantic_tradeoff.png
- figures/figure16_2_alpha_semantic_coherence.png
- figures/figure16_3_perturbation_stability.png
- figures/figure16_4_null_semantic_coherence.png
- artifacts/selected_partition.pkl
- artifacts/selected_fused_graph.pkl
- logs/manifest.json
- logs/audit_part16_final.json
""".strip()

with open(OUT_RUN / "README_PART16.txt", "w", encoding="utf-8") as f:
    f.write(readme)

print("\n" + "=" * 80)
print("✅ PARTE 16 CONCLUÍDA")
print("=" * 80)
print(readme)
print("\nResumo da partição selecionada:")
for k, v in reference_metrics.items():
    print(f"  {k}: {v}")


Mounted at /content/drive
PARTE 16 — SEMANTIC-TOPOLOGICAL COMMUNITY DETECTION
Run histórico carregado: /content/drive/MyDrive/tese_doutoral/runs/run_20260425_002158
Novo run da Parte 16:      /content/drive/MyDrive/tese_doutoral/semantic_topological_runs/run_20260920_010404

1. CARREGANDO ARTEFATOS HISTÓRICOS
✅ GCC pré-backbone: 404 nós, 1609 arestas
✅ gcc_nodes.csv: 404 linhas
✅ nodes_clean.csv: 510 linhas
✅ producoes_final: 38138 linhas
✅ Arquivo de produções: producoes_final.xlsx

2. AUDITORIA DE COLUNAS E MATCHING
Nome em gcc_nodes: nome_tokensort_norm
Instituição:        SG_ENTIDADE_ENSINO
Título:             titulo
Autores:            autores
Ano:                ano
Resumo:              None
DOI/ID:              doi
Publicações com >=2 autores PPGCI (auditoria atual): 8122
Referência do laboratório histórico:                  8129
Diferença relativa:                                   0.09%
Publicações com >=2 autores do GCC:                   8102
Publicações com >=1 autor do GCC

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/5.12k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/847 [00:00<?, ?it/s]

✅ Cache global salvo: /content/drive/MyDrive/tese_doutoral/semantic_embedding_cache/publication_embeddings_fb997efcd929b2ff.npy
✅ Pesquisadores com embedding: 404/404 (100.0%)

4. CONSTRUINDO CAMADAS TOPOLÓGICA E SEMÂNTICA
k=5: 404 nós, 1644 arestas
k=10: 404 nós, 3228 arestas
k=15: 404 nós, 4781 arestas
k=20: 404 nós, 6352 arestas

5. EXECUTANDO GRADE EXPERIMENTAL
[1/17] topology_only|k=None|a=1.0: 1609 arestas
[2/17] semantic_only|k=5|a=0.0: 1644 arestas
[3/17] fused|k=5|a=0.25: 2901 arestas
[4/17] fused|k=5|a=0.5: 2901 arestas
[5/17] fused|k=5|a=0.75: 2901 arestas
[6/17] semantic_only|k=10|a=0.0: 3228 arestas
[7/17] fused|k=10|a=0.25: 4323 arestas
[8/17] fused|k=10|a=0.5: 4323 arestas
[9/17] fused|k=10|a=0.75: 4323 arestas
[10/17] semantic_only|k=15|a=0.0: 4781 arestas
[11/17] fused|k=15|a=0.25: 5788 arestas
[12/17] fused|k=15|a=0.5: 5788 arestas
[13/17] fused|k=15|a=0.75: 5788 arestas
[14/17] semantic_only|k=20|a=0.0: 6352 arestas
[15/17] fused|k=20|a=0.25: 7261 arestas
[16/17] fus

In [ ]:
# ============================================================
# PARTE 16.1 — VALIDAÇÃO PÓS-HOC DA INTEGRAÇÃO SEMÂNTICO-TOPOLÓGICA
# ============================================================
# Objetivo:
#   Auditar a Parte 16 SEM recalcular embeddings e SEM reexecutar as Partes 0–15.
#
# Controles adicionados:
#   1) coerência semântica ajustada à granularidade/tamanho das comunidades;
#   2) p-valores empíricos para os modelos nulos (sem depender de z-scores extremos);
#   3) validação institucional por permutação;
#   4) comparação justa entre Louvain, Leiden e Infomap, sem eleger vencedor global.
#
# Entrada principal:
#   /content/drive/MyDrive/tese_doutoral/semantic_topological_runs/<run Parte 16>
#
# A Parte 16.1 cria um sub-run próprio dentro do run da Parte 16.
# ============================================================

# ============================================================
# 0. INSTALAÇÃO / IMPORTS / CONFIGURAÇÃO
# ============================================================
import os
import sys
import json
import hashlib
import math
import time
import pickle
import random
import platform
import subprocess
import importlib.util
from pathlib import Path
from datetime import datetime
from collections import defaultdict
from itertools import combinations


def _ensure_package(import_name, pip_name=None):
    if importlib.util.find_spec(import_name) is None:
        pkg = pip_name or import_name
        print(f"Instalando dependência ausente: {pkg}")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])


for imp, pipn in [
    ("community", "python-louvain"),
    ("igraph", "igraph"),
    ("leidenalg", "leidenalg"),
]:
    _ensure_package(imp, pipn)

import numpy as np
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
from scipy.stats import chi2
from sklearn.metrics import normalized_mutual_info_score, adjusted_rand_score
import community as community_louvain
import igraph as ig
import leidenalg as la

# ---- Google Drive ----
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)

print("=" * 88)
print("PARTE 16.1 — VALIDAÇÃO PÓS-HOC SEMÂNTICO-TOPOLÓGICA")
print("=" * 88)

# -------------------------
# CONFIGURAÇÃO PRINCIPAL
# -------------------------
PROJECT_ROOT = Path("/content/drive/MyDrive/tese_doutoral")
SEMANTIC_RUNS_ROOT = PROJECT_ROOT / "semantic_topological_runs"

# Run que acabou de ser executado. Se não existir, o código localizará o run completo mais recente.
PREFERRED_PART16_RUN = "run_20260920_010404"

GLOBAL_SEED = 42

# Ajuste da coerência semântica pela granularidade da partição.
# 999 -> menor p empírico possível = 1/1000 = 0,001.
N_SIZE_MATCHED = 999

# Teste de permutação Comunidade × Instituição.
# 4999 -> menor p empírico possível = 1/5000 = 0,0002.
N_INSTITUTION_PERM = 4999

# Reexecutar os dois modelos nulos da Parte 16 com resolução empírica maior.
# Não recalcula embeddings. Se quiser apenas usar os 100 nulos já salvos, mude para False.
RUN_EXTENDED_NULLS = True
N_NULL_EXTENDED = 999
DOUBLE_EDGE_SWAP_MULTIPLIER = 5

# Parâmetros herdados da Parte 16, caso o manifest não esteja disponível.
FALLBACK_ALGORITHMS = ["louvain", "leiden", "infomap"]
LOUVAIN_RESOLUTION = 1.0
LEIDEN_RESOLUTION = 1.0
INFOMAP_TRIALS = 10
MIN_SEMANTIC_SIMILARITY = 0.0

# ============================================================
# 1. UTILITÁRIOS DE ARQUIVO / LOCALIZAÇÃO DO RUN
# ============================================================


def stable_seed_offset(text, modulo=100000):
    h = hashlib.sha256(str(text).encode("utf-8")).hexdigest()
    return int(h[:12], 16) % modulo

def save_json(obj, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, ensure_ascii=False, default=str)


def load_json(path, default=None):
    path = Path(path)
    if not path.exists():
        return default
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def part16_run_is_complete(run_path):
    run_path = Path(run_path)
    required = [
        run_path / "artifacts" / "researcher_embeddings.npy",
        run_path / "artifacts" / "researcher_cosine_similarity.npy",
        run_path / "artifacts" / "topology_graph_gcc_common_nodes.pkl",
        run_path / "intermediate" / "researcher_semantic_coverage.csv",
        run_path / "tables" / "table16_3_grid_aggregated.csv",
        run_path / "tables" / "table16_4_selected_by_algorithm.csv",
        run_path / "tables" / "table16_7_null_model_runs.csv",
        run_path / "logs" / "selected_configuration.json",
    ]
    return all(p.exists() for p in required)


def locate_part16_run(root, preferred_name=None):
    root = Path(root)
    if not root.exists():
        raise FileNotFoundError(f"Diretório não encontrado: {root}")

    if preferred_name:
        preferred = root / preferred_name
        if part16_run_is_complete(preferred):
            return preferred

    candidates = [p for p in root.iterdir() if p.is_dir() and p.name.startswith("run_")]
    candidates = sorted(candidates, key=lambda p: p.stat().st_mtime, reverse=True)
    for p in candidates:
        if part16_run_is_complete(p):
            return p

    raise FileNotFoundError(
        "Nenhum run completo da Parte 16 foi localizado. "
        "Verifique /content/drive/MyDrive/tese_doutoral/semantic_topological_runs/."
    )


PART16_RUN = locate_part16_run(SEMANTIC_RUNS_ROOT, PREFERRED_PART16_RUN)
OUT_RUN = PART16_RUN / "posthoc_16_1" / f"run_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
for sub in ["tables", "figures", "logs", "artifacts", "intermediate"]:
    (OUT_RUN / sub).mkdir(parents=True, exist_ok=True)

print(f"Run da Parte 16 carregado: {PART16_RUN}")
print(f"Novo run da Parte 16.1:      {OUT_RUN}")

# ============================================================
# 2. CARREGAR ARTEFATOS DA PARTE 16
# ============================================================
print("\n" + "=" * 88)
print("1. CARREGANDO ARTEFATOS DA PARTE 16")
print("=" * 88)

manifest = load_json(PART16_RUN / "logs" / "manifest.json", default={}) or {}
selected_config = load_json(PART16_RUN / "logs" / "selected_configuration.json", default={}) or {}

researchers_df = pd.read_csv(PART16_RUN / "intermediate" / "researcher_semantic_coverage.csv")
if "researcher" not in researchers_df.columns:
    raise ValueError("researcher_semantic_coverage.csv não contém a coluna 'researcher'.")

node_order = researchers_df["researcher"].astype(str).tolist()
node_to_idx = {n: i for i, n in enumerate(node_order)}

researcher_embeddings = np.load(PART16_RUN / "artifacts" / "researcher_embeddings.npy")
semantic_similarity = np.load(PART16_RUN / "artifacts" / "researcher_cosine_similarity.npy")

with open(PART16_RUN / "artifacts" / "topology_graph_gcc_common_nodes.pkl", "rb") as f:
    G_top = pickle.load(f)

agg_df = pd.read_csv(PART16_RUN / "tables" / "table16_3_grid_aggregated.csv")
selected_df = pd.read_csv(PART16_RUN / "tables" / "table16_4_selected_by_algorithm.csv")
existing_null_df = pd.read_csv(PART16_RUN / "tables" / "table16_7_null_model_runs.csv")

algorithms = manifest.get("config", {}).get("algorithms", FALLBACK_ALGORITHMS)
seeds = manifest.get("config", {}).get("seeds", [42, 52, 62, 72, 82, 92, 102, 112, 122, 132])

SELECTED_GLOBAL_ALGO = str(selected_config.get("algorithm", "infomap"))
SELECTED_GLOBAL_K = int(selected_config.get("k", 5))
SELECTED_GLOBAL_ALPHA = float(selected_config.get("alpha", 0.75))

if researcher_embeddings.shape[0] != len(node_order):
    raise ValueError("Número de embeddings não coincide com researcher_semantic_coverage.csv.")
if semantic_similarity.shape != (len(node_order), len(node_order)):
    raise ValueError("Matriz de similaridade não coincide com o universo de pesquisadores.")
if set(G_top.nodes()) != set(node_order):
    raise ValueError("Nós do grafo topológico não coincidem com o universo semântico da Parte 16.")

# Reordena o grafo implicitamente apenas pelo uso dos nomes; métricas vetoriais seguem node_order.
print(f"✅ Pesquisadores: {len(node_order)}")
print(f"✅ Embeddings: {researcher_embeddings.shape}")
print(f"✅ Matriz semântica: {semantic_similarity.shape}")
print(f"✅ Topologia comum: {G_top.number_of_nodes()} nós, {G_top.number_of_edges()} arestas")
print(f"✅ Configuração global anterior: {SELECTED_GLOBAL_ALGO}, k={SELECTED_GLOBAL_K}, alpha={SELECTED_GLOBAL_ALPHA}")

# Instituição
if "institution" in researchers_df.columns:
    institution_series = researchers_df["institution"].copy()
else:
    institution_series = pd.Series([np.nan] * len(researchers_df))

# ============================================================
# 3. FUNÇÕES DE GRAFO / COMMUNITY DETECTION
# ============================================================
def build_fused_graph(Gt, Gs, alpha):
    if not (0.0 <= alpha <= 1.0):
        raise ValueError("alpha deve estar em [0,1]")
    H = nx.Graph()
    H.add_nodes_from(Gt.nodes())
    edge_keys = set(tuple(sorted((u, v))) for u, v in Gt.edges())
    edge_keys.update(tuple(sorted((u, v))) for u, v in Gs.edges())
    for u, v in edge_keys:
        wt = float(Gt[u][v].get("weight", 1.0)) if Gt.has_edge(u, v) else 0.0
        ws = float(Gs[u][v].get("weight", 1.0)) if Gs.has_edge(u, v) else 0.0
        wf = alpha * wt + (1.0 - alpha) * ws
        if wf > 0:
            H.add_edge(u, v, weight=float(wf), topo_weight=wt, semantic_weight=ws)
    return H


def normalize_graph_weight_mass(G):
    H = nx.Graph()
    H.add_nodes_from(G.nodes())
    total = sum(max(float(d.get("weight", 1.0)), 0.0) for _, _, d in G.edges(data=True))
    if total <= 0:
        return H
    for u, v, d in G.edges(data=True):
        w = max(float(d.get("weight", 1.0)), 0.0)
        if w > 0:
            H.add_edge(u, v, weight=w / total)
    return H


def build_semantic_knn_graph(nodes, sim_matrix, k, min_similarity=0.0):
    n = len(nodes)
    k_eff = min(int(k), n - 1)
    H = nx.Graph()
    H.add_nodes_from(nodes)
    for i, node in enumerate(nodes):
        sims = sim_matrix[i].copy()
        sims[i] = -np.inf
        idx = np.argpartition(-sims, k_eff)[:k_eff]
        idx = idx[np.argsort(-sims[idx])]
        for j in idx:
            s = float(sims[j])
            if s < min_similarity:
                continue
            u, v = node, nodes[int(j)]
            if H.has_edge(u, v):
                H[u][v]["weight"] = max(float(H[u][v]["weight"]), s)
            else:
                H.add_edge(u, v, weight=s)
    return normalize_graph_weight_mass(H)


def load_semantic_graph(k):
    p = PART16_RUN / "artifacts" / f"semantic_graph_k{int(k)}.pkl"
    if p.exists():
        with open(p, "rb") as f:
            return pickle.load(f)
    # fallback sem recodificar embeddings
    return build_semantic_knn_graph(node_order, semantic_similarity, int(k), MIN_SEMANTIC_SIMILARITY)


def nx_to_igraph(G):
    nodes = list(G.nodes())
    index = {n: i for i, n in enumerate(nodes)}
    edges = [(index[u], index[v]) for u, v in G.edges()]
    weights = [float(G[u][v].get("weight", 1.0)) for u, v in G.edges()]
    gi = ig.Graph(n=len(nodes), edges=edges, directed=False)
    gi.vs["name"] = nodes
    gi.es["weight"] = weights
    return gi, nodes


def relabel_partition(partition):
    groups = defaultdict(list)
    for n, c in partition.items():
        groups[c].append(n)
    ordered = sorted(groups.values(), key=lambda g: (-len(g), sorted(map(str, g))[0]))
    out = {}
    for new_c, group in enumerate(ordered):
        for n in group:
            out[n] = new_c
    return out


def detect_partition(G, algorithm, seed):
    if G.number_of_edges() == 0:
        return {n: i for i, n in enumerate(G.nodes())}

    if algorithm == "louvain":
        p = community_louvain.best_partition(
            G, weight="weight", resolution=LOUVAIN_RESOLUTION, random_state=int(seed)
        )
        return relabel_partition(p)

    gi, nodes = nx_to_igraph(G)

    if algorithm == "leiden":
        part = la.find_partition(
            gi,
            la.ModularityVertexPartition,
            weights="weight",
            n_iterations=-1,
            seed=int(seed),
        )
        return relabel_partition({nodes[i]: int(part.membership[i]) for i in range(len(nodes))})

    if algorithm == "infomap":
        rng = random.Random(int(seed))
        ig.set_random_number_generator(rng)
        part = gi.community_infomap(edge_weights="weight", trials=INFOMAP_TRIALS)
        return relabel_partition({nodes[i]: int(part.membership[i]) for i in range(len(nodes))})

    raise ValueError(f"Algoritmo não reconhecido: {algorithm}")


def partition_labels(partition):
    return np.array([int(partition[n]) for n in node_order], dtype=int)


def groups_from_labels(labels):
    groups = []
    for c in np.unique(labels):
        groups.append(np.flatnonzero(labels == c))
    return groups

# ============================================================
# 4. MÉTRICAS E NULOS DE GRANULARIDADE
# ============================================================
# Soma global dos pares uma única vez: acelera semantic_gap.
_TRIU_I, _TRIU_J = np.triu_indices(len(node_order), k=1)
_TOTAL_PAIR_SUM = float(semantic_similarity[_TRIU_I, _TRIU_J].sum())
_TOTAL_PAIR_N = int(len(_TRIU_I))


def semantic_metrics_from_labels(labels):
    groups = groups_from_labels(labels)

    # 1) Coerência membro-centroide
    centroid_vals = []
    for idx in groups:
        if len(idx) == 0:
            continue
        centroid = researcher_embeddings[idx].mean(axis=0)
        norm = np.linalg.norm(centroid)
        if norm <= 0:
            continue
        centroid = centroid / norm
        centroid_vals.extend((researcher_embeddings[idx] @ centroid).tolist())
    centroid_coherence = float(np.mean(centroid_vals)) if centroid_vals else np.nan

    # 2) Similaridade intra vs inter (usa matriz pré-computada)
    within_sum = 0.0
    within_n = 0
    for idx in groups:
        m = len(idx)
        if m < 2:
            continue
        block = semantic_similarity[np.ix_(idx, idx)]
        ii, jj = np.triu_indices(m, k=1)
        within_sum += float(block[ii, jj].sum())
        within_n += int(len(ii))

    between_sum = _TOTAL_PAIR_SUM - within_sum
    between_n = _TOTAL_PAIR_N - within_n
    within = within_sum / within_n if within_n else np.nan
    between = between_sum / between_n if between_n else np.nan
    gap = within - between if np.isfinite(within) and np.isfinite(between) else np.nan

    return {
        "semantic_centroid_coherence": centroid_coherence,
        "semantic_within_pairwise": float(within),
        "semantic_between_pairwise": float(between),
        "semantic_gap": float(gap),
    }


def structural_metrics(partition):
    labels = partition_labels(partition)
    communities = [set(node_order[i] for i in idx) for idx in groups_from_labels(labels)]
    modularity = nx.algorithms.community.quality.modularity(G_top, communities, weight="weight")

    total_w = sum(float(d.get("weight", 1.0)) for _, _, d in G_top.edges(data=True))
    internal_w = sum(
        float(d.get("weight", 1.0))
        for u, v, d in G_top.edges(data=True)
        if partition[u] == partition[v]
    )
    coverage = internal_w / total_w if total_w > 0 else np.nan

    deg = dict(G_top.degree(weight="weight"))
    total_vol = sum(deg.values())
    conds = []
    conds_weighted = []
    for comm in communities:
        if len(comm) == 0 or len(comm) == G_top.number_of_nodes():
            continue
        vol_s = sum(deg.get(n, 0.0) for n in comm)
        vol_t = total_vol - vol_s
        denom = min(vol_s, vol_t)
        if denom <= 0:
            continue
        cut = 0.0
        for u in comm:
            for v, d in G_top[u].items():
                if v not in comm:
                    cut += float(d.get("weight", 1.0))
        phi = cut / denom
        conds.append(phi)
        conds_weighted.append((phi, len(comm)))

    conductance_mean = float(np.mean(conds)) if conds else np.nan
    conductance_size_weighted = (
        float(sum(phi * size for phi, size in conds_weighted) / sum(size for _, size in conds_weighted))
        if conds_weighted else np.nan
    )

    return {
        "n_communities": len(communities),
        "structural_modularity": float(modularity),
        "structural_coverage": float(coverage),
        "conductance_mean": conductance_mean,
        "conductance_size_weighted": conductance_size_weighted,
    }


def empirical_p(observed, null_values, alternative="greater"):
    vals = np.asarray(null_values, dtype=float)
    vals = vals[np.isfinite(vals)]
    if len(vals) == 0 or not np.isfinite(observed):
        return np.nan
    if alternative == "greater":
        return float((1 + np.sum(vals >= observed)) / (len(vals) + 1))
    if alternative == "less":
        return float((1 + np.sum(vals <= observed)) / (len(vals) + 1))
    if alternative == "two-sided":
        pg = (1 + np.sum(vals >= observed)) / (len(vals) + 1)
        pl = (1 + np.sum(vals <= observed)) / (len(vals) + 1)
        return float(min(1.0, 2.0 * min(pg, pl)))
    raise ValueError("alternative deve ser greater, less ou two-sided")


def safe_z(observed, vals):
    vals = np.asarray(vals, dtype=float)
    vals = vals[np.isfinite(vals)]
    if len(vals) < 2:
        return np.nan
    sd = vals.std(ddof=1)
    if sd <= 0:
        return np.nan
    return float((observed - vals.mean()) / sd)


def size_matched_semantic_null(labels, n_rep=N_SIZE_MATCHED, seed=GLOBAL_SEED):
    """
    Embaralha os pesquisadores entre comunidades mantendo EXATAMENTE
    o número de comunidades e o tamanho de cada comunidade observada.
    Isso remove a vantagem mecânica de partições mais granulares.
    """
    rng = np.random.default_rng(int(seed))
    sizes = [len(idx) for idx in groups_from_labels(labels)]
    n = len(labels)

    centroid_null = np.empty(n_rep, dtype=float)
    gap_null = np.empty(n_rep, dtype=float)

    for b in range(n_rep):
        perm = rng.permutation(n)
        rand_labels = np.empty(n, dtype=int)
        pos = 0
        for c, size in enumerate(sizes):
            idx = perm[pos:pos + size]
            rand_labels[idx] = c
            pos += size
        met = semantic_metrics_from_labels(rand_labels)
        centroid_null[b] = met["semantic_centroid_coherence"]
        gap_null[b] = met["semantic_gap"]

    obs = semantic_metrics_from_labels(labels)
    centroid_mean = float(np.mean(centroid_null))
    gap_mean = float(np.mean(gap_null))

    # Medida descritiva de ganho sobre o espaço disponível até o teto 1.
    headroom_denom = max(1e-12, 1.0 - centroid_mean)

    summary = {
        **obs,
        "size_null_n": int(n_rep),
        "size_null_centroid_mean": centroid_mean,
        "size_null_centroid_sd": float(np.std(centroid_null, ddof=1)),
        "centroid_excess_over_size_null": float(obs["semantic_centroid_coherence"] - centroid_mean),
        "centroid_headroom_adjusted_gain": float((obs["semantic_centroid_coherence"] - centroid_mean) / headroom_denom),
        "centroid_size_null_z": safe_z(obs["semantic_centroid_coherence"], centroid_null),
        "centroid_size_null_p_emp": empirical_p(obs["semantic_centroid_coherence"], centroid_null, "greater"),
        "size_null_gap_mean": gap_mean,
        "size_null_gap_sd": float(np.std(gap_null, ddof=1)),
        "semantic_gap_excess_over_size_null": float(obs["semantic_gap"] - gap_mean),
        "semantic_gap_size_null_z": safe_z(obs["semantic_gap"], gap_null),
        "semantic_gap_size_null_p_emp": empirical_p(obs["semantic_gap"], gap_null, "greater"),
    }
    return summary, centroid_null, gap_null

# ============================================================
# 5. RECONSTRUIR TOPOLOGY / SEMANTIC / FUSED POR ALGORITMO
# ============================================================
print("\n" + "=" * 88)
print("2. AJUSTE DA COERÊNCIA SEMÂNTICA POR GRANULARIDADE")
print("=" * 88)

partition_records = []
partition_objects = {}
size_null_arrays = {}

for _, sel in selected_df.iterrows():
    algorithm = str(sel["algorithm"])
    k = int(round(float(sel["k"])))
    alpha = float(sel["alpha"])
    G_sem = load_semantic_graph(k)
    G_fused = build_fused_graph(G_top, G_sem, alpha)

    graphs = {
        "topology_only": G_top,
        "semantic_only": G_sem,
        "fused_selected": G_fused,
    }

    for condition, graph in graphs.items():
        key = (algorithm, condition)
        partition = detect_partition(graph, algorithm, GLOBAL_SEED)
        labels = partition_labels(partition)
        structural = structural_metrics(partition)
        semantic_adj, cent_null, gap_null = size_matched_semantic_null(
            labels,
            n_rep=N_SIZE_MATCHED,
            seed=GLOBAL_SEED + stable_seed_offset(f"{algorithm}|{condition}"),
        )

        record = {
            "algorithm": algorithm,
            "condition": condition,
            "k": None if condition == "topology_only" else k,
            "alpha": 1.0 if condition == "topology_only" else (0.0 if condition == "semantic_only" else alpha),
            **structural,
            **semantic_adj,
        }
        partition_records.append(record)
        partition_objects[key] = partition
        size_null_arrays[key] = {"centroid": cent_null, "gap": gap_null}
        print(
            f"{algorithm:8s} | {condition:15s} | K={structural['n_communities']:2d} | "
            f"Q={structural['structural_modularity']:.4f} | "
            f"SC={semantic_adj['semantic_centroid_coherence']:.4f} | "
            f"SC_excess={semantic_adj['centroid_excess_over_size_null']:.4f} | "
            f"p_emp={semantic_adj['centroid_size_null_p_emp']:.4f}"
        )

semantic_adjusted_df = pd.DataFrame(partition_records)
semantic_adjusted_df.to_csv(
    OUT_RUN / "tables" / "table16_1a_granularity_adjusted_semantics.csv", index=False
)

# ============================================================
# 5.1 GANHO FUSED VS TOPOLOGY — COMPARAÇÃO INTRA-ALGORITMO
# ============================================================
gain_rows = []
for algorithm in algorithms:
    sub = semantic_adjusted_df[semantic_adjusted_df["algorithm"] == algorithm]
    if sub.empty:
        continue
    topo = sub[sub["condition"] == "topology_only"]
    fused = sub[sub["condition"] == "fused_selected"]
    sem = sub[sub["condition"] == "semantic_only"]
    if topo.empty or fused.empty:
        continue
    topo = topo.iloc[0]
    fused = fused.iloc[0]
    semrow = sem.iloc[0] if not sem.empty else None

    gain_rows.append({
        "algorithm": algorithm,
        "selected_k": fused["k"],
        "selected_alpha": fused["alpha"],
        "topology_K": topo["n_communities"],
        "fused_K": fused["n_communities"],
        "structural_retention_ratio": fused["structural_modularity"] / topo["structural_modularity"] if topo["structural_modularity"] != 0 else np.nan,
        "delta_modularity_fused_minus_topology": fused["structural_modularity"] - topo["structural_modularity"],
        "delta_raw_centroid_coherence": fused["semantic_centroid_coherence"] - topo["semantic_centroid_coherence"],
        "delta_size_adjusted_centroid_excess": fused["centroid_excess_over_size_null"] - topo["centroid_excess_over_size_null"],
        "delta_size_adjusted_semantic_gap_excess": fused["semantic_gap_excess_over_size_null"] - topo["semantic_gap_excess_over_size_null"],
        "topology_centroid_excess": topo["centroid_excess_over_size_null"],
        "fused_centroid_excess": fused["centroid_excess_over_size_null"],
        "semantic_only_centroid_excess": semrow["centroid_excess_over_size_null"] if semrow is not None else np.nan,
        "fused_centroid_p_emp": fused["centroid_size_null_p_emp"],
        "fused_gap_p_emp": fused["semantic_gap_size_null_p_emp"],
    })

gain_df = pd.DataFrame(gain_rows)
gain_df.to_csv(
    OUT_RUN / "tables" / "table16_1b_fused_vs_topology_adjusted_gain.csv", index=False
)

print("\nFused vs topology — ganho ajustado por granularidade:")
print(gain_df.to_string(index=False))

# ============================================================
# 6. VALIDAÇÃO INSTITUCIONAL POR PERMUTAÇÃO
# ============================================================
print("\n" + "=" * 88)
print("3. VALIDAÇÃO INSTITUCIONAL POR PERMUTAÇÃO")
print("=" * 88)

valid_inst_mask = institution_series.notna() & (institution_series.astype(str).str.strip() != "")
inst_nodes = researchers_df.loc[valid_inst_mask, "researcher"].astype(str).tolist()
inst_values = researchers_df.loc[valid_inst_mask, "institution"].astype(str).values
inst_categories, inst_codes = np.unique(inst_values, return_inverse=True)
node_idx_inst = np.array([node_to_idx[n] for n in inst_nodes], dtype=int)


def cramers_v_from_codes(inst_code_arr, comm_code_arr, bias_corrected=True):
    inst_code_arr = np.asarray(inst_code_arr, dtype=int)
    comm_code_arr = np.asarray(comm_code_arr, dtype=int)
    _, comm_codes = np.unique(comm_code_arr, return_inverse=True)
    r = int(inst_code_arr.max()) + 1
    k = int(comm_codes.max()) + 1
    n = len(inst_code_arr)
    if r < 2 or k < 2 or n < 2:
        return np.nan

    tab = np.bincount(inst_code_arr * k + comm_codes, minlength=r * k).reshape(r, k).astype(float)
    row = tab.sum(axis=1, keepdims=True)
    col = tab.sum(axis=0, keepdims=True)
    expected = row @ col / n
    mask = expected > 0
    chi2_stat = float((((tab - expected) ** 2) / np.where(mask, expected, 1.0))[mask].sum())
    phi2 = chi2_stat / n

    if not bias_corrected:
        return float(math.sqrt(phi2 / max(1, min(k - 1, r - 1))))

    phi2corr = max(0.0, phi2 - ((k - 1) * (r - 1)) / max(1, n - 1))
    rcorr = r - ((r - 1) ** 2) / max(1, n - 1)
    kcorr = k - ((k - 1) ** 2) / max(1, n - 1)
    denom = min(kcorr - 1, rcorr - 1)
    return float(math.sqrt(phi2corr / denom)) if denom > 0 else np.nan


def institution_permutation_test(partition, n_perm=N_INSTITUTION_PERM, seed=GLOBAL_SEED):
    labels_all = partition_labels(partition)
    comm_labels = labels_all[node_idx_inst]
    _, comm_codes = np.unique(comm_labels, return_inverse=True)
    observed_v = cramers_v_from_codes(inst_codes, comm_codes, bias_corrected=False)
    observed_v_corr = cramers_v_from_codes(inst_codes, comm_codes, bias_corrected=True)

    rng = np.random.default_rng(int(seed))
    null_corr = np.empty(n_perm, dtype=float)
    for b in range(n_perm):
        perm_inst = rng.permutation(inst_codes)
        null_corr[b] = cramers_v_from_codes(perm_inst, comm_codes, bias_corrected=True)

    return {
        "institution_n": len(comm_codes),
        "institution_cramers_v": observed_v,
        "institution_cramers_v_bias_corrected": observed_v_corr,
        "institution_perm_n": int(n_perm),
        "institution_null_vcorr_mean": float(np.mean(null_corr)),
        "institution_null_vcorr_sd": float(np.std(null_corr, ddof=1)),
        "institution_vcorr_z": safe_z(observed_v_corr, null_corr),
        "institution_vcorr_p_emp": empirical_p(observed_v_corr, null_corr, "greater"),
        "institution_null_vcorr_q95": float(np.quantile(null_corr, 0.95)),
        "institution_null_vcorr_q99": float(np.quantile(null_corr, 0.99)),
    }, null_corr


institution_rows = []
institution_null_store = {}
for algorithm in algorithms:
    for condition in ["topology_only", "semantic_only", "fused_selected"]:
        key = (algorithm, condition)
        if key not in partition_objects:
            continue
        result, null_arr = institution_permutation_test(
            partition_objects[key],
            n_perm=N_INSTITUTION_PERM,
            seed=GLOBAL_SEED + stable_seed_offset(f"inst|{algorithm}|{condition}"),
        )
        row = {"algorithm": algorithm, "condition": condition, **result}
        institution_rows.append(row)
        institution_null_store[key] = null_arr
        print(
            f"{algorithm:8s} | {condition:15s} | "
            f"Vcorr={result['institution_cramers_v_bias_corrected']:.4f} | "
            f"p_emp={result['institution_vcorr_p_emp']:.5f}"
        )

institution_perm_df = pd.DataFrame(institution_rows)
institution_perm_df.to_csv(
    OUT_RUN / "tables" / "table16_1c_institution_permutation.csv", index=False
)

# ============================================================
# 7. MODELOS NULOS — P-VALORES EMPÍRICOS
# ============================================================
print("\n" + "=" * 88)
print("4. MODELOS NULOS COM P-VALORES EMPÍRICOS")
print("=" * 88)


def degree_preserving_random_graph(G, rng):
    H = nx.Graph()
    H.add_nodes_from(G.nodes())
    H.add_edges_from(G.edges())
    m = H.number_of_edges()
    if m >= 2:
        nswap = DOUBLE_EDGE_SWAP_MULTIPLIER * m
        max_tries = max(nswap * 20, 1000)
        try:
            nx.double_edge_swap(
                H,
                nswap=nswap,
                max_tries=max_tries,
                seed=int(rng.integers(1, 2**31 - 1)),
            )
        except Exception:
            pass
    weights = np.array([float(d.get("weight", 1.0)) for _, _, d in G.edges(data=True)], dtype=float)
    rng.shuffle(weights)
    for (u, v), w in zip(H.edges(), weights):
        H[u][v]["weight"] = float(w)
    return H


def evaluate_on_graph(partition, eval_graph):
    labels = partition_labels(partition)
    communities = [set(node_order[i] for i in idx) for idx in groups_from_labels(labels)]
    q = nx.algorithms.community.quality.modularity(eval_graph, communities, weight="weight")
    sem = semantic_metrics_from_labels(labels)
    return float(q), float(sem["semantic_centroid_coherence"]), float(sem["semantic_gap"])


def run_extended_nulls(n_null=N_NULL_EXTENDED):
    G_sem = load_semantic_graph(SELECTED_GLOBAL_K)
    G_fused = build_fused_graph(G_top, G_sem, SELECTED_GLOBAL_ALPHA)
    p_obs = detect_partition(G_fused, SELECTED_GLOBAL_ALGO, GLOBAL_SEED)
    q_obs, sc_obs, gap_obs = evaluate_on_graph(p_obs, G_top)

    rows = []
    for b in range(n_null):
        seed = GLOBAL_SEED + 50000 + b
        rng = np.random.default_rng(seed)

        # A) Topologia degree-preserving + semântica real
        Gt_null = degree_preserving_random_graph(G_top, rng)
        Gf_null_t = build_fused_graph(Gt_null, G_sem, SELECTED_GLOBAL_ALPHA)
        p_t = detect_partition(Gf_null_t, SELECTED_GLOBAL_ALGO, seed)
        q_t, sc_t, gap_t = evaluate_on_graph(p_t, Gt_null)
        rows.append({
            "replicate": b,
            "null_type": "degree_preserving_topology",
            "structural_modularity": q_t,
            "semantic_centroid_coherence": sc_t,
            "semantic_gap": gap_t,
        })

        # B) Topologia real + semântica permutada entre identidades
        perm = rng.permutation(len(node_order))
        sim_null = semantic_similarity[np.ix_(perm, perm)]
        Gs_null = build_semantic_knn_graph(node_order, sim_null, SELECTED_GLOBAL_K, MIN_SEMANTIC_SIMILARITY)
        Gf_null_s = build_fused_graph(G_top, Gs_null, SELECTED_GLOBAL_ALPHA)
        p_s = detect_partition(Gf_null_s, SELECTED_GLOBAL_ALGO, seed)
        q_s, sc_s, gap_s = evaluate_on_graph(p_s, G_top)
        rows.append({
            "replicate": b,
            "null_type": "permuted_semantic_labels",
            "structural_modularity": q_s,
            "semantic_centroid_coherence": sc_s,
            "semantic_gap": gap_s,
        })

        if (b + 1) % 100 == 0 or (b + 1) == n_null:
            print(f"  nulos estendidos: {b + 1}/{n_null}")

    return pd.DataFrame(rows), {
        "structural_modularity": q_obs,
        "semantic_centroid_coherence": sc_obs,
        "semantic_gap": gap_obs,
    }


if RUN_EXTENDED_NULLS:
    null_df, observed_global = run_extended_nulls(N_NULL_EXTENDED)
    null_source = "extended_part16_1"
else:
    null_df = existing_null_df.copy()
    with open(PART16_RUN / "artifacts" / "selected_partition.pkl", "rb") as f:
        p_obs = pickle.load(f)
    q_obs, sc_obs, gap_obs = evaluate_on_graph(p_obs, G_top)
    observed_global = {
        "structural_modularity": q_obs,
        "semantic_centroid_coherence": sc_obs,
        "semantic_gap": gap_obs,
    }
    null_source = "existing_part16_100_replicates"

null_df.to_csv(OUT_RUN / "tables" / "table16_1d_null_runs_empirical.csv", index=False)

null_summary_rows = []
for null_type, g in null_df.groupby("null_type"):
    qvals = pd.to_numeric(g["structural_modularity"], errors="coerce").dropna().values
    scvals = pd.to_numeric(g["semantic_centroid_coherence"], errors="coerce").dropna().values
    gapvals = pd.to_numeric(g["semantic_gap"], errors="coerce").dropna().values if "semantic_gap" in g.columns else np.array([])

    row = {
        "null_source": null_source,
        "null_type": null_type,
        "n": len(g),
        "modularity_observed": observed_global["structural_modularity"],
        "modularity_null_mean": float(np.mean(qvals)),
        "modularity_null_sd": float(np.std(qvals, ddof=1)),
        "modularity_p_greater_emp": empirical_p(observed_global["structural_modularity"], qvals, "greater"),
        "modularity_p_two_sided_emp": empirical_p(observed_global["structural_modularity"], qvals, "two-sided"),
        "semantic_coherence_observed": observed_global["semantic_centroid_coherence"],
        "semantic_coherence_null_mean": float(np.mean(scvals)),
        "semantic_coherence_null_sd": float(np.std(scvals, ddof=1)),
        "semantic_coherence_p_greater_emp": empirical_p(observed_global["semantic_centroid_coherence"], scvals, "greater"),
        "semantic_gap_observed": observed_global["semantic_gap"],
        "semantic_gap_null_mean": float(np.mean(gapvals)) if len(gapvals) else np.nan,
        "semantic_gap_null_sd": float(np.std(gapvals, ddof=1)) if len(gapvals) > 1 else np.nan,
        "semantic_gap_p_greater_emp": empirical_p(observed_global["semantic_gap"], gapvals, "greater") if len(gapvals) else np.nan,
    }
    null_summary_rows.append(row)

null_empirical_df = pd.DataFrame(null_summary_rows)
null_empirical_df.to_csv(
    OUT_RUN / "tables" / "table16_1e_null_model_empirical_pvalues.csv", index=False
)
print(null_empirical_df.to_string(index=False))



def as_bool(v, default=True):
    if pd.isna(v):
        return default
    if isinstance(v, (bool, np.bool_)):
        return bool(v)
    if isinstance(v, str):
        return v.strip().lower() in {"true", "1", "yes", "y"}
    return bool(v)

# ============================================================
# 8. TABELA INTEGRADA — COMPARAÇÃO JUSTA ENTRE ALGORITMOS
# ============================================================
print("\n" + "=" * 88)
print("5. COMPARAÇÃO JUSTA ENTRE ALGORITMOS")
print("=" * 88)

fair_rows = []
for algorithm in algorithms:
    fused = semantic_adjusted_df[
        (semantic_adjusted_df["algorithm"] == algorithm) &
        (semantic_adjusted_df["condition"] == "fused_selected")
    ]
    topo = semantic_adjusted_df[
        (semantic_adjusted_df["algorithm"] == algorithm) &
        (semantic_adjusted_df["condition"] == "topology_only")
    ]
    inst = institution_perm_df[
        (institution_perm_df["algorithm"] == algorithm) &
        (institution_perm_df["condition"] == "fused_selected")
    ]
    grid_sel = selected_df[selected_df["algorithm"] == algorithm]
    if fused.empty or topo.empty or grid_sel.empty:
        continue
    f = fused.iloc[0]
    t = topo.iloc[0]
    s = grid_sel.iloc[0]
    i = inst.iloc[0] if not inst.empty else None

    fair_rows.append({
        "algorithm": algorithm,
        "selected_k": int(round(float(s["k"]))),
        "selected_alpha": float(s["alpha"]),
        "selection_constraint_met": as_bool(s.get("selection_constraint_met", True)),
        "fused_n_communities_seed42": int(f["n_communities"]),
        "topology_n_communities_seed42": int(t["n_communities"]),
        "fused_modularity_seed42": float(f["structural_modularity"]),
        "topology_modularity_seed42": float(t["structural_modularity"]),
        "structural_retention_ratio_seed42": float(f["structural_modularity"] / t["structural_modularity"]),
        "fused_centroid_coherence_raw": float(f["semantic_centroid_coherence"]),
        "fused_centroid_size_null_mean": float(f["size_null_centroid_mean"]),
        "fused_centroid_excess_size_adjusted": float(f["centroid_excess_over_size_null"]),
        "fused_centroid_headroom_adjusted_gain": float(f["centroid_headroom_adjusted_gain"]),
        "fused_centroid_p_emp": float(f["centroid_size_null_p_emp"]),
        "fused_semantic_gap": float(f["semantic_gap"]),
        "fused_semantic_gap_excess_size_adjusted": float(f["semantic_gap_excess_over_size_null"]),
        "fused_semantic_gap_p_emp": float(f["semantic_gap_size_null_p_emp"]),
        "delta_size_adjusted_centroid_excess_vs_topology": float(f["centroid_excess_over_size_null"] - t["centroid_excess_over_size_null"]),
        "stability_nmi_mean_part16": float(s.get("stability_nmi_mean", np.nan)),
        "stability_ari_mean_part16": float(s.get("stability_ari_mean", np.nan)),
        "conductance_mean_part16": float(s.get("conductance_mean_mean", np.nan)),
        "institution_vcorr_fused": float(i["institution_cramers_v_bias_corrected"]) if i is not None else np.nan,
        "institution_perm_p_emp": float(i["institution_vcorr_p_emp"]) if i is not None else np.nan,
    })

fair_df = pd.DataFrame(fair_rows)
fair_df.to_csv(
    OUT_RUN / "tables" / "table16_1f_algorithm_fair_comparison.csv", index=False
)
print(fair_df.to_string(index=False))

# Consenso de parâmetros — NÃO define vencedor global.
consensus_k = len(set(fair_df["selected_k"].dropna().tolist())) == 1 if not fair_df.empty else False
consensus_alpha = len(set(np.round(fair_df["selected_alpha"].dropna().values, 10).tolist())) == 1 if not fair_df.empty else False

consensus_df = pd.DataFrame([{
    "algorithms_compared": ", ".join(fair_df["algorithm"].tolist()),
    "all_algorithms_same_k": consensus_k,
    "consensus_k": int(fair_df["selected_k"].iloc[0]) if consensus_k and not fair_df.empty else np.nan,
    "all_algorithms_same_alpha": consensus_alpha,
    "consensus_alpha": float(fair_df["selected_alpha"].iloc[0]) if consensus_alpha and not fair_df.empty else np.nan,
    "global_winner_selected": False,
    "reporting_principle": (
        "Report algorithm-specific solutions in parallel; do not rank algorithms by raw semantic coherence because community granularity differs."
    ),
}])
consensus_df.to_csv(
    OUT_RUN / "tables" / "table16_1g_algorithm_consensus.csv", index=False
)

# ============================================================
# 9. CONCORDÂNCIA DAS PARTIÇÕES FUSED AJUSTADAS
# ============================================================
concordance_rows = []
for a, b in combinations(algorithms, 2):
    ka = (a, "fused_selected")
    kb = (b, "fused_selected")
    if ka not in partition_objects or kb not in partition_objects:
        continue
    la_ = partition_labels(partition_objects[ka])
    lb_ = partition_labels(partition_objects[kb])
    concordance_rows.append({
        "algorithm_a": a,
        "algorithm_b": b,
        "nmi": float(normalized_mutual_info_score(la_, lb_)),
        "ari": float(adjusted_rand_score(la_, lb_)),
    })

concordance_df = pd.DataFrame(concordance_rows)
concordance_df.to_csv(
    OUT_RUN / "tables" / "table16_1h_fused_algorithm_concordance.csv", index=False
)

# ============================================================
# 10. FIGURAS
# ============================================================
print("\n" + "=" * 88)
print("6. GERANDO FIGURAS DA PARTE 16.1")
print("=" * 88)

# Fig. 16.1a — modularidade vs coerência semântica ajustada
fig, ax = plt.subplots(figsize=(9, 6))
markers = {"louvain": "o", "leiden": "s", "infomap": "^"}
for algorithm in algorithms:
    sub = semantic_adjusted_df[
        (semantic_adjusted_df["algorithm"] == algorithm) &
        (semantic_adjusted_df["condition"] == "fused_selected")
    ]
    if sub.empty:
        continue
    ax.scatter(
        sub["structural_modularity"],
        sub["centroid_excess_over_size_null"],
        marker=markers.get(algorithm, "o"),
        s=90,
        label=algorithm.capitalize(),
    )
    for _, r in sub.iterrows():
        ax.annotate(
            f"K={int(r['n_communities'])}",
            (r["structural_modularity"], r["centroid_excess_over_size_null"]),
            xytext=(5, 5), textcoords="offset points", fontsize=8,
        )
ax.set_xlabel("Structural modularity on co-authorship graph")
ax.set_ylabel("Semantic coherence excess over size-matched null")
ax.set_title("Fused partitions after granularity adjustment")
ax.legend()
ax.grid(alpha=0.2)
fig.tight_layout()
fig.savefig(OUT_RUN / "figures" / "figure16_1a_adjusted_semantic_tradeoff.png", dpi=400, bbox_inches="tight")
plt.close(fig)

# Fig. 16.1b — ganho ajustado fused - topology
fig, ax = plt.subplots(figsize=(8, 5))
if not gain_df.empty:
    ax.bar(gain_df["algorithm"], gain_df["delta_size_adjusted_centroid_excess"])
ax.axhline(0, linewidth=1)
ax.set_xlabel("Community-detection algorithm")
ax.set_ylabel("Δ size-adjusted semantic coherence (fused − topology)")
ax.set_title("Semantic gain after controlling for partition granularity")
ax.grid(axis="y", alpha=0.2)
fig.tight_layout()
fig.savefig(OUT_RUN / "figures" / "figure16_1b_adjusted_semantic_gain.png", dpi=400, bbox_inches="tight")
plt.close(fig)

# Fig. 16.1c — V corrigido por representação para cada algoritmo
fig, ax = plt.subplots(figsize=(10, 5.5))
pivot = institution_perm_df.pivot(index="algorithm", columns="condition", values="institution_cramers_v_bias_corrected")
order_cols = [c for c in ["topology_only", "semantic_only", "fused_selected"] if c in pivot.columns]
pivot[order_cols].plot(kind="bar", ax=ax)
ax.set_xlabel("Community-detection algorithm")
ax.set_ylabel("Bias-corrected Cramér's V")
ax.set_title("External institutional alignment (permutation-tested)")
ax.legend(title="Representation")
ax.grid(axis="y", alpha=0.2)
fig.tight_layout()
fig.savefig(OUT_RUN / "figures" / "figure16_1c_institutional_alignment.png", dpi=400, bbox_inches="tight")
plt.close(fig)

# Fig. 16.1d — distribuições dos nulos para coerência semântica
fig, ax = plt.subplots(figsize=(9, 5.5))
for null_type, g in null_df.groupby("null_type"):
    vals = pd.to_numeric(g["semantic_centroid_coherence"], errors="coerce").dropna().values
    ax.hist(vals, bins=30, alpha=0.45, label=null_type)
ax.axvline(observed_global["semantic_centroid_coherence"], linestyle="--", linewidth=2, label="observed")
ax.set_xlabel("Semantic centroid coherence")
ax.set_ylabel("Frequency")
ax.set_title("Empirical null distributions for semantic coherence")
ax.legend()
fig.tight_layout()
fig.savefig(OUT_RUN / "figures" / "figure16_1d_null_semantic_distributions.png", dpi=400, bbox_inches="tight")
plt.close(fig)

# ============================================================
# 11. LOG / AUDITORIA / INTERPRETAÇÃO AUTOMÁTICA CONSERVADORA
# ============================================================
summary = {
    "created_at": datetime.now().isoformat(timespec="seconds"),
    "part16_run": str(PART16_RUN),
    "part16_1_run": str(OUT_RUN),
    "n_researchers": len(node_order),
    "n_size_matched": N_SIZE_MATCHED,
    "n_institution_perm": N_INSTITUTION_PERM,
    "run_extended_nulls": RUN_EXTENDED_NULLS,
    "n_null_used": N_NULL_EXTENDED if RUN_EXTENDED_NULLS else int(existing_null_df["replicate"].nunique()),
    "global_part16_configuration": {
        "algorithm": SELECTED_GLOBAL_ALGO,
        "k": SELECTED_GLOBAL_K,
        "alpha": SELECTED_GLOBAL_ALPHA,
    },
    "algorithm_parameter_consensus": {
        "same_k": bool(consensus_k),
        "k": int(fair_df["selected_k"].iloc[0]) if consensus_k and not fair_df.empty else None,
        "same_alpha": bool(consensus_alpha),
        "alpha": float(fair_df["selected_alpha"].iloc[0]) if consensus_alpha and not fair_df.empty else None,
    },
    "reporting_decision": (
        "Do not designate a global best algorithm from raw semantic coherence. "
        "Report algorithm-specific fused solutions and compare semantic excess against size-matched null partitions."
    ),
    "environment": {
        "python": sys.version,
        "platform": platform.platform(),
        "numpy": np.__version__,
        "pandas": pd.__version__,
        "networkx": nx.__version__,
        "igraph": ig.__version__,
    },
}
save_json(summary, OUT_RUN / "logs" / "audit_part16_1_final.json")

method_note = f"""
PARTE 16.1 — DECISÕES METODOLÓGICAS
===================================

1. A Parte 16.1 NÃO recalculou embeddings de publicações.
   Foram reutilizados researcher_embeddings.npy e researcher_cosine_similarity.npy da Parte 16.

2. A comparação entre algoritmos NÃO escolhe vencedor global com base na coerência semântica bruta.
   Motivo: partições com maior número de comunidades podem obter coerência maior mecanicamente.

3. A coerência semântica foi comparada contra partições aleatórias que preservam exatamente
   o número e o tamanho das comunidades observadas (N={N_SIZE_MATCHED}).

4. A associação Comunidade × Instituição foi testada por permutação dos rótulos institucionais
   entre pesquisadores (N={N_INSTITUTION_PERM}), preservando simultaneamente os tamanhos das
   comunidades e a distribuição marginal das instituições.

5. Os modelos nulos são reportados por p-valores empíricos com correção +1.
   Isso substitui a interpretação substantiva de z-scores extremos.

6. Para modularidade sob o null semântico, são reportados tanto p unilateral (observado > null)
   quanto p empírico bicaudal. A hipótese principal desse null é semântica, não a maximização
   da modularidade.

7. A configuração semântico-topológica é tratada como trade-off entre preservação estrutural
   e coerência semântica, e não como dominância universal sobre as representações baselines.
"""
with open(OUT_RUN / "logs" / "methodological_decisions.txt", "w", encoding="utf-8") as f:
    f.write(method_note)

print("\n" + "=" * 88)
print("✅ PARTE 16.1 CONCLUÍDA")
print("=" * 88)
print(f"Saídas: {OUT_RUN}")
print("\nTabelas principais:")
for name in [
    "table16_1a_granularity_adjusted_semantics.csv",
    "table16_1b_fused_vs_topology_adjusted_gain.csv",
    "table16_1c_institution_permutation.csv",
    "table16_1e_null_model_empirical_pvalues.csv",
    "table16_1f_algorithm_fair_comparison.csv",
    "table16_1g_algorithm_consensus.csv",
    "table16_1h_fused_algorithm_concordance.csv",
]:
    print(f"- tables/{name}")

print("\nFiguras principais:")
for name in [
    "figure16_1a_adjusted_semantic_tradeoff.png",
    "figure16_1b_adjusted_semantic_gain.png",
    "figure16_1c_institutional_alignment.png",
    "figure16_1d_null_semantic_distributions.png",
]:
    print(f"- figures/{name}")

print("\nPrincípio de reporte:")
print("- Não escolher vencedor global por coerência semântica bruta.")
print("- Reportar Louvain, Leiden e Infomap em paralelo.")
print("- Usar coerência semântica excedente ao null de tamanho como comparação justa.")
print("- Usar p-valores empíricos para nulos e validação institucional.")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
PARTE 16.1 — VALIDAÇÃO PÓS-HOC SEMÂNTICO-TOPOLÓGICA
Run da Parte 16 carregado: /content/drive/MyDrive/tese_doutoral/semantic_topological_runs/run_20260920_010404
Novo run da Parte 16.1:      /content/drive/MyDrive/tese_doutoral/semantic_topological_runs/run_20260920_010404/posthoc_16_1/run_20260920_020905

1. CARREGANDO ARTEFATOS DA PARTE 16
✅ Pesquisadores: 404
✅ Embeddings: (404, 768)
✅ Matriz semântica: (404, 404)
✅ Topologia comum: 404 nós, 1609 arestas
✅ Configuração global anterior: infomap, k=5, alpha=0.75

2. AJUSTE DA COERÊNCIA SEMÂNTICA POR GRANULARIDADE
louvain  | topology_only   | K=12 | Q=0.6151 | SC=0.8702 | SC_excess=0.0096 | p_emp=0.0010
louvain  | semantic_only   | K=10 | Q=0.2672 | SC=0.9015 | SC_excess=0.0416 | p_emp=0.0010
louvain  | fused_selected  | K=10 | Q=0.5920 | SC=0.8778 | SC_excess=0.0179 | p_emp=0.0010
leiden   | topology_only   

In [ ]:
# ============================================================
# PARTE 16.2 — IMPRESSÃO CONSOLIDADA PARA O ARTIGO
# Semantic-Topological Community Detection
# ============================================================
# Objetivo:
# 1) localizar automaticamente o run da Parte 16 e o post-hoc 16.1;
# 2) imprimir todas as tabelas candidatas ao artigo principal;
# 3) imprimir todas as tabelas complementares/suplementares;
# 4) imprimir logs/auditorias e resumo dos resultados centrais;
# 5) salvar um relatório TXT consolidado e, opcionalmente, um XLSX.
# ============================================================

from pathlib import Path
import json
import math
import pandas as pd
import numpy as np

# ----------------------------
# CONFIGURAÇÃO
# ----------------------------
PROJECT_ROOT = Path('/content/drive/MyDrive/tese_doutoral')
SEM_ROOT = PROJECT_ROOT / 'semantic_topological_runs'

# Se quiser imprimir literalmente TODAS as linhas das tabelas de runs brutos:
PRINT_RAW_RUNS = True

# Salvar também um workbook com todas as tabelas em abas separadas:
EXPORT_EXCEL = True

# ----------------------------
# MONTAR DRIVE, SE NECESSÁRIO
# ----------------------------
try:
    from google.colab import drive
    if not Path('/content/drive/MyDrive').exists():
        drive.mount('/content/drive')
except Exception:
    pass

# ----------------------------
# FUNÇÕES AUXILIARES
# ----------------------------
def safe_json(path):
    path = Path(path)
    if not path.exists():
        return None
    with open(path, 'r', encoding='utf-8') as f:
        return json.load(f)


def find_latest_part16_run():
    if not SEM_ROOT.exists():
        raise FileNotFoundError(f'Pasta não encontrada: {SEM_ROOT}')

    candidates = []
    for p in SEM_ROOT.glob('run_*'):
        if not p.is_dir():
            continue
        required = [
            p / 'tables' / 'table16_3_grid_aggregated.csv',
            p / 'tables' / 'table16_5_selected_vs_baselines.csv',
            p / 'logs' / 'audit_part16_final.json',
        ]
        if all(x.exists() for x in required):
            candidates.append(p)

    if not candidates:
        raise FileNotFoundError('Nenhum run completo da Parte 16 foi encontrado.')
    return sorted(candidates, key=lambda x: x.name)[-1]


def find_latest_posthoc_run(part16_run):
    root = part16_run / 'posthoc_16_1'
    if not root.exists():
        raise FileNotFoundError(f'Pasta pós-hoc não encontrada: {root}')

    candidates = []
    for p in root.glob('run_*'):
        required = [
            p / 'tables' / 'table16_1f_algorithm_fair_comparison.csv',
            p / 'tables' / 'table16_1e_null_model_empirical_pvalues.csv',
            p / 'logs' / 'audit_part16_1_final.json',
        ]
        if p.is_dir() and all(x.exists() for x in required):
            candidates.append(p)

    if not candidates:
        raise FileNotFoundError('Nenhum run completo da Parte 16.1 foi encontrado.')
    return sorted(candidates, key=lambda x: x.name)[-1]


def load_csv(path):
    path = Path(path)
    if not path.exists():
        return None
    return pd.read_csv(path)


def title(text, char='='):
    print('\n' + char * 110)
    print(text)
    print(char * 110)


def print_df(name, df, raw=False):
    if df is None:
        print(f'⚠️ {name}: arquivo não encontrado')
        return

    print(f'\n{name}')
    print('-' * min(110, max(50, len(name))))
    print(f'Dimensões: {df.shape[0]} linhas × {df.shape[1]} colunas')

    if raw and not PRINT_RAW_RUNS:
        print(df.head(30).to_string(index=False))
        print(f'... [exibidas 30 de {len(df)} linhas; defina PRINT_RAW_RUNS=True para imprimir tudo]')
        return

    with pd.option_context(
        'display.max_rows', None,
        'display.max_columns', None,
        'display.width', 240,
        'display.max_colwidth', 80,
        'display.float_format', lambda x: f'{x:.6f}' if isinstance(x, (float, np.floating)) else str(x),
    ):
        print(df.to_string(index=False))


def clean_sheet_name(name):
    bad = ['\\', '/', '*', '[', ']', ':', '?']
    for b in bad:
        name = name.replace(b, '_')
    return name[:31]

# ----------------------------
# LOCALIZAR RUNS
# ----------------------------
PART16_RUN = find_latest_part16_run()
POSTHOC_RUN = find_latest_posthoc_run(PART16_RUN)

OUT_DIR = POSTHOC_RUN / 'article_printout'
OUT_DIR.mkdir(parents=True, exist_ok=True)

print('=' * 110)
print('PARTE 16.2 — IMPRESSÃO CONSOLIDADA PARA O ARTIGO')
print('=' * 110)
print(f'Run Parte 16:   {PART16_RUN}')
print(f'Run Parte 16.1: {POSTHOC_RUN}')
print(f'Saídas 16.2:    {OUT_DIR}')

# ----------------------------
# CARREGAR TABELAS — PARTE 16
# ----------------------------
P16 = {
    '16_1_data_coverage': load_csv(PART16_RUN / 'tables' / 'table16_1_data_coverage.csv'),
    '16_2_grid_runs': load_csv(PART16_RUN / 'tables' / 'table16_2_grid_runs.csv'),
    '16_3_grid_aggregated': load_csv(PART16_RUN / 'tables' / 'table16_3_grid_aggregated.csv'),
    '16_4_selected_by_algorithm': load_csv(PART16_RUN / 'tables' / 'table16_4_selected_by_algorithm.csv'),
    '16_5_selected_vs_baselines': load_csv(PART16_RUN / 'tables' / 'table16_5_selected_vs_baselines.csv'),
    '16_6_perturbation_runs': load_csv(PART16_RUN / 'tables' / 'table16_6_perturbation_runs.csv'),
    '16_6_perturbation_stability': load_csv(PART16_RUN / 'tables' / 'table16_6_perturbation_stability.csv'),
    '16_7_null_model_runs': load_csv(PART16_RUN / 'tables' / 'table16_7_null_model_runs.csv'),
    '16_7_null_model_summary': load_csv(PART16_RUN / 'tables' / 'table16_7_null_model_summary.csv'),
    '16_8_external_institutional_validation': load_csv(PART16_RUN / 'tables' / 'table16_8_external_institutional_validation.csv'),
    '16_9_selected_community_profiles': load_csv(PART16_RUN / 'tables' / 'table16_9_selected_community_profiles.csv'),
    '16_10_partition_concordance': load_csv(PART16_RUN / 'tables' / 'table16_10_partition_concordance.csv'),
}

# ----------------------------
# CARREGAR TABELAS — PARTE 16.1
# ----------------------------
P161 = {
    '16_1a_granularity_adjusted_semantics': load_csv(POSTHOC_RUN / 'tables' / 'table16_1a_granularity_adjusted_semantics.csv'),
    '16_1b_fused_vs_topology_adjusted_gain': load_csv(POSTHOC_RUN / 'tables' / 'table16_1b_fused_vs_topology_adjusted_gain.csv'),
    '16_1c_institution_permutation': load_csv(POSTHOC_RUN / 'tables' / 'table16_1c_institution_permutation.csv'),
    '16_1d_null_runs_empirical': load_csv(POSTHOC_RUN / 'tables' / 'table16_1d_null_runs_empirical.csv'),
    '16_1e_null_model_empirical_pvalues': load_csv(POSTHOC_RUN / 'tables' / 'table16_1e_null_model_empirical_pvalues.csv'),
    '16_1f_algorithm_fair_comparison': load_csv(POSTHOC_RUN / 'tables' / 'table16_1f_algorithm_fair_comparison.csv'),
    '16_1g_algorithm_consensus': load_csv(POSTHOC_RUN / 'tables' / 'table16_1g_algorithm_consensus.csv'),
    '16_1h_fused_algorithm_concordance': load_csv(POSTHOC_RUN / 'tables' / 'table16_1h_fused_algorithm_concordance.csv'),
}

# ----------------------------
# LOGS E AUDITORIAS
# ----------------------------
manifest = safe_json(PART16_RUN / 'logs' / 'manifest.json')
selected_config = safe_json(PART16_RUN / 'logs' / 'selected_configuration.json')
audit16 = safe_json(PART16_RUN / 'logs' / 'audit_part16_final.json')
audit161 = safe_json(POSTHOC_RUN / 'logs' / 'audit_part16_1_final.json')
hardware = safe_json(PART16_RUN / 'logs' / 'hardware.json')

# ============================================================
# A. RESUMO EXECUTIVO DOS RESULTADOS CENTRAIS
# ============================================================
title('A. RESUMO EXECUTIVO — RESULTADOS CENTRAIS PARA O ARTIGO')

fair = P161['16_1f_algorithm_fair_comparison']
if fair is not None:
    cols = [c for c in [
        'algorithm', 'selected_k', 'selected_alpha',
        'fused_n_communities_seed42', 'topology_n_communities_seed42',
        'fused_modularity_seed42', 'topology_modularity_seed42',
        'structural_retention_ratio_seed42',
        'fused_centroid_coherence_raw',
        'fused_centroid_excess_size_adjusted',
        'delta_size_adjusted_centroid_excess_vs_topology',
        'stability_nmi_mean_part16', 'stability_ari_mean_part16',
        'conductance_mean_part16',
        'institution_vcorr_fused', 'institution_perm_p_emp'
    ] if c in fair.columns]
    print_df('Resumo comparativo dos três algoritmos', fair[cols])

cons = P161['16_1g_algorithm_consensus']
if cons is not None:
    print_df('Consenso de parametrização entre algoritmos', cons)

# Perturbação
pert = P16['16_6_perturbation_stability']
if pert is not None:
    print_df('Robustez da configuração fused a perturbações', pert)

# Null models empíricos
null_emp = P161['16_1e_null_model_empirical_pvalues']
if null_emp is not None:
    print_df('Modelos nulos — p-valores empíricos', null_emp)

# Institucional
inst = P161['16_1c_institution_permutation']
if inst is not None:
    print_df('Validação institucional por permutação', inst)

# ============================================================
# B. TABELAS CANDIDATAS AO CORPO PRINCIPAL DO ARTIGO
# ============================================================
title('B. TABELAS CANDIDATAS AO CORPO PRINCIPAL DO ARTIGO')

MAIN_TABLES = [
    ('Tabela M1 — Cobertura e auditoria dos dados', P16['16_1_data_coverage']),
    ('Tabela M2 — Comparação topology-only, semantic-only e fused', P16['16_5_selected_vs_baselines']),
    ('Tabela M3 — Ganho semântico ajustado por granularidade', P161['16_1b_fused_vs_topology_adjusted_gain']),
    ('Tabela M4 — Comparação justa entre Louvain, Leiden e Infomap', P161['16_1f_algorithm_fair_comparison']),
    ('Tabela M5 — Robustez a perturbações', P16['16_6_perturbation_stability']),
    ('Tabela M6 — Modelos nulos com p-valores empíricos', P161['16_1e_null_model_empirical_pvalues']),
    ('Tabela M7 — Validação institucional por permutação', P161['16_1c_institution_permutation']),
    ('Tabela M8 — Concordância entre algoritmos na representação fused', P161['16_1h_fused_algorithm_concordance']),
]

for name, df in MAIN_TABLES:
    print_df(name, df)

# ============================================================
# C. TABELAS COMPLEMENTARES / SUPLEMENTARES
# ============================================================
title('C. TABELAS COMPLEMENTARES / MATERIAL SUPLEMENTAR')

SUPP_TABLES = [
    ('Tabela S1 — Grade experimental completa por execução/seed', P16['16_2_grid_runs'], True),
    ('Tabela S2 — Grade experimental agregada', P16['16_3_grid_aggregated'], False),
    ('Tabela S3 — Configurações fused selecionadas por algoritmo', P16['16_4_selected_by_algorithm'], False),
    ('Tabela S4 — Todas as execuções de perturbação', P16['16_6_perturbation_runs'], True),
    ('Tabela S5 — Runs dos modelos nulos originais da Parte 16', P16['16_7_null_model_runs'], True),
    ('Tabela S6 — Resumo dos modelos nulos originais da Parte 16', P16['16_7_null_model_summary'], False),
    ('Tabela S7 — Validação institucional original ex post', P16['16_8_external_institutional_validation'], False),
    ('Tabela S8 — Perfis das comunidades da partição originalmente selecionada', P16['16_9_selected_community_profiles'], False),
    ('Tabela S9 — Concordância entre partições/representações da Parte 16', P16['16_10_partition_concordance'], False),
    ('Tabela S10 — Coerência semântica ajustada por granularidade para todas as condições', P161['16_1a_granularity_adjusted_semantics'], False),
    ('Tabela S11 — Todas as permutações/nulos empíricos da Parte 16.1', P161['16_1d_null_runs_empirical'], True),
    ('Tabela S12 — Consenso de parametrização entre algoritmos', P161['16_1g_algorithm_consensus'], False),
]

for name, df, raw in SUPP_TABLES:
    print_df(name, df, raw=raw)

# ============================================================
# D. LOGS, CONFIGURAÇÕES E RASTREABILIDADE
# ============================================================
title('D. CONFIGURAÇÕES, AUDITORIAS E RASTREABILIDADE')

for name, obj in [
    ('Manifest da Parte 16', manifest),
    ('Configuração selecionada na Parte 16', selected_config),
    ('Auditoria final da Parte 16', audit16),
    ('Auditoria final da Parte 16.1', audit161),
    ('Hardware/ambiente', hardware),
]:
    print(f'\n{name}')
    print('-' * 90)
    if obj is None:
        print('(arquivo não encontrado)')
    else:
        print(json.dumps(obj, indent=2, ensure_ascii=False))

# ============================================================
# E. FIGURAS DISPONÍVEIS
# ============================================================
title('E. FIGURAS DISPONÍVEIS PARA O ARTIGO E SUPLEMENTO')

figure_files = []
for root in [PART16_RUN / 'figures', POSTHOC_RUN / 'figures']:
    if root.exists():
        figure_files.extend(sorted(root.glob('*')))

for p in figure_files:
    print(p)

# ============================================================
# F. EXPORTAR RELATÓRIO TXT CONSOLIDADO
# ============================================================
report_path = OUT_DIR / 'relatorio_completo_parte16_16_1.txt'

sections = []
sections.append('PARTE 16 / 16.1 — RELATÓRIO COMPLETO PARA REDAÇÃO DO ARTIGO\n')
sections.append(f'Run Parte 16: {PART16_RUN}\n')
sections.append(f'Run Parte 16.1: {POSTHOC_RUN}\n\n')

for label, df in list(P16.items()) + list(P161.items()):
    sections.append('\n' + '='*100 + '\n')
    sections.append(label + '\n')
    sections.append('='*100 + '\n')
    if df is None:
        sections.append('(não encontrado)\n')
    else:
        sections.append(df.to_string(index=False) + '\n')

sections.append('\n' + '='*100 + '\nLOGS E AUDITORIAS\n' + '='*100 + '\n')
for label, obj in [
    ('manifest', manifest),
    ('selected_configuration', selected_config),
    ('audit_part16_final', audit16),
    ('audit_part16_1_final', audit161),
    ('hardware', hardware),
]:
    sections.append(f'\n## {label}\n')
    sections.append(json.dumps(obj, indent=2, ensure_ascii=False) if obj is not None else '(não encontrado)')
    sections.append('\n')

report_path.write_text(''.join(sections), encoding='utf-8')
print(f'\n✅ Relatório TXT consolidado salvo em:\n{report_path}')

# ============================================================
# G. EXPORTAR WORKBOOK COM TODAS AS TABELAS
# ============================================================
if EXPORT_EXCEL:
    xlsx_path = OUT_DIR / 'tabelas_completas_parte16_16_1.xlsx'
    try:
        with pd.ExcelWriter(xlsx_path, engine='openpyxl') as writer:
            for label, df in P16.items():
                if df is not None:
                    df.to_excel(writer, sheet_name=clean_sheet_name(label), index=False)
            for label, df in P161.items():
                if df is not None:
                    df.to_excel(writer, sheet_name=clean_sheet_name(label), index=False)
        print(f'✅ Workbook consolidado salvo em:\n{xlsx_path}')
    except Exception as e:
        print(f'⚠️ Não foi possível gerar o XLSX: {e}')

# ============================================================
# H. MANIFESTO DAS SAÍDAS DA PARTE 16.2
# ============================================================
manifest_162 = {
    'part16_run': str(PART16_RUN),
    'posthoc_run': str(POSTHOC_RUN),
    'article_printout_dir': str(OUT_DIR),
    'main_tables': [x[0] for x in MAIN_TABLES],
    'supplementary_tables': [x[0] for x in SUPP_TABLES],
    'report_txt': str(report_path),
    'workbook_xlsx': str(OUT_DIR / 'tabelas_completas_parte16_16_1.xlsx') if EXPORT_EXCEL else None,
    'figures': [str(p) for p in figure_files],
}

with open(OUT_DIR / 'manifest_parte16_2.json', 'w', encoding='utf-8') as f:
    json.dump(manifest_162, f, indent=2, ensure_ascii=False)

print('\n' + '=' * 110)
print('✅ PARTE 16.2 CONCLUÍDA')
print('=' * 110)
print('Esta parte apenas lê e consolida resultados já calculados; nenhum resultado científico foi recalculado.')

PARTE 16.2 — IMPRESSÃO CONSOLIDADA PARA O ARTIGO
Run Parte 16:   /content/drive/MyDrive/tese_doutoral/semantic_topological_runs/run_20260920_010404
Run Parte 16.1: /content/drive/MyDrive/tese_doutoral/semantic_topological_runs/run_20260920_010404/posthoc_16_1/run_20260920_020905
Saídas 16.2:    /content/drive/MyDrive/tese_doutoral/semantic_topological_runs/run_20260920_010404/posthoc_16_1/run_20260920_020905/article_printout

A. RESUMO EXECUTIVO — RESULTADOS CENTRAIS PARA O ARTIGO

Resumo comparativo dos três algoritmos
--------------------------------------------------
Dimensões: 3 linhas × 16 colunas
algorithm  selected_k  selected_alpha  fused_n_communities_seed42  topology_n_communities_seed42  fused_modularity_seed42  topology_modularity_seed42  structural_retention_ratio_seed42  fused_centroid_coherence_raw  fused_centroid_excess_size_adjusted  delta_size_adjusted_centroid_excess_vs_topology  stability_nmi_mean_part16  stability_ari_mean_part16  conductance_mean_part16  instituti

In [ ]:
# ============================================================
# PARTE 16.3 — ARTICLE-READY OUTPUTS + SUPPLEMENTARY ALGORITHM SENSITIVITY
# ============================================================
# Objetivos:
# (1) Gerar automaticamente as tabelas e figuras que irão para o artigo.
# (2) Garantir correspondência direta lab <-> manuscrito.
# (3) Adicionar Label Propagation e Walktrap apenas como sensibilidade suplementar,
#     sem reexecutar embeddings nem a grade principal k x alpha.
# ============================================================

from google.colab import drive
drive.mount("/content/drive", force_remount=False)

from pathlib import Path
from datetime import datetime
from collections import defaultdict
import json
import math
import pickle
import random
import warnings

import numpy as np
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt

from sklearn.metrics import normalized_mutual_info_score, adjusted_rand_score

try:
    import igraph as ig
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "igraph"])
    import igraph as ig

warnings.filterwarnings("ignore")

# ------------------------------------------------------------
# 0. CONFIGURAÇÃO E LOCALIZAÇÃO AUTOMÁTICA DOS RUNS
# ------------------------------------------------------------
BASE = Path("/content/drive/MyDrive/tese_doutoral")
SEM_ROOT = BASE / "semantic_topological_runs"

PREFERRED_PART16 = SEM_ROOT / "run_20260920_010404"

N_SIZE_MATCHED_SUPP = 999
SUPP_SEEDS = [42, 52, 62, 72, 82, 92, 102, 112, 122, 132]
WALKTRAP_STEPS = 4

def newest_dir(parent, pattern="run_*"):
    dirs = [p for p in parent.glob(pattern) if p.is_dir()]
    if not dirs:
        return None
    return max(dirs, key=lambda p: p.stat().st_mtime)

if PREFERRED_PART16.exists():
    PART16 = PREFERRED_PART16
else:
    candidates = []
    if SEM_ROOT.exists():
        for p in SEM_ROOT.glob("run_*"):
            if (p / "tables" / "table16_3_grid_aggregated.csv").exists():
                candidates.append(p)
    if not candidates:
        raise FileNotFoundError("Nenhum run válido da Parte 16 foi encontrado.")
    PART16 = max(candidates, key=lambda p: p.stat().st_mtime)

POSTHOC_ROOT = PART16 / "posthoc_16_1"
PART16_1 = newest_dir(POSTHOC_ROOT)
if PART16_1 is None:
    raise FileNotFoundError("Nenhum run da Parte 16.1 foi encontrado.")

OUT = PART16 / "article_ready_16_3" / f"run_{datetime.now():%Y%m%d_%H%M%S}"
TABLES = OUT / "tables"
FIGURES = OUT / "figures"
SUPP = OUT / "supplementary"
LOGS = OUT / "logs"
for d in [TABLES, FIGURES, SUPP, LOGS]:
    d.mkdir(parents=True, exist_ok=True)

print("="*92)
print("PARTE 16.3 — ARTICLE-READY OUTPUTS + SUPPLEMENTARY SENSITIVITY")
print("="*92)
print("Parte 16:  ", PART16)
print("Parte 16.1:", PART16_1)
print("Saída 16.3:", OUT)

# ------------------------------------------------------------
# 1. CARREGAR RESULTADOS CONSOLIDADOS
# ------------------------------------------------------------
def read_csv(path):
    if not path.exists():
        raise FileNotFoundError(path)
    return pd.read_csv(path)

coverage = read_csv(PART16 / "tables" / "table16_1_data_coverage.csv")
grid_agg = read_csv(PART16 / "tables" / "table16_3_grid_aggregated.csv")
perturb = read_csv(PART16 / "tables" / "table16_6_perturbation_stability.csv")

gain = read_csv(PART16_1 / "tables" / "table16_1b_fused_vs_topology_adjusted_gain.csv")
inst = read_csv(PART16_1 / "tables" / "table16_1c_institution_permutation.csv")
nulls = read_csv(PART16_1 / "tables" / "table16_1e_null_model_empirical_pvalues.csv")
fair = read_csv(PART16_1 / "tables" / "table16_1f_algorithm_fair_comparison.csv")
consensus = read_csv(PART16_1 / "tables" / "table16_1g_algorithm_consensus.csv")

def resolve_column(df, *candidates):
    """Retorna a primeira coluna existente dentre nomes alternativos esperados."""
    for c in candidates:
        if c in df.columns:
            return c
    raise KeyError(
        f"Nenhuma das colunas esperadas foi encontrada: {candidates}. "
        f"Colunas disponíveis: {list(df.columns)}"
    )

# Schema real da Parte 16.1:
# institution_cramers_v_bias_corrected / institution_vcorr_p_emp.
# Mantemos aliases para compatibilidade com versões futuras/anteriores.
INST_VCORR_COL = resolve_column(
    inst,
    "institution_cramers_v_bias_corrected",
    "cramers_v_bias_corrected",
)
INST_P_COL = resolve_column(
    inst,
    "institution_vcorr_p_emp",
    "p_emp",
)

required_inst = {"algorithm", "condition", INST_VCORR_COL, INST_P_COL}
missing_inst = required_inst.difference(inst.columns)
if missing_inst:
    raise KeyError(f"Colunas institucionais ausentes: {sorted(missing_inst)}")

print("Schema institucional detectado:")
print("  V corrigido:", INST_VCORR_COL)
print("  p empírico:", INST_P_COL)

# ------------------------------------------------------------
# 2. TABLE I — EXPERIMENTAL PROTOCOL (GERADA PELO LAB)
# ------------------------------------------------------------
cov = coverage.iloc[0]

table_I = pd.DataFrame([
    ["Research network",
     f"{int(cov['n_nodes_full_ppgci'])} identified researchers; "
     f"GCC {int(cov['n_nodes_gcc'])} nodes / 1,609 edges"],
    ["Publication data",
     f"{int(cov['n_publications_total']):,} records; "
     f"{int(cov['n_publications_ge1_gcc_with_text']):,} publication titles used semantically"],
    ["Semantic representation",
     "768-dimensional paraphrase-multilingual-mpnet-base-v2 embeddings; title-only"],
    ["Semantic k-NN grid", "k = {5, 10, 15, 20}"],
    ["Fusion grid", "alpha = {0.25, 0.50, 0.75}"],
    ["Primary algorithms", "Louvain, Leiden, Infomap"],
    ["Random seeds", "10 per configuration"],
    ["Structural-retention constraint", ">= 95% of topology-only modularity"],
    ["Size-matched semantic nulls", "999 per reported partition"],
    ["Topology / semantic null models", "999 replicates each"],
    ["Institution-label permutations", "4,999"],
    ["Perturbation analysis", "50 replicates per scenario"],
], columns=["Item", "Setting"])

table_I.to_csv(TABLES / "Table_I_Experimental_Protocol.csv", index=False)

# ------------------------------------------------------------
# 3. TABLE II — CROSS-ALGORITHM STRUCTURAL/SEMANTIC TRADE-OFF
# ------------------------------------------------------------
alg_order = ["louvain", "leiden", "infomap"]
name_map = {"louvain":"Louvain", "leiden":"Leiden", "infomap":"Infomap"}

rows = []
for alg in alg_order:
    r = fair.loc[fair["algorithm"].str.lower() == alg].iloc[0]
    rows.append({
        "Algorithm": name_map[alg],
        "k_selected": int(r["selected_k"]),
        "alpha_selected": float(r["selected_alpha"]),
        "K_topology": int(r["topology_n_communities_seed42"]),
        "K_fused": int(r["fused_n_communities_seed42"]),
        "Q_topology": float(r["topology_modularity_seed42"]),
        "Q_fused": float(r["fused_modularity_seed42"]),
        "Q_retained_pct": 100.0 * float(r["structural_retention_ratio_seed42"]),
        "Delta_SC_adj": float(r["delta_size_adjusted_centroid_excess_vs_topology"]),
        "NMI_stability": float(r["stability_nmi_mean_part16"]),
        "ARI_stability": float(r["stability_ari_mean_part16"]),
        "Conductance_fused": float(r["conductance_mean_part16"]),
        "Institution_Vcorr_fused": float(r["institution_vcorr_fused"]),
    })
table_II = pd.DataFrame(rows)
table_II.to_csv(TABLES / "Table_II_Cross_Algorithm_Tradeoff.csv", index=False)

# Versão compacta já arredondada para copiar ao manuscrito
table_II_print = table_II.copy()
for c in ["Q_topology","Q_fused","NMI_stability","ARI_stability","Conductance_fused",
          "Institution_Vcorr_fused"]:
    table_II_print[c] = table_II_print[c].map(lambda x: f"{x:.3f}")
table_II_print["Q_retained_pct"] = table_II_print["Q_retained_pct"].map(lambda x: f"{x:.1f}%")
table_II_print["Delta_SC_adj"] = table_II_print["Delta_SC_adj"].map(lambda x: f"{x:+.4f}")
table_II_print.to_csv(TABLES / "Table_II_Cross_Algorithm_Tradeoff_PRINT.csv", index=False)

# ------------------------------------------------------------
# 4. TABLE III — ROBUSTNESS AND STATISTICAL VALIDATION
# ------------------------------------------------------------
# Perturbações
pmap = {
    "semantic_embedding_noise": "Semantic embedding noise",
    "topology_edge_dropout": "Topology edge dropout",
    "combined": "Combined perturbation",
}
table_III_rows = []
for _, r in perturb.iterrows():
    table_III_rows.append({
        "Validation": pmap.get(r["perturbation"], r["perturbation"]),
        "Observed": f"NMI {r['nmi_mean']:.3f} / ARI {r['ari_mean']:.3f}",
        "Reference": f"{int(r['n'])} perturbations",
        "p_emp": "",
    })

# Nulos
deg = nulls.loc[nulls["null_type"] == "degree_preserving_topology"].iloc[0]
sem = nulls.loc[nulls["null_type"] == "permuted_semantic_labels"].iloc[0]

table_III_rows += [
    {
        "Validation": "Degree-preserving topology: modularity",
        "Observed": f"{deg['modularity_observed']:.4f}",
        "Reference": f"null mean {deg['modularity_null_mean']:.4f}; n={int(deg['n'])}",
        "p_emp": f"{deg['modularity_p_greater_emp']:.3f}",
    },
    {
        "Validation": "Permuted semantic labels: centroid coherence",
        "Observed": f"{sem['semantic_coherence_observed']:.4f}",
        "Reference": f"null mean {sem['semantic_coherence_null_mean']:.4f}; n={int(sem['n'])}",
        "p_emp": f"{sem['semantic_coherence_p_greater_emp']:.3f}",
    },
    {
        "Validation": "Permuted semantic labels: semantic gap",
        "Observed": f"{sem['semantic_gap_observed']:.4f}",
        "Reference": f"null mean {sem['semantic_gap_null_mean']:.4f}; n={int(sem['n'])}",
        "p_emp": f"{sem['semantic_gap_p_greater_emp']:.3f}",
    },
]

# Instituição: reporta faixa fused entre algoritmos, sem transformar em ranking
inst_f = inst.loc[inst["condition"] == "fused_selected"].copy()
vmin = inst_f[INST_VCORR_COL].min()
vmax = inst_f[INST_VCORR_COL].max()
pmax = inst_f[INST_P_COL].max()
table_III_rows.append({
    "Validation": "Institution-label permutation (fused partitions)",
    "Observed": f"bias-corrected V = {vmin:.3f}–{vmax:.3f}",
    "Reference": "4,999 permutations / algorithm",
    "p_emp": f"≤ {pmax:.4f}",
})

table_III = pd.DataFrame(table_III_rows)
table_III.to_csv(TABLES / "Table_III_Robustness_Statistical_Validation.csv", index=False)

# ------------------------------------------------------------
# 5. FIG. 2 — STRUCTURAL RETENTION VS ADJUSTED SEMANTIC GAIN
# ------------------------------------------------------------
fig, ax = plt.subplots(figsize=(7.2, 5.0))
for _, r in table_II.iterrows():
    ax.scatter(r["Q_retained_pct"], r["Delta_SC_adj"], s=110)
    # pequenos deslocamentos para legibilidade
    dx, dy = 0.0, 0.0
    if r["Algorithm"] == "Louvain":
        dx, dy = 0.18, 0.00020
    elif r["Algorithm"] == "Leiden":
        dx, dy = 0.18, -0.00035
    elif r["Algorithm"] == "Infomap":
        dx, dy = -0.35, 0.00020
    ax.annotate(r["Algorithm"],
                (r["Q_retained_pct"], r["Delta_SC_adj"]),
                xytext=(r["Q_retained_pct"]+dx, r["Delta_SC_adj"]+dy),
                fontsize=11)

ax.axvline(95, linestyle="--", linewidth=1.3)
ax.set_xlabel("Topology-only modularity retained (%)")
ax.set_ylabel("Granularity-adjusted semantic gain")
ax.set_xlim(94.6, max(99.4, table_II["Q_retained_pct"].max()+0.4))
ymin = min(0.0042, table_II["Delta_SC_adj"].min()-0.0007)
ymax = max(0.0090, table_II["Delta_SC_adj"].max()+0.0007)
ax.set_ylim(ymin, ymax)
ax.grid(alpha=0.25)
fig.tight_layout()
fig.savefig(FIGURES / "Fig_2_Structural_Retention_vs_Adjusted_Semantic_Gain.png", dpi=600, bbox_inches="tight")
fig.savefig(FIGURES / "Fig_2_Structural_Retention_vs_Adjusted_Semantic_Gain.pdf", bbox_inches="tight")
plt.close(fig)

# ------------------------------------------------------------
# 6. FIG. 3 — ROBUSTNESS + INSTITUTIONAL ALIGNMENT (MULTIPART)
# ------------------------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(10.5, 4.0))

# (a) perturbation robustness
p_order = ["semantic_embedding_noise", "topology_edge_dropout", "combined"]
pp = perturb.set_index("perturbation").loc[p_order].reset_index()
x = np.arange(len(pp))
width = 0.35
axes[0].bar(x - width/2, pp["nmi_mean"], width, label="NMI")
axes[0].bar(x + width/2, pp["ari_mean"], width, label="ARI")
axes[0].set_xticks(x)
axes[0].set_xticklabels(["Semantic\nnoise", "Edge\ndropout", "Combined"])
axes[0].set_ylim(0, 1)
axes[0].set_ylabel("Partition agreement")
axes[0].legend(frameon=False)
axes[0].set_title("(a) Perturbation robustness")

# (b) institutional alignment
conds = ["topology_only", "semantic_only", "fused_selected"]
cond_labels = ["Topology", "Semantic", "Fused"]
x = np.arange(3)
width = 0.25
for i, alg in enumerate(alg_order):
    sub = inst.loc[inst["algorithm"].str.lower() == alg].set_index("condition")
    vals = [float(sub.loc[c, INST_VCORR_COL]) for c in conds]
    axes[1].bar(x + (i-1)*width, vals, width, label=name_map[alg])
axes[1].set_xticks(x)
axes[1].set_xticklabels(cond_labels)
axes[1].set_ylim(0, 0.8)
axes[1].set_ylabel("Bias-corrected Cramér's V")
axes[1].legend(frameon=False, fontsize=8)
axes[1].set_title("(b) External institutional alignment")

fig.tight_layout()
fig.savefig(FIGURES / "Fig_3_Robustness_and_External_Validation.png", dpi=600, bbox_inches="tight")
fig.savefig(FIGURES / "Fig_3_Robustness_and_External_Validation.pdf", bbox_inches="tight")
plt.close(fig)

# ------------------------------------------------------------
# 7. COPIAR/REFERENCIAR FIGURAS COMPLEMENTARES JÁ EXISTENTES
# ------------------------------------------------------------
supp_figure_manifest = []
existing_fig_dirs = [
    PART16 / "figures",
    PART16_1 / "figures",
]
for fd in existing_fig_dirs:
    if fd.exists():
        for p in sorted(fd.glob("*")):
            if p.suffix.lower() in [".png", ".pdf"]:
                supp_figure_manifest.append(str(p))

pd.DataFrame({"existing_supplementary_figure": supp_figure_manifest}).to_csv(
    SUPP / "Supplementary_Figure_Manifest.csv", index=False
)

# ============================================================
# 8. SENSIBILIDADE SUPLEMENTAR: LABEL PROPAGATION + WALKTRAP
# ============================================================
print("\n" + "="*92)
print("SENSIBILIDADE SUPLEMENTAR — LABEL PROPAGATION E WALKTRAP")
print("="*92)

with open(PART16 / "artifacts" / "topology_graph_gcc_common_nodes.pkl", "rb") as f:
    G_top = pickle.load(f)
with open(PART16 / "artifacts" / "semantic_graph_k5.pkl", "rb") as f:
    G_sem = pickle.load(f)
with open(PART16 / "artifacts" / "selected_fused_graph.pkl", "rb") as f:
    G_fused = pickle.load(f)

researchers = pd.read_csv(PART16 / "intermediate" / "researcher_semantic_coverage.csv")
nodes = researchers["researcher"].astype(str).tolist()
emb = np.load(PART16 / "artifacts" / "researcher_embeddings.npy")
sim = np.load(PART16 / "artifacts" / "researcher_cosine_similarity.npy")

# Harmonizar nós (geralmente já são strings)
def ensure_node_compatibility(G, node_list):
    if set(node_list) == set(G.nodes()):
        return G
    # tentativa sem conversão destrutiva
    str_map = {str(n): n for n in G.nodes()}
    if set(node_list).issubset(set(str_map)):
        H = nx.relabel_nodes(G, {orig: str(orig) for orig in G.nodes()}, copy=True)
        return H
    raise RuntimeError("Os identificadores dos nós não coincidem com researcher_semantic_coverage.csv.")

G_top = ensure_node_compatibility(G_top, nodes)
G_sem = ensure_node_compatibility(G_sem, nodes)
G_fused = ensure_node_compatibility(G_fused, nodes)

def relabel_partition(partition):
    groups = defaultdict(list)
    for n, c in partition.items():
        groups[c].append(n)
    ordered = sorted(groups.values(), key=lambda g: (-len(g), sorted(map(str, g))[0]))
    out = {}
    for new_c, group in enumerate(ordered):
        for n in group:
            out[n] = new_c
    return out

def nx_to_igraph(G):
    ns = list(G.nodes())
    index = {n:i for i,n in enumerate(ns)}
    edges = [(index[u], index[v]) for u,v in G.edges()]
    weights = [float(G[u][v].get("weight", 1.0)) for u,v in G.edges()]
    gi = ig.Graph(n=len(ns), edges=edges, directed=False)
    gi.vs["name"] = ns
    gi.es["weight"] = weights
    return gi, ns

def detect_labelprop(G, seed):
    comms = list(nx.algorithms.community.asyn_lpa_communities(
        G, weight="weight", seed=seed
    ))
    p = {}
    for c, group in enumerate(comms):
        for n in group:
            p[n] = c
    return relabel_partition(p)

def detect_walktrap(G):
    gi, ns = nx_to_igraph(G)
    wt = gi.community_walktrap(weights="weight", steps=WALKTRAP_STEPS)
    cl = wt.as_clustering()
    p = {ns[i]: int(cl.membership[i]) for i in range(len(ns))}
    return relabel_partition(p)

def labels_from_partition(p):
    return np.array([p[n] for n in nodes], dtype=int)

def partition_sets(p):
    d = defaultdict(set)
    for n,c in p.items():
        d[c].add(n)
    return list(d.values())

def structural_modularity(p):
    return float(nx.algorithms.community.quality.modularity(
        G_top, partition_sets(p), weight="weight"
    ))

def centroid_coherence_from_labels(labels):
    vals = []
    for c in np.unique(labels):
        idx = np.where(labels == c)[0]
        centroid = emb[idx].mean(axis=0)
        norm = np.linalg.norm(centroid)
        if norm <= 0:
            continue
        centroid = centroid / norm
        vals.extend((emb[idx] @ centroid).tolist())
    return float(np.mean(vals))

# Índices triangulares e similaridades pré-computados para acelerar 999 permutações.
_TRI_I, _TRI_J = np.triu_indices(len(nodes), k=1)
_SIM_UPPER = sim[_TRI_I, _TRI_J].astype(float)

def semantic_gap_from_labels(labels):
    labels = np.asarray(labels)
    same = labels[_TRI_I] == labels[_TRI_J]
    if not np.any(same) or np.all(same):
        return np.nan
    within = float(_SIM_UPPER[same].mean())
    between = float(_SIM_UPPER[~same].mean())
    return within - between

def size_matched_null(labels, nrep=999, seed=20260921):
    rng = np.random.default_rng(seed)
    obs_sc = centroid_coherence_from_labels(labels)
    obs_gap = semantic_gap_from_labels(labels)
    null_sc, null_gap = [], []
    for _ in range(nrep):
        perm = rng.permutation(labels)  # preserva exatamente os tamanhos
        null_sc.append(centroid_coherence_from_labels(perm))
        null_gap.append(semantic_gap_from_labels(perm))
    null_sc = np.asarray(null_sc, float)
    null_gap = np.asarray(null_gap, float)
    return {
        "sc_observed": obs_sc,
        "sc_null_mean": float(np.nanmean(null_sc)),
        "sc_excess": float(obs_sc - np.nanmean(null_sc)),
        "sc_p_emp": float((1 + np.sum(null_sc >= obs_sc)) / (nrep + 1)),
        "gap_observed": obs_gap,
        "gap_null_mean": float(np.nanmean(null_gap)),
        "gap_excess": float(obs_gap - np.nanmean(null_gap)),
        "gap_p_emp": float((1 + np.sum(null_gap >= obs_gap)) / (nrep + 1)),
    }

def pairwise_stability(partitions):
    if len(partitions) < 2:
        return np.nan, np.nan
    nmis, aris = [], []
    labs = [labels_from_partition(p) for p in partitions]
    for i in range(len(labs)):
        for j in range(i+1, len(labs)):
            nmis.append(normalized_mutual_info_score(labs[i], labs[j]))
            aris.append(adjusted_rand_score(labs[i], labs[j]))
    return float(np.mean(nmis)), float(np.mean(aris))

graphs = {
    "topology_only": G_top,
    "semantic_only": G_sem,
    "fused": G_fused,
}

supp_rows = []
canonical_partitions = {}

# Label Propagation: 10 seeds
for cond, G in graphs.items():
    parts = [detect_labelprop(G, s) for s in SUPP_SEEDS]
    p0 = parts[0]
    canonical_partitions[("label_propagation", cond)] = p0
    nmi_stab, ari_stab = pairwise_stability(parts)
    labs = labels_from_partition(p0)
    nullres = size_matched_null(
        labs, nrep=N_SIZE_MATCHED_SUPP,
        seed=20260921 + {"topology_only":1,"semantic_only":2,"fused":3}[cond]
    )
    supp_rows.append({
        "algorithm": "Label Propagation",
        "condition": cond,
        "n_communities": len(set(p0.values())),
        "structural_modularity": structural_modularity(p0),
        **nullres,
        "stability_nmi": nmi_stab,
        "stability_ari": ari_stab,
        "runs": len(parts),
    })

# Walktrap: determinístico
for cond, G in graphs.items():
    p0 = detect_walktrap(G)
    canonical_partitions[("walktrap", cond)] = p0
    labs = labels_from_partition(p0)
    nullres = size_matched_null(
        labs, nrep=N_SIZE_MATCHED_SUPP,
        seed=20260931 + {"topology_only":1,"semantic_only":2,"fused":3}[cond]
    )
    supp_rows.append({
        "algorithm": "Walktrap",
        "condition": cond,
        "n_communities": len(set(p0.values())),
        "structural_modularity": structural_modularity(p0),
        **nullres,
        "stability_nmi": np.nan,
        "stability_ari": np.nan,
        "runs": 1,
    })

supp_alg = pd.DataFrame(supp_rows)
supp_alg.to_csv(SUPP / "Table_S_Additional_Algorithm_Sensitivity_Full.csv", index=False)

# Resumo fused vs topology
summary_rows = []
for alg in ["Label Propagation", "Walktrap"]:
    sub = supp_alg[supp_alg["algorithm"] == alg].set_index("condition")
    t = sub.loc["topology_only"]
    f = sub.loc["fused"]
    summary_rows.append({
        "Algorithm": alg,
        "K_topology": int(t["n_communities"]),
        "K_fused": int(f["n_communities"]),
        "Q_topology": float(t["structural_modularity"]),
        "Q_fused": float(f["structural_modularity"]),
        "Q_retained_pct": float(100*f["structural_modularity"]/t["structural_modularity"]) if t["structural_modularity"] != 0 else np.nan,
        "SC_excess_topology": float(t["sc_excess"]),
        "SC_excess_fused": float(f["sc_excess"]),
        "Delta_SC_adj": float(f["sc_excess"] - t["sc_excess"]),
        "fused_SC_p_emp": float(f["sc_p_emp"]),
        "stability_NMI": float(f["stability_nmi"]) if pd.notna(f["stability_nmi"]) else np.nan,
        "stability_ARI": float(f["stability_ari"]) if pd.notna(f["stability_ari"]) else np.nan,
    })

supp_summary = pd.DataFrame(summary_rows)
supp_summary.to_csv(SUPP / "Table_S_Additional_Algorithm_Sensitivity_Summary.csv", index=False)

# Figura suplementar combinando 5 algoritmos — após o run, serve para decidir
# se vale mencionar a análise suplementar no texto.
primary_plot = table_II[["Algorithm","Q_retained_pct","Delta_SC_adj"]].copy()
extra_plot = supp_summary[["Algorithm","Q_retained_pct","Delta_SC_adj"]].copy()
all_plot = pd.concat([primary_plot, extra_plot], ignore_index=True)

fig, ax = plt.subplots(figsize=(7.2, 5.0))
for _, r in all_plot.iterrows():
    ax.scatter(r["Q_retained_pct"], r["Delta_SC_adj"], s=90)
    ax.annotate(r["Algorithm"],
                (r["Q_retained_pct"], r["Delta_SC_adj"]),
                xytext=(5,5), textcoords="offset points", fontsize=9)
ax.axvline(95, linestyle="--", linewidth=1.2)
ax.axhline(0, linestyle=":", linewidth=1.0)
ax.set_xlabel("Topology-only modularity retained (%)")
ax.set_ylabel("Granularity-adjusted semantic gain")
ax.grid(alpha=0.25)
fig.tight_layout()
fig.savefig(SUPP / "Fig_S_Algorithm_Sensitivity_Five_Methods.png", dpi=600, bbox_inches="tight")
fig.savefig(SUPP / "Fig_S_Algorithm_Sensitivity_Five_Methods.pdf", bbox_inches="tight")
plt.close(fig)

# ------------------------------------------------------------
# 9. WORKBOOK ÚNICO
# ------------------------------------------------------------
xlsx_path = TABLES / "Article_Ready_Tables.xlsx"
with pd.ExcelWriter(xlsx_path, engine="openpyxl") as writer:
    table_I.to_excel(writer, sheet_name="Table_I_Protocol", index=False)
    table_II.to_excel(writer, sheet_name="Table_II_Tradeoff", index=False)
    table_III.to_excel(writer, sheet_name="Table_III_Validation", index=False)
    supp_alg.to_excel(writer, sheet_name="Supp_Alg_Full", index=False)
    supp_summary.to_excel(writer, sheet_name="Supp_Alg_Summary", index=False)

# ------------------------------------------------------------
# 10. RELATÓRIO TXT PARA REDAÇÃO
# ------------------------------------------------------------
report = []
report.append("PARTE 16.3 — ARTICLE-READY OUTPUTS\n")
report.append("="*80 + "\n\n")
report.append("TABLE I — EXPERIMENTAL PROTOCOL\n")
report.append(table_I.to_string(index=False))
report.append("\n\nTABLE II — CROSS-ALGORITHM TRADE-OFF\n")
report.append(table_II_print.to_string(index=False))
report.append("\n\nTABLE III — ROBUSTNESS AND STATISTICAL VALIDATION\n")
report.append(table_III.to_string(index=False))
report.append("\n\nSUPPLEMENTARY — ADDITIONAL ALGORITHM SENSITIVITY\n")
report.append(supp_summary.to_string(index=False))
report.append("\n\nCONSENSUS FROM PART 16.1\n")
report.append(consensus.to_string(index=False))
report.append("\n")

(OUT / "Article_Ready_Report.txt").write_text("".join(report), encoding="utf-8")

# ------------------------------------------------------------
# 11. MANIFEST / AUDITORIA
# ------------------------------------------------------------
manifest = {
    "created_at": datetime.now().isoformat(),
    "part16_run": str(PART16),
    "part16_1_run": str(PART16_1),
    "output_run": str(OUT),
    "main_article_outputs": {
        "table_I": str(TABLES / "Table_I_Experimental_Protocol.csv"),
        "table_II": str(TABLES / "Table_II_Cross_Algorithm_Tradeoff.csv"),
        "table_III": str(TABLES / "Table_III_Robustness_Statistical_Validation.csv"),
        "figure_2_png": str(FIGURES / "Fig_2_Structural_Retention_vs_Adjusted_Semantic_Gain.png"),
        "figure_3_png": str(FIGURES / "Fig_3_Robustness_and_External_Validation.png"),
    },
    "supplementary_algorithm_sensitivity": {
        "algorithms": ["Label Propagation", "Walktrap"],
        "conditions": ["topology_only", "semantic_only", "fused"],
        "fused_parameters_inherited_from_primary_grid": {"k": 5, "alpha": 0.75},
        "size_matched_nulls": N_SIZE_MATCHED_SUPP,
        "label_propagation_seeds": SUPP_SEEDS,
        "walktrap_steps": WALKTRAP_STEPS,
        "purpose": "Sensitivity only; not used to reselect k or alpha and not included in primary optimization.",
    },
    "reporting_principle": (
        "Primary article reports Louvain, Leiden, and Infomap in parallel. "
        "Label Propagation and Walktrap are supplementary sensitivity analyses only. "
        "No global algorithm winner is selected."
    ),
}
(LOGS / "manifest_part16_3.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")

# ------------------------------------------------------------
# 12. IMPRESSÃO FINAL
# ------------------------------------------------------------
print("\n" + "="*92)
print("TABLE I — EXPERIMENTAL PROTOCOL")
print("="*92)
print(table_I.to_string(index=False))

print("\n" + "="*92)
print("TABLE II — CROSS-ALGORITHM STRUCTURAL/SEMANTIC TRADE-OFF")
print("="*92)
print(table_II_print.to_string(index=False))

print("\n" + "="*92)
print("TABLE III — ROBUSTNESS AND STATISTICAL VALIDATION")
print("="*92)
print(table_III.to_string(index=False))

print("\n" + "="*92)
print("SUPPLEMENTARY — LABEL PROPAGATION + WALKTRAP")
print("="*92)
print(supp_summary.to_string(index=False))

print("\n" + "="*92)
print("✅ PARTE 16.3 CONCLUÍDA")
print("="*92)
print("Saídas:", OUT)
print("\nArquivos para o corpo principal:")
print("-", TABLES / "Table_I_Experimental_Protocol.csv")
print("-", TABLES / "Table_II_Cross_Algorithm_Tradeoff.csv")
print("-", TABLES / "Table_III_Robustness_Statistical_Validation.csv")
print("-", FIGURES / "Fig_2_Structural_Retention_vs_Adjusted_Semantic_Gain.png")
print("-", FIGURES / "Fig_3_Robustness_and_External_Validation.png")
print("\nSensibilidade suplementar:")
print("-", SUPP / "Table_S_Additional_Algorithm_Sensitivity_Summary.csv")
print("-", SUPP / "Fig_S_Algorithm_Sensitivity_Five_Methods.png")
print("\nRelatório consolidado:")
print("-", OUT / "Article_Ready_Report.txt")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
PARTE 16.3 — ARTICLE-READY OUTPUTS + SUPPLEMENTARY SENSITIVITY
Parte 16:   /content/drive/MyDrive/tese_doutoral/semantic_topological_runs/run_20260920_010404
Parte 16.1: /content/drive/MyDrive/tese_doutoral/semantic_topological_runs/run_20260920_010404/posthoc_16_1/run_20260920_020905
Saída 16.3: /content/drive/MyDrive/tese_doutoral/semantic_topological_runs/run_20260920_010404/article_ready_16_3/run_20260921_135029
Schema institucional detectado:
  V corrigido: institution_cramers_v_bias_corrected
  p empírico: institution_vcorr_p_emp

SENSIBILIDADE SUPLEMENTAR — LABEL PROPAGATION E WALKTRAP

TABLE I — EXPERIMENTAL PROTOCOL
                           Item                                                                      Setting
               Research network                      510 identified researchers; GCC 404 nodes / 1,609 edges
               Publi

In [ ]:
# ============================================================
# PARTE 16.4 — REVIEWER-ORIENTED SENSITIVITY AND INTERPRETABILITY
# ============================================================
# Objetivos:
# 1) Refinar/estender a grade k x alpha nas bordas do ótimo observado.
# 2) Avaliar sensibilidade ao limiar de retenção estrutural.
# 3) Contextualizar a magnitude do ganho semântico.
# 4) Produzir exemplos qualitativos auditáveis de comunidades.
# 5) Testar sensibilidade da transformação log1p dos pesos topológicos.
# ============================================================

import os
import sys
import json
import math
import time
import pickle
import random
import subprocess
from pathlib import Path
from datetime import datetime
from itertools import combinations
from collections import defaultdict

# ------------------------------------------------------------
# 0. DEPENDÊNCIAS
# ------------------------------------------------------------
def ensure_packages():
    pkgs = ["python-louvain", "igraph", "leidenalg", "openpyxl"]
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + pkgs)

ensure_packages()

import numpy as np
import pandas as pd
import networkx as nx
from scipy.optimize import linear_sum_assignment
from sklearn.metrics import normalized_mutual_info_score, adjusted_rand_score
import community as community_louvain
import igraph as ig
import leidenalg as la
import matplotlib.pyplot as plt

# ------------------------------------------------------------
# 0.1 DRIVE / CONFIGURAÇÃO
# ------------------------------------------------------------
try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
except Exception:
    pass

PROJECT_ROOT = Path("/content/drive/MyDrive/tese_doutoral")
SEM_ROOT = PROJECT_ROOT / "semantic_topological_runs"
PREFERRED_PART16 = SEM_ROOT / "run_20260920_010404"

GLOBAL_SEED = 42
SEEDS = [42, 52, 62, 72, 82, 92, 102, 112, 122, 132]
ALGORITHMS = ["louvain", "leiden", "infomap"]
LOUVAIN_RESOLUTION = 1.0
LEIDEN_RESOLUTION = 1.0
INFOMAP_TRIALS = 10

# Revisão solicitada
RETENTION_THRESHOLDS = [0.90, 0.925, 0.95, 0.975]

# Refinamento em torno das bordas observadas:
# - k < 5 para verificar o limite inferior;
# - k = 25 para verificar extensão além da grade original;
# - alpha > .75 para verificar o limite superior.
REFINE_K = [3, 4, 5, 6, 8, 25]
REFINE_ALPHA = [0.65, 0.70, 0.75, 0.80, 0.85, 0.90]

N_SIZE_NULL = 999

# Sensibilidade da transformação topológica, mantendo k=5, alpha=.75 fixos.
WEIGHT_TRANSFORMS = ["raw", "sqrt", "log1p"]
WEIGHT_SENS_K = 5
WEIGHT_SENS_ALPHA = 0.75

# Parâmetros exatos do protocolo original
SEMANTIC_NOISE_SIGMA = 0.03
TOPOLOGY_EDGE_DROPOUT = 0.10
MIN_SEMANTIC_SIMILARITY = 0.0

# Exemplos qualitativos
QUAL_REFERENCE_ALGO = "infomap"
QUAL_K = 5
QUAL_ALPHA = 0.75
QUAL_MIN_COMMUNITY_SIZE = 5
QUAL_N_POSITIVE_EXAMPLES = 3
QUAL_N_CONTRAST_EXAMPLES = 1
QUAL_TITLES_PER_COMMUNITY = 4

random.seed(GLOBAL_SEED)
np.random.seed(GLOBAL_SEED)

# ------------------------------------------------------------
# 0.2 LOCALIZAÇÃO AUTOMÁTICA DOS RUNS
# ------------------------------------------------------------
def newest_dir(parent, pattern="run_*"):
    ds = [p for p in Path(parent).glob(pattern) if p.is_dir()]
    return max(ds, key=lambda p: p.stat().st_mtime) if ds else None

def valid_part16(p):
    p = Path(p)
    return (
        (p / "tables" / "table16_3_grid_aggregated.csv").exists()
        and (p / "artifacts" / "topology_graph_gcc_common_nodes.pkl").exists()
        and (p / "artifacts" / "researcher_embeddings.npy").exists()
        and (p / "artifacts" / "researcher_cosine_similarity.npy").exists()
    )

if valid_part16(PREFERRED_PART16):
    PART16 = PREFERRED_PART16
else:
    candidates = [p for p in SEM_ROOT.glob("run_*") if valid_part16(p)]
    if not candidates:
        raise FileNotFoundError("Nenhum run válido da Parte 16 foi encontrado.")
    PART16 = max(candidates, key=lambda p: p.stat().st_mtime)

POSTHOC_ROOT = PART16 / "posthoc_16_1"
PART16_1 = newest_dir(POSTHOC_ROOT)
if PART16_1 is None:
    raise FileNotFoundError("Nenhum run da Parte 16.1 foi encontrado.")

OUT = PART16 / "reviewer_revision_16_4" / f"run_{datetime.now():%Y%m%d_%H%M%S}"
TABLES = OUT / "tables"
FIGURES = OUT / "figures"
SUPP = OUT / "supplementary"
LOGS = OUT / "logs"
for d in [TABLES, FIGURES, SUPP, LOGS]:
    d.mkdir(parents=True, exist_ok=True)

print("="*100)
print("PARTE 16.4 — REVIEWER-ORIENTED SENSITIVITY AND INTERPRETABILITY")
print("="*100)
print("Parte 16:  ", PART16)
print("Parte 16.1:", PART16_1)
print("Saída 16.4:", OUT)

# ------------------------------------------------------------
# 1. CARREGAR ARTEFATOS E TABELAS
# ------------------------------------------------------------
grid_runs = pd.read_csv(PART16 / "tables" / "table16_2_grid_runs.csv")
grid_agg = pd.read_csv(PART16 / "tables" / "table16_3_grid_aggregated.csv")
semantic_adjusted = pd.read_csv(
    PART16_1 / "tables" / "table16_1a_granularity_adjusted_semantics.csv"
)

with open(PART16 / "artifacts" / "topology_graph_gcc_common_nodes.pkl", "rb") as f:
    G_top_log = pickle.load(f)

researchers = pd.read_csv(PART16 / "intermediate" / "researcher_semantic_coverage.csv")
node_order = researchers["researcher"].astype(str).tolist()
researcher_embeddings = np.load(PART16 / "artifacts" / "researcher_embeddings.npy").astype(np.float32)
semantic_similarity = np.load(PART16 / "artifacts" / "researcher_cosine_similarity.npy").astype(np.float32)

# Garante nomes string no grafo
if set(node_order) != set(G_top_log.nodes()):
    G_top_log = nx.relabel_nodes(G_top_log, {n: str(n) for n in G_top_log.nodes()}, copy=True)

if set(node_order) != set(G_top_log.nodes()):
    raise RuntimeError("A ordem/lista de pesquisadores não coincide com o grafo topológico.")

node_to_idx = {n:i for i,n in enumerate(node_order)}

# ------------------------------------------------------------
# 2. FUNÇÕES CENTRAIS
# ------------------------------------------------------------
def normalize_graph_weight_mass(G, transform=None):
    H = nx.Graph()
    H.add_nodes_from(G.nodes())
    vals = []
    for u, v, d in G.edges(data=True):
        raw = float(d.get("raw_weight", d.get("weight", 1.0)))
        w = transform(raw) if transform is not None else raw
        w = max(float(w), 0.0)
        if w > 0:
            vals.append((u, v, w, raw))
    total = sum(x[2] for x in vals)
    if total <= 0:
        return H
    for u, v, w, raw in vals:
        H.add_edge(u, v, weight=w/total, raw_weight=raw)
    return H

def topology_from_existing_raw_weights(G_log, transform_name):
    # O artefato topológico preserva raw_weight em cada aresta.
    Hraw = nx.Graph()
    Hraw.add_nodes_from(G_log.nodes())
    for u, v, d in G_log.edges(data=True):
        raw = float(d.get("raw_weight", 1.0))
        Hraw.add_edge(u, v, weight=raw, raw_weight=raw)

    if transform_name == "raw":
        return normalize_graph_weight_mass(Hraw, transform=lambda x: x)
    if transform_name == "sqrt":
        return normalize_graph_weight_mass(Hraw, transform=lambda x: math.sqrt(x))
    if transform_name == "log1p":
        return normalize_graph_weight_mass(Hraw, transform=lambda x: math.log1p(x))
    raise ValueError(transform_name)

def build_semantic_knn_graph(nodes, sim_matrix, k, min_similarity=0.0):
    n = len(nodes)
    k_eff = min(int(k), n-1)
    H = nx.Graph()
    H.add_nodes_from(nodes)
    for i, node in enumerate(nodes):
        sims = sim_matrix[i].copy()
        sims[i] = -np.inf
        idx = np.argpartition(-sims, k_eff)[:k_eff]
        idx = idx[np.argsort(-sims[idx])]
        for j in idx:
            s = float(sims[j])
            if s < min_similarity:
                continue
            u, v = node, nodes[int(j)]
            if u == v:
                continue
            if H.has_edge(u, v):
                H[u][v]["weight"] = max(float(H[u][v]["weight"]), s)
            else:
                H.add_edge(u, v, weight=s)
    return normalize_graph_weight_mass(H, transform=None)

def build_fused_graph(Gt, Gs, alpha):
    H = nx.Graph()
    H.add_nodes_from(Gt.nodes())
    edge_keys = {tuple(sorted((u,v))) for u,v in Gt.edges()}
    edge_keys.update(tuple(sorted((u,v))) for u,v in Gs.edges())
    for u, v in edge_keys:
        wt = float(Gt[u][v]["weight"]) if Gt.has_edge(u,v) else 0.0
        ws = float(Gs[u][v]["weight"]) if Gs.has_edge(u,v) else 0.0
        wf = float(alpha)*wt + (1.0-float(alpha))*ws
        if wf > 0:
            H.add_edge(u, v, weight=wf, topo_weight=wt, semantic_weight=ws)
    return H

def nx_to_igraph(G):
    nodes = list(G.nodes())
    idx = {n:i for i,n in enumerate(nodes)}
    edges = [(idx[u], idx[v]) for u,v in G.edges()]
    weights = [float(G[u][v].get("weight",1.0)) for u,v in G.edges()]
    gi = ig.Graph(n=len(nodes), edges=edges, directed=False)
    gi.vs["name"] = nodes
    gi.es["weight"] = weights
    return gi, nodes

def relabel_partition(partition):
    groups = defaultdict(list)
    for n,c in partition.items():
        groups[c].append(n)
    ordered = sorted(groups.values(), key=lambda g: (-len(g), sorted(map(str,g))[0]))
    out = {}
    for new_c, group in enumerate(ordered):
        for n in group:
            out[n] = new_c
    return out

def detect_partition(G, algorithm, seed):
    if G.number_of_edges() == 0:
        return {n:i for i,n in enumerate(G.nodes())}
    if algorithm == "louvain":
        p = community_louvain.best_partition(
            G, weight="weight", resolution=LOUVAIN_RESOLUTION, random_state=int(seed)
        )
        return relabel_partition(p)

    gi, nodes = nx_to_igraph(G)
    if algorithm == "leiden":
        part = la.find_partition(
            gi, la.ModularityVertexPartition,
            weights="weight", n_iterations=-1, seed=int(seed)
        )
        return relabel_partition({nodes[i]:int(part.membership[i]) for i in range(len(nodes))})
    if algorithm == "infomap":
        rng = random.Random(int(seed))
        ig.set_random_number_generator(rng)
        part = gi.community_infomap(edge_weights="weight", trials=INFOMAP_TRIALS)
        return relabel_partition({nodes[i]:int(part.membership[i]) for i in range(len(nodes))})
    raise ValueError(algorithm)

def labels_from_partition(partition):
    return np.array([int(partition[n]) for n in node_order], dtype=int)

def groups_from_labels(labels):
    return [np.flatnonzero(labels == c) for c in np.unique(labels)]

def partition_sets(partition):
    d = defaultdict(set)
    for n,c in partition.items():
        d[c].add(n)
    return list(d.values())

def structural_modularity(partition, G_eval):
    return float(nx.algorithms.community.quality.modularity(
        G_eval, partition_sets(partition), weight="weight"
    ))

def semantic_centroid_coherence_from_labels(labels):
    vals = []
    for idx in groups_from_labels(labels):
        if len(idx) == 0:
            continue
        centroid = researcher_embeddings[idx].mean(axis=0)
        norm = np.linalg.norm(centroid)
        if norm <= 0:
            continue
        centroid = centroid / norm
        vals.extend((researcher_embeddings[idx] @ centroid).tolist())
    return float(np.mean(vals)) if vals else np.nan

def semantic_centroid_coherence(partition):
    return semantic_centroid_coherence_from_labels(labels_from_partition(partition))

def conductance_mean(partition, G_eval):
    comms = partition_sets(partition)
    deg = dict(G_eval.degree(weight="weight"))
    total_vol = sum(deg.values())
    vals = []
    for comm in comms:
        if len(comm) == 0 or len(comm) == G_eval.number_of_nodes():
            continue
        vol_s = sum(deg.get(n,0.0) for n in comm)
        vol_t = total_vol - vol_s
        denom = min(vol_s, vol_t)
        if denom <= 0:
            continue
        cut = 0.0
        for u in comm:
            for v,d in G_eval[u].items():
                if v not in comm:
                    cut += float(d.get("weight",1.0))
        vals.append(cut/denom)
    return float(np.mean(vals)) if vals else np.nan

def pairwise_stability(partitions):
    if len(partitions) < 2:
        return np.nan, np.nan
    labs = [labels_from_partition(p) for p in partitions]
    nmis, aris = [], []
    for i in range(len(labs)):
        for j in range(i+1, len(labs)):
            nmis.append(normalized_mutual_info_score(labs[i], labs[j]))
            aris.append(adjusted_rand_score(labs[i], labs[j]))
    return float(np.mean(nmis)), float(np.mean(aris))

def empirical_p(observed, vals):
    vals = np.asarray(vals, float)
    vals = vals[np.isfinite(vals)]
    return float((1 + np.sum(vals >= observed))/(len(vals)+1)) if len(vals) else np.nan

def size_matched_centroid_null(partition, n_rep=999, seed=42):
    labels = labels_from_partition(partition)
    sizes = [len(g) for g in groups_from_labels(labels)]
    obs = semantic_centroid_coherence_from_labels(labels)
    rng = np.random.default_rng(int(seed))
    null = np.empty(int(n_rep), dtype=float)
    n = len(labels)
    for b in range(int(n_rep)):
        perm = rng.permutation(n)
        rand_labels = np.empty(n, dtype=int)
        pos = 0
        for c,size in enumerate(sizes):
            ix = perm[pos:pos+size]
            rand_labels[ix] = c
            pos += size
        null[b] = semantic_centroid_coherence_from_labels(rand_labels)
    mean = float(np.mean(null))
    sd = float(np.std(null, ddof=1))
    excess = float(obs - mean)
    headroom = float(excess / max(1e-12, 1.0-mean))
    z = float((obs-mean)/sd) if sd > 0 else np.nan
    return {
        "semantic_centroid_coherence": obs,
        "size_null_centroid_mean": mean,
        "size_null_centroid_sd": sd,
        "centroid_excess_over_size_null": excess,
        "centroid_headroom_adjusted_gain": headroom,
        "centroid_size_null_z": z,
        "centroid_size_null_p_emp": empirical_p(obs, null),
    }

# ------------------------------------------------------------
# 3. TOPOLOGY BASELINES E GRAFOS SEMÂNTICOS NOVOS
# ------------------------------------------------------------
G_top = G_top_log  # baseline original log1p normalizado

topology_seed42 = {}
topology_excess = {}
topology_mod_seed42 = {}

for alg in ALGORITHMS:
    p = detect_partition(G_top, alg, GLOBAL_SEED)
    topology_seed42[alg] = p
    topology_mod_seed42[alg] = structural_modularity(p, G_top)

    r = semantic_adjusted[
        (semantic_adjusted["algorithm"].str.lower()==alg)
        & (semantic_adjusted["condition"]=="topology_only")
    ]
    if r.empty:
        # fallback: recalcula nulo se necessário
        nullm = size_matched_centroid_null(p, N_SIZE_NULL, GLOBAL_SEED + 100)
        topology_excess[alg] = nullm["centroid_excess_over_size_null"]
    else:
        topology_excess[alg] = float(r.iloc[0]["centroid_excess_over_size_null"])

semantic_graph_cache = {}
def get_semantic_graph(k):
    k = int(k)
    if k not in semantic_graph_cache:
        semantic_graph_cache[k] = build_semantic_knn_graph(
            node_order, semantic_similarity, k, MIN_SEMANTIC_SIMILARITY
        )
    return semantic_graph_cache[k]

# ------------------------------------------------------------
# 4. REFINAMENTO / EXTENSÃO DA GRADE k x alpha
# ------------------------------------------------------------
print("\n" + "="*100)
print("A. REFINAMENTO / EXTENSÃO DA GRADE k x alpha")
print("="*100)

refine_rows = []
refine_partitions = defaultdict(list)

for k in REFINE_K:
    Gs = get_semantic_graph(k)
    for alpha in REFINE_ALPHA:
        Gf = build_fused_graph(G_top, Gs, alpha)
        for alg in ALGORITHMS:
            key = (alg, int(k), float(alpha))
            print(f"k={k:2d}, alpha={alpha:.2f}, alg={alg}")
            for seed in SEEDS:
                t0 = time.perf_counter()
                p = detect_partition(Gf, alg, seed)
                elapsed = time.perf_counter() - t0
                refine_partitions[key].append(p)
                refine_rows.append({
                    "algorithm": alg,
                    "k": int(k),
                    "alpha": float(alpha),
                    "seed": int(seed),
                    "n_communities": len(partition_sets(p)),
                    "structural_modularity": structural_modularity(p, G_top),
                    "semantic_centroid_coherence": semantic_centroid_coherence(p),
                    "conductance_mean": conductance_mean(p, G_top),
                    "runtime_seconds": elapsed,
                })

refine_runs = pd.DataFrame(refine_rows)
refine_runs.to_csv(TABLES / "Table16_4A_Refined_Grid_Runs.csv", index=False)

agg_rows = []
for (alg,k,alpha), g in refine_runs.groupby(["algorithm","k","alpha"]):
    parts = refine_partitions[(alg,int(k),float(alpha))]
    nmi, ari = pairwise_stability(parts)
    agg_rows.append({
        "algorithm": alg,
        "k": int(k),
        "alpha": float(alpha),
        "n_runs": len(g),
        "n_communities_mean": g["n_communities"].mean(),
        "n_communities_sd": g["n_communities"].std(ddof=1),
        "structural_modularity_mean": g["structural_modularity"].mean(),
        "structural_modularity_sd": g["structural_modularity"].std(ddof=1),
        "semantic_centroid_coherence_mean": g["semantic_centroid_coherence"].mean(),
        "semantic_centroid_coherence_sd": g["semantic_centroid_coherence"].std(ddof=1),
        "conductance_mean_mean": g["conductance_mean"].mean(),
        "conductance_mean_sd": g["conductance_mean"].std(ddof=1),
        "runtime_seconds_mean": g["runtime_seconds"].mean(),
        "runtime_seconds_sd": g["runtime_seconds"].std(ddof=1),
        "stability_nmi_mean": nmi,
        "stability_ari_mean": ari,
    })
refine_agg = pd.DataFrame(agg_rows)
refine_agg.to_csv(TABLES / "Table16_4B_Refined_Grid_Aggregated.csv", index=False)

# ------------------------------------------------------------
# 5. UNIR GRADE ORIGINAL + NOVA E REMOVER DUPLICATAS
# ------------------------------------------------------------
orig_fused = grid_agg[grid_agg["condition"]=="fused"].copy()
keep_cols = [
    "algorithm","k","alpha","structural_modularity_mean",
    "structural_modularity_sd","semantic_centroid_coherence_mean",
    "semantic_centroid_coherence_sd","conductance_mean_mean",
    "stability_nmi_mean","stability_ari_mean"
]
orig_fused = orig_fused[keep_cols].copy()
orig_fused["source_grid"] = "original"

refined_for_union = refine_agg[keep_cols].copy()
refined_for_union["source_grid"] = "refined"

candidate_union = pd.concat([orig_fused, refined_for_union], ignore_index=True)
candidate_union["k"] = candidate_union["k"].astype(int)
candidate_union["alpha_round"] = candidate_union["alpha"].round(10)

# Para duplicatas, mantém o refinamento recém-executado.
candidate_union["source_priority"] = candidate_union["source_grid"].map(
    {"original":0, "refined":1}
)
candidate_union = (
    candidate_union.sort_values("source_priority")
    .drop_duplicates(["algorithm","k","alpha_round"], keep="last")
    .drop(columns=["source_priority","alpha_round"])
    .reset_index(drop=True)
)
candidate_union.to_csv(TABLES / "Table16_4C_Expanded_Candidate_Grid.csv", index=False)

# ------------------------------------------------------------
# 6. SENSIBILIDADE AO LIMIAR DE RETENÇÃO
# ------------------------------------------------------------
print("\n" + "="*100)
print("B. SENSIBILIDADE AO LIMIAR DE RETENÇÃO ESTRUTURAL")
print("="*100)

null_cache = {}

def adjusted_candidate_stats(alg, k, alpha, seed_offset=0):
    key = (alg, int(k), round(float(alpha),10))
    if key in null_cache:
        return null_cache[key]
    Gs = get_semantic_graph(int(k))
    Gf = build_fused_graph(G_top, Gs, float(alpha))
    p = detect_partition(Gf, alg, GLOBAL_SEED)
    q = structural_modularity(p, G_top)
    sem = size_matched_centroid_null(
        p, N_SIZE_NULL,
        seed=GLOBAL_SEED + 10000 + seed_offset + int(k)*101 + int(round(alpha*1000))
    )
    out = {
        "seed42_n_communities": len(partition_sets(p)),
        "seed42_modularity": q,
        "seed42_retention_ratio": q/topology_mod_seed42[alg],
        **sem,
        "delta_SC_adj_vs_topology": sem["centroid_excess_over_size_null"] - topology_excess[alg],
    }
    null_cache[key] = out
    return out

threshold_rows = []
selected_95 = {}

for tau_i, tau in enumerate(RETENTION_THRESHOLDS):
    for alg in ALGORITHMS:
        base_mean_row = grid_agg[
            (grid_agg["condition"]=="topology_only")
            & (grid_agg["algorithm"]==alg)
        ]
        base_mean = float(base_mean_row.iloc[0]["structural_modularity_mean"])

        cand = candidate_union[candidate_union["algorithm"]==alg].copy()
        cand["mean_retention_ratio"] = cand["structural_modularity_mean"] / base_mean
        eligible = cand[cand["mean_retention_ratio"] >= tau].copy()

        if eligible.empty:
            best_ret = cand.sort_values("mean_retention_ratio", ascending=False).iloc[0]
            threshold_rows.append({
                "threshold": tau,
                "algorithm": alg,
                "constraint_met": False,
                "selected_k": np.nan,
                "selected_alpha": np.nan,
                "source_grid": "",
                "mean_retention_ratio": np.nan,
                "seed42_retention_ratio": np.nan,
                "delta_SC_adj_vs_topology": np.nan,
                "fused_excess_over_size_null": np.nan,
                "fused_p_emp": np.nan,
                "best_available_retention_ratio": float(best_ret["mean_retention_ratio"]),
            })
            continue

        eligible = eligible.sort_values(
            ["semantic_centroid_coherence_mean","stability_nmi_mean","conductance_mean_mean"],
            ascending=[False,False,True]
        )
        s = eligible.iloc[0]
        k = int(s["k"])
        alpha = float(s["alpha"])
        adj = adjusted_candidate_stats(alg,k,alpha,seed_offset=tau_i*1000)

        threshold_rows.append({
            "threshold": tau,
            "algorithm": alg,
            "constraint_met": True,
            "selected_k": k,
            "selected_alpha": alpha,
            "source_grid": s["source_grid"],
            "mean_retention_ratio": float(s["mean_retention_ratio"]),
            "seed42_retention_ratio": float(adj["seed42_retention_ratio"]),
            "seed42_n_communities": int(adj["seed42_n_communities"]),
            "delta_SC_adj_vs_topology": float(adj["delta_SC_adj_vs_topology"]),
            "fused_excess_over_size_null": float(adj["centroid_excess_over_size_null"]),
            "fused_headroom_adjusted_gain": float(adj["centroid_headroom_adjusted_gain"]),
            "fused_p_emp": float(adj["centroid_size_null_p_emp"]),
            "best_available_retention_ratio": float(cand["mean_retention_ratio"].max()),
        })

        if np.isclose(tau, 0.95):
            selected_95[alg] = (k,alpha,s["source_grid"],adj)

threshold_df = pd.DataFrame(threshold_rows)
threshold_df.to_csv(TABLES / "Table16_4D_Retention_Threshold_Sensitivity.csv", index=False)

print(threshold_df.to_string(index=False))

# ------------------------------------------------------------
# 7. COMPARAÇÃO ÓTIMO ORIGINAL vs GRADE EXPANDIDA (tau=.95)
# ------------------------------------------------------------
original_selection = pd.read_csv(PART16 / "tables" / "table16_4_selected_by_algorithm.csv")

comparison_rows = []
for alg in ALGORITHMS:
    o = original_selection[original_selection["algorithm"]==alg].iloc[0]
    newrow = threshold_df[
        (threshold_df["algorithm"]==alg)
        & np.isclose(threshold_df["threshold"],0.95)
    ].iloc[0]

    comparison_rows.append({
        "algorithm": alg,
        "original_k": int(round(float(o["k"]))),
        "original_alpha": float(o["alpha"]),
        "expanded_k": int(newrow["selected_k"]) if pd.notna(newrow["selected_k"]) else np.nan,
        "expanded_alpha": float(newrow["selected_alpha"]) if pd.notna(newrow["selected_alpha"]) else np.nan,
        "expanded_source_grid": newrow["source_grid"],
        "same_k": bool(pd.notna(newrow["selected_k"]) and int(round(float(o["k"])))==int(newrow["selected_k"])),
        "same_alpha": bool(pd.notna(newrow["selected_alpha"]) and np.isclose(float(o["alpha"]),float(newrow["selected_alpha"]))),
        "expanded_seed42_retention": newrow["seed42_retention_ratio"],
        "expanded_delta_SC_adj": newrow["delta_SC_adj_vs_topology"],
        "expanded_p_emp": newrow["fused_p_emp"],
    })

selection_comparison = pd.DataFrame(comparison_rows)
selection_comparison.to_csv(TABLES / "Table16_4E_Original_vs_Expanded_Selection.csv", index=False)

# ------------------------------------------------------------
# 8. CONTEXTUALIZAÇÃO DO TAMANHO DE EFEITO
# ------------------------------------------------------------
print("\n" + "="*100)
print("C. CONTEXTUALIZAÇÃO DO TAMANHO DE EFEITO")
print("="*100)

effect_rows = []
for alg in ALGORITHMS:
    topo = semantic_adjusted[
        (semantic_adjusted["algorithm"]==alg)
        & (semantic_adjusted["condition"]=="topology_only")
    ].iloc[0]
    fused = semantic_adjusted[
        (semantic_adjusted["algorithm"]==alg)
        & (semantic_adjusted["condition"]=="fused_selected")
    ].iloc[0]

    delta_excess = float(
        fused["centroid_excess_over_size_null"] - topo["centroid_excess_over_size_null"]
    )
    relative_increase = (
        100.0 * delta_excess / float(topo["centroid_excess_over_size_null"])
        if float(topo["centroid_excess_over_size_null"]) != 0 else np.nan
    )
    effect_rows.append({
        "algorithm": alg,
        "topology_raw_coherence": float(topo["semantic_centroid_coherence"]),
        "fused_raw_coherence": float(fused["semantic_centroid_coherence"]),
        "raw_coherence_delta": float(
            fused["semantic_centroid_coherence"] - topo["semantic_centroid_coherence"]
        ),
        "topology_excess_over_size_null": float(topo["centroid_excess_over_size_null"]),
        "fused_excess_over_size_null": float(fused["centroid_excess_over_size_null"]),
        "delta_adjusted_excess": delta_excess,
        "relative_increase_in_excess_pct": relative_increase,
        "fused_headroom_adjusted_gain": float(fused["centroid_headroom_adjusted_gain"]),
        "fused_size_null_sd": float(fused["size_null_centroid_sd"]),
        "fused_size_null_z": float(fused["centroid_size_null_z"]),
        "fused_p_emp": float(fused["centroid_size_null_p_emp"]),
    })

effect_df = pd.DataFrame(effect_rows)
effect_df.to_csv(TABLES / "Table16_4F_Effect_Size_Context.csv", index=False)
print(effect_df.to_string(index=False))

# ------------------------------------------------------------
# 9. SENSIBILIDADE À TRANSFORMAÇÃO DOS PESOS TOPOLÓGICOS
# ------------------------------------------------------------
print("\n" + "="*100)
print("D. SENSIBILIDADE À TRANSFORMAÇÃO DOS PESOS (raw / sqrt / log1p)")
print("="*100)

Gs_fixed = get_semantic_graph(WEIGHT_SENS_K)
weight_rows = []
weight_seed42_cache = {}

for transform_name in WEIGHT_TRANSFORMS:
    Gt = topology_from_existing_raw_weights(G_top_log, transform_name)

    for alg in ALGORITHMS:
        # topology-only
        topo_parts = []
        topo_run = []
        for seed in SEEDS:
            p = detect_partition(Gt, alg, seed)
            topo_parts.append(p)
            topo_run.append({
                "q": structural_modularity(p,Gt),
                "sc": semantic_centroid_coherence(p),
            })
        qtop_mean = float(np.mean([x["q"] for x in topo_run]))

        # fused fixo k=5 alpha=.75
        Gf = build_fused_graph(Gt, Gs_fixed, WEIGHT_SENS_ALPHA)
        fused_parts = []
        fused_run = []
        for seed in SEEDS:
            p = detect_partition(Gf, alg, seed)
            fused_parts.append(p)
            fused_run.append({
                "q": structural_modularity(p,Gt),
                "sc": semantic_centroid_coherence(p),
            })
        qf_mean = float(np.mean([x["q"] for x in fused_run]))
        nmi, ari = pairwise_stability(fused_parts)

        # adjusted semantic gain no seed42
        pt = detect_partition(Gt, alg, GLOBAL_SEED)
        pf = detect_partition(Gf, alg, GLOBAL_SEED)
        topo_null = size_matched_centroid_null(
            pt, N_SIZE_NULL, seed=GLOBAL_SEED + 21000 + WEIGHT_TRANSFORMS.index(transform_name)*100 + ALGORITHMS.index(alg)
        )
        fused_null = size_matched_centroid_null(
            pf, N_SIZE_NULL, seed=GLOBAL_SEED + 22000 + WEIGHT_TRANSFORMS.index(transform_name)*100 + ALGORITHMS.index(alg)
        )

        weight_rows.append({
            "weight_transform": transform_name,
            "algorithm": alg,
            "k_fixed": WEIGHT_SENS_K,
            "alpha_fixed": WEIGHT_SENS_ALPHA,
            "topology_modularity_mean": qtop_mean,
            "fused_modularity_mean": qf_mean,
            "mean_retention_ratio": qf_mean/qtop_mean if qtop_mean else np.nan,
            "seed42_topology_excess": topo_null["centroid_excess_over_size_null"],
            "seed42_fused_excess": fused_null["centroid_excess_over_size_null"],
            "seed42_delta_SC_adj": (
                fused_null["centroid_excess_over_size_null"] -
                topo_null["centroid_excess_over_size_null"]
            ),
            "fused_p_emp": fused_null["centroid_size_null_p_emp"],
            "fused_stability_nmi": nmi,
            "fused_stability_ari": ari,
        })

weight_df = pd.DataFrame(weight_rows)
weight_df.to_csv(TABLES / "Table16_4G_Weight_Transform_Sensitivity.csv", index=False)
print(weight_df.to_string(index=False))

# ------------------------------------------------------------
# 10. EXEMPLOS QUALITATIVOS — TOPOLOGY VS FUSED
# ------------------------------------------------------------
print("\n" + "="*100)
print("E. EXEMPLOS QUALITATIVOS DE COMUNIDADES")
print("="*100)

G_sem_qual = get_semantic_graph(QUAL_K)
G_fused_qual = build_fused_graph(G_top, G_sem_qual, QUAL_ALPHA)

p_top = detect_partition(G_top, QUAL_REFERENCE_ALGO, GLOBAL_SEED)
p_fused = detect_partition(G_fused_qual, QUAL_REFERENCE_ALGO, GLOBAL_SEED)

def community_members(partition):
    d = defaultdict(set)
    for n,c in partition.items():
        d[int(c)].add(n)
    return dict(d)

def community_coherence(members):
    idx = np.array([node_to_idx[n] for n in members], dtype=int)
    if len(idx)==0:
        return np.nan
    centroid = researcher_embeddings[idx].mean(axis=0)
    norm = np.linalg.norm(centroid)
    if norm<=0:
        return np.nan
    centroid /= norm
    return float(np.mean(researcher_embeddings[idx] @ centroid))

top_comms = community_members(p_top)
fused_comms = community_members(p_fused)

top_ids = sorted(top_comms)
fused_ids = sorted(fused_comms)

# matriz de Jaccard
J = np.zeros((len(fused_ids), len(top_ids)), dtype=float)
for i,fc in enumerate(fused_ids):
    A = fused_comms[fc]
    for j,tc in enumerate(top_ids):
        B = top_comms[tc]
        union = len(A|B)
        J[i,j] = len(A&B)/union if union else 0.0

# matching one-to-one maximizando Jaccard
ri, cj = linear_sum_assignment(-J)
match_map = {fused_ids[i]: top_ids[j] for i,j in zip(ri,cj)}

qual_rows = []
for fc in fused_ids:
    if fc not in match_map:
        continue
    tc = match_map[fc]
    fm = fused_comms[fc]
    tm = top_comms[tc]
    fcoh = community_coherence(fm)
    tcoh = community_coherence(tm)
    qual_rows.append({
        "fused_community": fc,
        "topology_community_match": tc,
        "fused_size": len(fm),
        "topology_size": len(tm),
        "jaccard_overlap": len(fm&tm)/len(fm|tm) if len(fm|tm) else np.nan,
        "topology_community_coherence": tcoh,
        "fused_community_coherence": fcoh,
        "delta_community_coherence": fcoh-tcoh,
    })

qual_full = pd.DataFrame(qual_rows)
qual_full.to_csv(SUPP / "Table_S_Community_Matching_Full.csv", index=False)

# Carrega publicações e embeddings para títulos representativos
mapping_path = PART16 / "intermediate" / "publication_author_mapping.jsonl"
mapping_records = []
with open(mapping_path, "r", encoding="utf-8") as f:
    for line in f:
        if line.strip():
            mapping_records.append(json.loads(line))
mapping_df = pd.DataFrame(mapping_records).drop_duplicates("row_index")

cache_npy = sorted((PART16 / "cache").glob("publication_embeddings_*.npy"))
cache_csv = sorted((PART16 / "cache").glob("publication_embeddings_*.csv"))
if not cache_npy or not cache_csv:
    raise FileNotFoundError("Cache de embeddings de publicações não encontrado no run da Parte 16.")

# associa por stem quando possível
emb_path = cache_npy[0]
meta_path = PART16 / "cache" / (emb_path.stem + ".csv")
if not meta_path.exists():
    meta_path = cache_csv[0]

pub_emb = np.load(emb_path).astype(np.float32)
pub_meta = pd.read_csv(meta_path)

if len(pub_emb) != len(pub_meta):
    raise RuntimeError("Número de embeddings de publicação difere do metadata cache.")

pub_lookup = pub_meta.merge(
    mapping_df[["row_index","gcc_authors","title","year"]],
    on="row_index", how="left", suffixes=("_cache","")
)

def normalize_title(t):
    return " ".join(str(t).lower().split())

def representative_titles(members, n_titles=4):
    members = set(members)
    positions = []
    for pos, row in pub_lookup.iterrows():
        authors = row.get("gcc_authors", [])
        if isinstance(authors, str):
            try:
                # json lines vira lista, mas merge/csv pode converter string
                authors = json.loads(authors.replace("'", '"'))
            except Exception:
                authors = []
        if not isinstance(authors, list):
            authors = []
        if members.intersection(set(map(str,authors))):
            positions.append(pos)

    if not positions:
        return []

    idx_members = np.array([node_to_idx[n] for n in members if n in node_to_idx], dtype=int)
    centroid = researcher_embeddings[idx_members].mean(axis=0)
    norm = np.linalg.norm(centroid)
    if norm <= 0:
        return []
    centroid = centroid/norm

    scores = pub_emb[positions] @ centroid
    order = np.argsort(-scores)

    out = []
    seen = set()
    for oi in order:
        pos = positions[int(oi)]
        title = pub_lookup.iloc[pos].get("title", None)
        if pd.isna(title) or not str(title).strip():
            title = pub_lookup.iloc[pos].get("title_cache", "")
        title = str(title).strip()
        key = normalize_title(title)
        if not key or key in seen:
            continue
        seen.add(key)
        year = pub_lookup.iloc[pos].get("year", None)
        out.append({
            "title": title,
            "year": None if pd.isna(year) else year,
            "centroid_similarity": float(scores[int(oi)]),
        })
        if len(out) >= n_titles:
            break
    return out

eligible = qual_full[qual_full["fused_size"] >= QUAL_MIN_COMMUNITY_SIZE].copy()
positive = eligible.sort_values("delta_community_coherence", ascending=False).head(QUAL_N_POSITIVE_EXAMPLES)
contrast = eligible.sort_values("delta_community_coherence", ascending=True).head(QUAL_N_CONTRAST_EXAMPLES)

selected_examples = pd.concat([positive, contrast], ignore_index=True).drop_duplicates(
    ["fused_community","topology_community_match"]
)

example_rows = []
md = ["# Qualitative community examples\n",
      "Reference algorithm: Infomap, seed 42; topology-only vs fused k=5, alpha=0.75.\n",
      "Examples are selected transparently: the largest positive community-level coherence changes "
      f"among communities with at least {QUAL_MIN_COMMUNITY_SIZE} researchers, plus one contrast case.\n\n"]

positive_keys = set(positive["fused_community"].tolist())

for _,r in selected_examples.iterrows():
    fc = int(r["fused_community"])
    tc = int(r["topology_community_match"])
    f_titles = representative_titles(fused_comms[fc], QUAL_TITLES_PER_COMMUNITY)
    t_titles = representative_titles(top_comms[tc], QUAL_TITLES_PER_COMMUNITY)
    reason = "largest positive coherence change" if fc in positive_keys else "contrast case"

    example_rows.append({
        **r.to_dict(),
        "selection_reason": reason,
        "topology_representative_titles": " || ".join(x["title"] for x in t_titles),
        "fused_representative_titles": " || ".join(x["title"] for x in f_titles),
    })

    md.append(f"## Fused community {fc} vs topology community {tc}\n")
    md.append(
        f"- Selection: {reason}\n"
        f"- Sizes: topology={int(r['topology_size'])}, fused={int(r['fused_size'])}\n"
        f"- Jaccard overlap: {r['jaccard_overlap']:.3f}\n"
        f"- Community semantic coherence: topology={r['topology_community_coherence']:.4f}, "
        f"fused={r['fused_community_coherence']:.4f}, "
        f"delta={r['delta_community_coherence']:+.4f}\n"
    )
    md.append("- Representative topology-only titles:\n")
    for x in t_titles:
        md.append(f"  - {x['title']} ({x['centroid_similarity']:.3f})\n")
    md.append("- Representative fused titles:\n")
    for x in f_titles:
        md.append(f"  - {x['title']} ({x['centroid_similarity']:.3f})\n")
    md.append("\n")

examples_df = pd.DataFrame(example_rows)
examples_df.to_csv(SUPP / "Table_S_Qualitative_Community_Examples.csv", index=False)
(SUPP / "Qualitative_Community_Examples.md").write_text("".join(md), encoding="utf-8")

# ------------------------------------------------------------
# 11. RUNTIME — CONTEXTUALIZAÇÃO DE EFICIÊNCIA
# ------------------------------------------------------------
runtime_existing = (
    grid_runs.groupby(["algorithm","condition"])
    .agg(
        n_runs=("runtime_seconds","count"),
        runtime_mean_s=("runtime_seconds","mean"),
        runtime_median_s=("runtime_seconds","median"),
        runtime_sd_s=("runtime_seconds","std"),
        runtime_max_s=("runtime_seconds","max"),
    )
    .reset_index()
)
runtime_existing.to_csv(SUPP / "Table_S_Runtime_Primary_Grid.csv", index=False)

runtime_refined = (
    refine_runs.groupby("algorithm")
    .agg(
        n_runs=("runtime_seconds","count"),
        runtime_mean_s=("runtime_seconds","mean"),
        runtime_median_s=("runtime_seconds","median"),
        runtime_sd_s=("runtime_seconds","std"),
        runtime_max_s=("runtime_seconds","max"),
    )
    .reset_index()
)
runtime_refined.to_csv(SUPP / "Table_S_Runtime_Refined_Grid.csv", index=False)

# ------------------------------------------------------------
# 12. FIGURAS DE SENSIBILIDADE
# ------------------------------------------------------------
# Uma figura por algoritmo, sem subplots.
for alg in ALGORITHMS:
    sub = threshold_df[
        (threshold_df["algorithm"]==alg)
        & threshold_df["constraint_met"].fillna(False)
    ].copy()
    if not sub.empty:
        fig, ax = plt.subplots(figsize=(6.6,4.4))
        ax.plot(sub["threshold"]*100, sub["delta_SC_adj_vs_topology"], marker="o")
        for _,r in sub.iterrows():
            ax.annotate(
                f"k={int(r['selected_k'])}, a={r['selected_alpha']:.2f}",
                (r["threshold"]*100, r["delta_SC_adj_vs_topology"]),
                xytext=(4,5), textcoords="offset points", fontsize=8
            )
        ax.set_xlabel("Structural-retention threshold (%)")
        ax.set_ylabel("Granularity-adjusted semantic gain")
        ax.set_title(f"Retention-threshold sensitivity — {alg.capitalize()}")
        ax.grid(alpha=0.25)
        fig.tight_layout()
        fig.savefig(FIGURES / f"Fig16_4_threshold_sensitivity_{alg}.png", dpi=500, bbox_inches="tight")
        plt.close(fig)

for alg in ALGORITHMS:
    sub = candidate_union[candidate_union["algorithm"]==alg].copy()
    base_mean = float(grid_agg[
        (grid_agg["condition"]=="topology_only") & (grid_agg["algorithm"]==alg)
    ].iloc[0]["structural_modularity_mean"])
    sub["retained_pct"] = 100*sub["structural_modularity_mean"]/base_mean
    fig, ax = plt.subplots(figsize=(6.6,4.4))
    ax.scatter(sub["retained_pct"], sub["semantic_centroid_coherence_mean"])
    ax.axvline(95, linestyle="--", linewidth=1.0)
    ax.set_xlabel("Mean topology-only modularity retained (%)")
    ax.set_ylabel("Mean semantic centroid coherence")
    ax.set_title(f"Expanded parameter grid — {alg.capitalize()}")
    ax.grid(alpha=0.25)
    fig.tight_layout()
    fig.savefig(FIGURES / f"Fig16_4_expanded_grid_{alg}.png", dpi=500, bbox_inches="tight")
    plt.close(fig)

# ------------------------------------------------------------
# 13. METHODS AUDIT — RESPOSTAS DIRETAS AO PARECER
# ------------------------------------------------------------
methods_audit = f"""PART 16.4 — METHODS AUDIT

Semantic perturbation
---------------------
Noise model: independent Gaussian perturbation per embedding dimension.
epsilon ~ Normal(0, {SEMANTIC_NOISE_SIGMA}^2)
After perturbation, researcher embeddings are L2-normalized.
Cosine similarities are recomputed and the semantic k-NN graph is rebuilt.
Topology edge dropout fraction: {TOPOLOGY_EDGE_DROPOUT:.0%}.
Perturbation replicates in the primary study: 50 per scenario.

Semantic k-NN construction
--------------------------
- Self-similarity is excluded.
- Each researcher selects the k highest cosine-similarity neighbors.
- Minimum accepted semantic similarity: {MIN_SEMANTIC_SIMILARITY}.
- The directed k-NN relation is symmetrized by union: an undirected edge is retained
  if either endpoint selects the other.
- If an undirected edge is encountered twice, the maximum observed cosine similarity
  is retained.
- Semantic edge weights are mass-normalized so their total weight is 1.
- The original implementation uses NumPy argpartition followed by descending sorting;
  no random tie-breaking or semantic jitter is introduced.

Topological layer
-----------------
Primary transform: log1p(raw co-authorship weight), followed by mass normalization.
Part 16.4 additionally evaluates raw, sqrt, and log1p transforms while holding
k={WEIGHT_SENS_K} and alpha={WEIGHT_SENS_ALPHA} fixed.

Original parameter grid
-----------------------
k = {{5,10,15,20}}
alpha = {{0.25,0.50,0.75}}

Part 16.4 refinement / extension
--------------------------------
k = {REFINE_K}
alpha = {REFINE_ALPHA}
10 seeds per configuration.
Retention thresholds = {RETENTION_THRESHOLDS}.
Size-matched null partitions = {N_SIZE_NULL} for each reported selected partition.

Interpretation rule
-------------------
No global algorithm winner is selected.
The sensitivity analysis asks whether the structural-semantic trade-off and the
selected region of parameter space remain stable under reasonable analytical choices.
"""
(LOGS / "Methods_Audit_Part16_4.txt").write_text(methods_audit, encoding="utf-8")

# ------------------------------------------------------------
# 14. RELATÓRIO CONSOLIDADO
# ------------------------------------------------------------
report = []
report.append("PARTE 16.4 — REVIEWER-ORIENTED REVISION REPORT\n")
report.append("="*95 + "\n\n")
report.append("1. ORIGINAL VS EXPANDED GRID AT 95% RETENTION\n")
report.append(selection_comparison.to_string(index=False))
report.append("\n\n2. RETENTION-THRESHOLD SENSITIVITY\n")
report.append(threshold_df.to_string(index=False))
report.append("\n\n3. EFFECT-SIZE CONTEXT\n")
report.append(effect_df.to_string(index=False))
report.append("\n\n4. WEIGHT-TRANSFORM SENSITIVITY\n")
report.append(weight_df.to_string(index=False))
report.append("\n\n5. QUALITATIVE COMMUNITY EXAMPLES\n")
report.append(examples_df.to_string(index=False))
report.append("\n\n6. METHODS AUDIT\n")
report.append(methods_audit)

(OUT / "Part16_4_Reviewer_Revision_Report.txt").write_text(
    "".join(report), encoding="utf-8"
)

# Workbook único
xlsx = OUT / "Part16_4_Reviewer_Revision_Tables.xlsx"
with pd.ExcelWriter(xlsx, engine="openpyxl") as writer:
    refine_runs.to_excel(writer, sheet_name="A_Refined_Runs", index=False)
    refine_agg.to_excel(writer, sheet_name="B_Refined_Aggregated", index=False)
    candidate_union.to_excel(writer, sheet_name="C_Expanded_Grid", index=False)
    threshold_df.to_excel(writer, sheet_name="D_Threshold_Sensitivity", index=False)
    selection_comparison.to_excel(writer, sheet_name="E_Selection_Comparison", index=False)
    effect_df.to_excel(writer, sheet_name="F_Effect_Size", index=False)
    weight_df.to_excel(writer, sheet_name="G_Weight_Transform", index=False)
    qual_full.to_excel(writer, sheet_name="H_Community_Matching", index=False)
    examples_df.to_excel(writer, sheet_name="I_Qual_Examples", index=False)
    runtime_existing.to_excel(writer, sheet_name="J_Runtime_Primary", index=False)
    runtime_refined.to_excel(writer, sheet_name="K_Runtime_Refined", index=False)

manifest = {
    "created_at": datetime.now().isoformat(),
    "part16_run": str(PART16),
    "part16_1_run": str(PART16_1),
    "output_run": str(OUT),
    "retention_thresholds": RETENTION_THRESHOLDS,
    "refine_k": REFINE_K,
    "refine_alpha": REFINE_ALPHA,
    "n_size_null": N_SIZE_NULL,
    "semantic_noise_sigma_primary": SEMANTIC_NOISE_SIGMA,
    "topology_edge_dropout_primary": TOPOLOGY_EDGE_DROPOUT,
    "weight_transform_sensitivity": {
        "transforms": WEIGHT_TRANSFORMS,
        "fixed_k": WEIGHT_SENS_K,
        "fixed_alpha": WEIGHT_SENS_ALPHA,
    },
    "qualitative_examples": {
        "reference_algorithm": QUAL_REFERENCE_ALGO,
        "k": QUAL_K,
        "alpha": QUAL_ALPHA,
        "seed": GLOBAL_SEED,
        "selection_rule": (
            "largest positive matched-community coherence changes among fused communities "
            f"with >= {QUAL_MIN_COMMUNITY_SIZE} researchers, plus one contrast case"
        ),
        "researcher_names_exported": False,
    },
    "reporting_principle": (
        "No global algorithm winner. Sensitivity analyses test whether the reported "
        "structural-semantic trade-off is robust to thresholds, local/boundary parameter "
        "refinement, and topological weight transformation."
    ),
}
(LOGS / "manifest_part16_4.json").write_text(
    json.dumps(manifest, indent=2, ensure_ascii=False), encoding="utf-8"
)

# ------------------------------------------------------------
# 15. IMPRESSÃO FINAL
# ------------------------------------------------------------
print("\n" + "="*100)
print("✅ PARTE 16.4 CONCLUÍDA")
print("="*100)

print("\nORIGINAL VS EXPANDED GRID @ 95%:")
print(selection_comparison.to_string(index=False))

print("\nTHRESHOLD SENSITIVITY:")
print(threshold_df.to_string(index=False))

print("\nEFFECT SIZE:")
print(effect_df.to_string(index=False))

print("\nWEIGHT TRANSFORM SENSITIVITY:")
print(weight_df.to_string(index=False))

print("\nQUALITATIVE EXAMPLES:")
print(examples_df[[
    "fused_community","topology_community_match","fused_size",
    "jaccard_overlap","delta_community_coherence","selection_reason"
]].to_string(index=False))

print("\nArquivos principais:")
print("-", OUT / "Part16_4_Reviewer_Revision_Report.txt")
print("-", OUT / "Part16_4_Reviewer_Revision_Tables.xlsx")
print("-", LOGS / "Methods_Audit_Part16_4.txt")
print("-", TABLES / "Table16_4D_Retention_Threshold_Sensitivity.csv")
print("-", TABLES / "Table16_4E_Original_vs_Expanded_Selection.csv")
print("-", TABLES / "Table16_4F_Effect_Size_Context.csv")
print("-", TABLES / "Table16_4G_Weight_Transform_Sensitivity.csv")
print("-", SUPP / "Qualitative_Community_Examples.md")
print("\nSaída completa:", OUT)


Mounted at /content/drive
PARTE 16.4 — REVIEWER-ORIENTED SENSITIVITY AND INTERPRETABILITY
Parte 16:   /content/drive/MyDrive/tese_doutoral/semantic_topological_runs/run_20260920_010404
Parte 16.1: /content/drive/MyDrive/tese_doutoral/semantic_topological_runs/run_20260920_010404/posthoc_16_1/run_20260920_020905
Saída 16.4: /content/drive/MyDrive/tese_doutoral/semantic_topological_runs/run_20260920_010404/reviewer_revision_16_4/run_20260922_021632

A. REFINAMENTO / EXTENSÃO DA GRADE k x alpha
k= 3, alpha=0.65, alg=louvain
k= 3, alpha=0.65, alg=leiden
k= 3, alpha=0.65, alg=infomap
k= 3, alpha=0.70, alg=louvain
k= 3, alpha=0.70, alg=leiden
k= 3, alpha=0.70, alg=infomap
k= 3, alpha=0.75, alg=louvain
k= 3, alpha=0.75, alg=leiden
k= 3, alpha=0.75, alg=infomap
k= 3, alpha=0.80, alg=louvain
k= 3, alpha=0.80, alg=leiden
k= 3, alpha=0.80, alg=infomap
k= 3, alpha=0.85, alg=louvain
k= 3, alpha=0.85, alg=leiden
k= 3, alpha=0.85, alg=infomap
k= 3, alpha=0.90, alg=louvain
k= 3, alpha=0.90, alg=leiden